In [ ]:
import os
os.environ["HF_TOKEN"] = "YOUR_TOKEN_HERE"

In [ ]:
# ╔═══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 1: SETUP                                                          ║
# ║  Run this cell, then restart runtime if prompted.                        ║
# ╚═══════════════════════════════════════════════════════════════════════════╝

!pip install transformers datasets accelerate matplotlib numpy tqdm scipy peft

In [ ]:
# ╔═══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 2: IMPORTS & DEVICE                                               ║
# ╚═══════════════════════════════════════════════════════════════════════════╝

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import math
import re
import random
import time
import gc
import warnings
warnings.filterwarnings("ignore", message=".*attention_mask.*")
warnings.filterwarnings("ignore", message=".*max_new_tokens.*")
warnings.filterwarnings("ignore", message=".*peft_config.*")
from dataclasses import dataclass, field
from typing import List, Dict, Tuple, Optional
from collections import Counter
from tqdm.auto import tqdm
from scipy import stats as scipy_stats

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ╔═══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 3: LOAD MODEL & TOKENIZER                                         ║
# ╚═══════════════════════════════════════════════════════════════════════════╝

from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.bfloat16
).to(device)
model.eval()

# Verify architecture assumptions
NUM_LAYERS = model.config.num_hidden_layers   # Expected: 22
HIDDEN_DIM = model.config.hidden_size         # Expected: 2048
VOCAB_SIZE = model.config.vocab_size

print(f"Model: {MODEL_NAME}")
print(f"  Layers: {NUM_LAYERS}")
print(f"  Hidden dim: {HIDDEN_DIM}")
print(f"  Vocab size: {VOCAB_SIZE}")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.0f}M")

assert NUM_LAYERS == 22, f"Expected 22 layers, got {NUM_LAYERS}. Adjust source/target layers."
assert HIDDEN_DIM == 2048, f"Expected D=2048, got {HIDDEN_DIM}. Adjust bottleneck sizing."

In [ ]:
# ╔═══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 4: LOAD & CLEAN DATASET                                           ║
# ╚═══════════════════════════════════════════════════════════════════════════╝

from datasets import load_dataset

dataset = load_dataset("wikitext", "wikitext-103-raw-v1", split="train")

# Filter short passages and clean markup artifacts (spec §5.1 junk filtering)
texts = [t for t in dataset["text"] if len(t.strip()) > 200]
texts = [re.sub(r'[@=\[\]]+', ' ', t).strip() for t in texts]
texts = [re.sub(r'\s+', ' ', t) for t in texts]  # Collapse whitespace
texts = [t for t in texts if len(t) > 200]

random.seed(42)
random.shuffle(texts)

# Splits
HEAD_TRAIN_SIZE = 500       # For training prediction heads (spec §4.1)
HEAD_VAL_SIZE = 100         # For validating heads (spec §4.3)
WAKE_CORPUS_SIZE = 3000     # For wake phase
EVAL_SIZE = 300             # Held-out evaluation

head_train_corpus = texts[:HEAD_TRAIN_SIZE]
head_val_corpus = texts[HEAD_TRAIN_SIZE:HEAD_TRAIN_SIZE + HEAD_VAL_SIZE]
wake_corpus = texts[HEAD_TRAIN_SIZE + HEAD_VAL_SIZE:
                    HEAD_TRAIN_SIZE + HEAD_VAL_SIZE + WAKE_CORPUS_SIZE]
eval_corpus = texts[HEAD_TRAIN_SIZE + HEAD_VAL_SIZE + WAKE_CORPUS_SIZE:
                    HEAD_TRAIN_SIZE + HEAD_VAL_SIZE + WAKE_CORPUS_SIZE + EVAL_SIZE]

print(f"Head training corpus: {len(head_train_corpus)} passages")
print(f"Head validation corpus: {len(head_val_corpus)} passages")
print(f"Wake corpus: {len(wake_corpus)} passages")
print(f"Eval corpus: {len(eval_corpus)} passages")
print(f"Sample: {wake_corpus[0][:200]}...")

In [ ]:
# ╔═══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 5: RUNNING STATISTICS (spec §3.2)                                 ║
# ╚═══════════════════════════════════════════════════════════════════════════╝

class RunningStats:
    """
    Exponential moving average of mean and variance for normalizing
    prediction errors across levels to zero-mean, unit-variance.

    Uses Welford-style online variance with EMA (spec §3.2):
        μ(n) = α·μ(n-1) + (1-α)·x(n)
        σ²(n) = α·σ²(n-1) + (1-α)·(x-μ_old)·(x-μ_new)
    """
    def __init__(self, momentum: float = 0.99):
        self.momentum = momentum
        self.running_mean: Optional[float] = None
        self.running_var: Optional[float] = None
        self.n_updates: int = 0

    def update(self, x: torch.Tensor):
        """Update stats with a batch of scalar error values. x: any shape tensor."""
        # Filter out zeros (cold-start / junk-filtered positions)
        x_flat = x.flatten()
        x_nz = x_flat[x_flat > 0]
        if len(x_nz) < 2:
            return

        batch_mean = x_nz.mean().item()
        batch_var = x_nz.var().item()

        if self.running_mean is None:
            self.running_mean = batch_mean
            self.running_var = batch_var
        else:
            m = self.momentum
            old_mean = self.running_mean
            self.running_mean = m * self.running_mean + (1 - m) * batch_mean
            # Welford-style: use delta from old and new mean
            self.running_var = m * self.running_var + (1 - m) * (batch_mean - old_mean) * (batch_mean - self.running_mean)
            # Ensure variance doesn't go negative from numerical issues
            self.running_var = max(self.running_var, 1e-10)

        self.n_updates += 1

    def normalize(self, x: torch.Tensor) -> torch.Tensor:
        """Normalize to approximately zero-mean, unit-variance."""
        if self.running_mean is None or self.n_updates < 5:
            return x  # Not enough data yet — return raw
        return (x - self.running_mean) / (math.sqrt(self.running_var) + 1e-8)

    def get_threshold(self, z: float = 1.5) -> float:
        """Return mean + z*std threshold."""
        if self.running_mean is None:
            return float('inf')
        return self.running_mean + z * math.sqrt(self.running_var + 1e-8)

    def __repr__(self):
        if self.running_mean is None:
            return "RunningStats(uninitialized)"
        return (f"RunningStats(μ={self.running_mean:.6f}, "
                f"σ={math.sqrt(self.running_var):.6f}, n={self.n_updates})")

In [ ]:
# ╔═══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 6 — REPLACE ENTIRELY                                              ║
# ║  PREDICTION HEADS (spec §2.2)                                            ║
# ║                                                                          ║
# ║  Changes from v1:                                                        ║
# ║    1. bottleneck_factor: 0.5 → 0.25 (force compressed prediction)        ║
# ║    2. use_deltas=True: predict layer DELTAS not full hidden states        ║
# ║       (spec §7.3 — decomposes residual stream, decorrelates heads)        ║
# ╚═══════════════════════════════════════════════════════════════════════════╝

class LayerPredictionHead(nn.Module):
    """
    Lightweight MLP that predicts hidden states at layer target from layer source.

    Architecture (spec §2.2):
        P(h) = W₂ · GELU(W₁ · LN(h) + b₁) + b₂

    v2 changes:
        - Bottleneck D/4 instead of D/2. Forces more lossy compression,
          so the head can't memorize the mapping. Errors on unusual tokens
          become larger and more discriminative.
    """
    def __init__(self, hidden_size: int, bottleneck_factor: float = 0.25):
        super().__init__()
        bottleneck = int(hidden_size * bottleneck_factor)
        self.ln = nn.LayerNorm(hidden_size)
        self.down = nn.Linear(hidden_size, bottleneck)
        self.up = nn.Linear(bottleneck, hidden_size)

    def forward(self, source_hidden: torch.Tensor) -> torch.Tensor:
        """source_hidden: [B, T, D] → predicted target hidden: [B, T, D]"""
        h = self.ln(source_hidden.float())
        h = F.gelu(self.down(h))
        h = self.up(h)
        return h


class PredictionHeadSystem(nn.Module):
    """
    Three prediction heads spanning TinyLlama's 22 layers (spec §2.2):
        Surface:    Layer 7  → predicts Layer 0  output
        Structural: Layer 14 → predicts Layer 7  output
        Semantic:   Layer 21 → predicts Layer 14 output

    CRITICAL INDEXING (spec §5.2):
        hidden_states[0]  = embedding output (before any transformer layer)
        hidden_states[l+1] = output of transformer layer l
        So "layer 7 output" = hidden_states[8]

    v2: use_deltas=True predicts layer DELTAS (h_l - h_{l-1}) instead of
    full hidden states. This isolates each layer's unique computational
    contribution and removes the residual stream that caused high
    cross-correlation between structural and semantic heads (spec §7.3).
    """

    # Level definitions: (name, source_layer, target_layer)
    LEVELS = [
        ("surface",     7,  0),
        ("structural", 14,  7),
        ("semantic",   21, 14),
    ]

    def __init__(
        self,
        hidden_size: int,
        bottleneck_factor: float = 0.25,
        use_deltas: bool = True,
    ):
        super().__init__()
        self.hidden_size = hidden_size
        self.use_deltas = use_deltas

        self.heads = nn.ModuleDict()
        for name, src, tgt in self.LEVELS:
            self.heads[name] = LayerPredictionHead(hidden_size, bottleneck_factor)

        total_params = sum(p.numel() for p in self.parameters())
        print(f"PredictionHeadSystem initialized: {total_params / 1e6:.2f}M params")
        print(f"  Mode: {'DELTA' if use_deltas else 'FULL HIDDEN STATE'}")
        print(f"  Bottleneck: D × {bottleneck_factor} = {int(hidden_size * bottleneck_factor)}")
        for name, src, tgt in self.LEVELS:
            head_params = sum(p.numel() for p in self.heads[name].parameters())
            if use_deltas:
                print(f"  {name:12s}: ΔL{src} → ΔL{tgt}  ({head_params / 1e6:.2f}M)")
            else:
                print(f"  {name:12s}: L{src} → L{tgt}  ({head_params / 1e6:.2f}M)")

    def compute_errors(
        self, hidden_states: Tuple[torch.Tensor, ...]
    ) -> Dict[str, torch.Tensor]:
        """
        Compute prediction error at each level.

        Args:
            hidden_states: tuple of (NUM_LAYERS+1) tensors, each [B, T, D]
                           from model(..., output_hidden_states=True).hidden_states

        Returns:
            Dict mapping level name → [B, T] scalar prediction errors.

        In delta mode (spec §7.3):
            source_input = h[src+1] - h[src]    (layer src's unique contribution)
            target       = h[tgt+1] - h[tgt]    (layer tgt's unique contribution)
        In full mode:
            source_input = h[src+1]
            target       = h[tgt+1]

        CRITICAL: Targets are detached (spec §3.1).
        """
        errors = {}
        for name, src, tgt in self.LEVELS:
            if self.use_deltas:
                # Layer delta = output - input of that layer
                # h[l+1] = output of layer l, h[l] = input to layer l
                source_input = hidden_states[src + 1] - hidden_states[src]
                target = (hidden_states[tgt + 1] - hidden_states[tgt]).detach().float()
            else:
                source_input = hidden_states[src + 1]
                target = hidden_states[tgt + 1].detach().float()

            predicted = self.heads[name](source_input)  # [B, T, D]
            error = torch.mean((predicted - target) ** 2, dim=-1)  # [B, T]
            errors[name] = error

        return errors

In [ ]:
def validate_heads_post_merge(
    model: nn.Module,
    head_system: PredictionHeadSystem,
    tokenizer,
    val_texts: List[str],
    device: torch.device,
    n_passages: int = 50,
) -> float:
    """
    Check whether prediction heads still produce discriminative errors
    after model weights have changed. Measures CV of head errors as proxy.
    If CV drops below threshold, heads need retraining.
    """
    model.eval()
    head_system.eval()
    sample = val_texts[:n_passages]

    per_level_errors = {name: [] for name, _, _ in PredictionHeadSystem.LEVELS}

    with torch.no_grad():
        for text in sample:
            enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=512).to(device)
            if enc.input_ids.shape[1] < 10:
                continue
            outputs = model(**enc, output_hidden_states=True)
            hs = outputs.hidden_states

            for name, src, tgt in PredictionHeadSystem.LEVELS:
                src_h = hs[src + 1][:, :-1, :]
                tgt_h = hs[tgt + 1][:, 1:, :]
                if head_system.use_deltas:
                    src_h = hs[src + 1][:, :-1, :] - hs[src][:, :-1, :]
                    tgt_h = hs[tgt + 1][:, 1:, :] - hs[tgt][:, 1:, :]
                pred = head_system.heads[name](src_h)
                err = (pred - tgt_h).pow(2).mean(dim=-1).squeeze(0)
                per_level_errors[name].extend(err.cpu().tolist())

    cvs = {}
    for name, errors in per_level_errors.items():
        if len(errors) > 0:
            mu = np.mean(errors)
            sigma = np.std(errors)
            cvs[name] = sigma / max(mu, 1e-10)
        else:
            cvs[name] = 0.0

    min_cv = min(cvs.values())
    mean_cv = np.mean(list(cvs.values()))
    print(f"  Head validation — CV per level: " +
          ", ".join(f"{k}={v:.3f}" for k, v in cvs.items()) +
          f" | mean={mean_cv:.3f}")

    return mean_cv


In [ ]:
# ╔═══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 7 — REPLACE ENTIRELY                                              ║
# ║  TRAIN PREDICTION HEADS (spec §4)                                        ║
# ║                                                                          ║
# ║  Changes from v1:                                                        ║
# ║    1. max_epochs: 10 → 3 (aggressive undertraining)                      ║
# ║    2. min_improvement: 0.02 → 0.05 (stop earlier in diminishing returns) ║
# ║    3. LR schedule: cosine → constant with warmup (avoid slow convergence ║
# ║       at low LR that squeezed out tiny improvements in v1)               ║
# ║    4. Added periodic CV monitoring during training to detect              ║
# ║       when heads are becoming non-discriminative                          ║
# ╚═══════════════════════════════════════════════════════════════════════════╝

def train_prediction_heads(
    base_model: nn.Module,
    tokenizer,
    head_system: PredictionHeadSystem,
    train_texts: List[str],
    val_texts: List[str],
    device: torch.device,
    max_epochs: int = 3,             # v2: was 10, now 3
    lr: float = 1e-3,
    max_length: int = 2048,
    patience: int = 2,               # v2: was 3, now 2
    min_improvement: float = 0.05,   # v2: was 0.02, now 0.05
    log_interval: int = 100,
    cv_check_interval: int = 200,    # v2: new — check CV during training
) -> PredictionHeadSystem:
    """
    Train prediction heads on frozen base model (spec §4.1-4.2).

    v2 philosophy: UNDERTRAIN deliberately. We want heads that are good enough
    to predict easy tokens but bad enough to produce large, discriminative
    errors on hard tokens. Training to convergence kills the signal.
    """

    print("=" * 65)
    print("TRAINING PREDICTION HEADS (v2 — aggressive undertraining)")
    print(f"  Base model: frozen ({sum(p.numel() for p in base_model.parameters())/1e6:.0f}M params)")
    print(f"  Head params: {sum(p.numel() for p in head_system.parameters())/1e6:.2f}M (trainable)")
    print(f"  Train passages: {len(train_texts)}, Val passages: {len(val_texts)}")
    print(f"  Max epochs: {max_epochs}, Patience: {patience}, Min improvement: {min_improvement}")
    print(f"  Delta mode: {head_system.use_deltas}")
    print("=" * 65)

    base_model.eval()
    for p in base_model.parameters():
        p.requires_grad = False

    head_system.to(device)
    head_system.train()

    optimizer = torch.optim.AdamW(head_system.parameters(), lr=lr, weight_decay=0.01)

    # v2: linear warmup then constant, instead of cosine annealing.
    # Cosine was the problem — it kept reducing LR, allowing heads to
    # slowly converge over 10 epochs. Constant LR makes later epochs
    # produce diminishing improvements, triggering early stopping sooner.
    total_steps = max_epochs * len(train_texts)
    warmup_steps = min(100, len(train_texts))


    def lr_lambda(step):
        if step < warmup_steps:
            return step / warmup_steps
        return 1.0

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

    best_val_loss = float('inf')
    no_improve_count = 0
    best_state_dict = None

    # v2: track per-level errors during training for CV monitoring
    recent_errors_by_level = {name: [] for name, _, _ in PredictionHeadSystem.LEVELS}

    for epoch in range(max_epochs):
        head_system.train()
        epoch_loss = 0.0
        n_examples = 0

        for i, text in enumerate(tqdm(train_texts, desc=f"Epoch {epoch+1}/{max_epochs}")):
            encodings = tokenizer(
                text, return_tensors="pt", truncation=True, max_length=max_length
            ).to(device)

            if encodings.input_ids.shape[1] < 30:
                continue

            with torch.no_grad():
                outputs = base_model(**encodings, output_hidden_states=True)

            errors = head_system.compute_errors(outputs.hidden_states)
            loss = sum(e.mean() for e in errors.values())

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(head_system.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()

            epoch_loss += loss.item()
            n_examples += 1

            # v2: collect per-level error stats for CV monitoring
            for name in errors:
                err_vals = errors[name][0].detach().cpu().numpy()
                recent_errors_by_level[name].append(float(err_vals.std() / (err_vals.mean() + 1e-10)))

            if (i + 1) % log_interval == 0:
                print(f"  [{i+1}/{len(train_texts)}] "
                      f"Loss: {epoch_loss / n_examples:.6f}  "
                      f"LR: {scheduler.get_last_lr()[0]:.2e}")

            # v2: periodic CV check — are we losing discriminability?
            if (i + 1) % cv_check_interval == 0 and len(recent_errors_by_level["structural"]) >= cv_check_interval:
                for name in recent_errors_by_level:
                    recent_cvs = recent_errors_by_level[name][-cv_check_interval:]
                    mean_cv = np.mean(recent_cvs)
                    if mean_cv < 0.3:
                        print(f"  ⚠ {name} CV={mean_cv:.3f} — approaching non-discriminative")

        avg_train_loss = epoch_loss / max(n_examples, 1)

        # ---- Validation ----
        head_system.eval()
        val_loss = 0.0
        val_cv_by_level = {name: [] for name, _, _ in PredictionHeadSystem.LEVELS}
        n_val = 0

        with torch.no_grad():
            for text in val_texts:
                encodings = tokenizer(
                    text, return_tensors="pt", truncation=True, max_length=max_length
                ).to(device)
                if encodings.input_ids.shape[1] < 30:
                    continue
                outputs = base_model(**encodings, output_hidden_states=True)
                errors = head_system.compute_errors(outputs.hidden_states)
                val_loss += sum(e.mean().item() for e in errors.values())
                for name in errors:
                    err_vals = errors[name][0].cpu().numpy()
                    nz = err_vals[err_vals > 0]
                    if len(nz) > 10:
                        val_cv_by_level[name].append(float(nz.std() / (nz.mean() + 1e-10)))
                n_val += 1

        avg_val_loss = val_loss / max(n_val, 1)

        # v2: report per-level CV on validation set
        print(f"  Epoch {epoch+1}: Train loss={avg_train_loss:.6f}, "
              f"Val loss={avg_val_loss:.6f}")
        for name in val_cv_by_level:
            if val_cv_by_level[name]:
                cv = np.mean(val_cv_by_level[name])
                status = "✓" if cv >= 0.5 else "⚠ LOW"
                print(f"    {name:12s} val CV: {cv:.3f} {status}")

        # ---- Early stopping (spec §4.2 — v2: more aggressive) ----
        if best_val_loss < float('inf'):
            relative_improvement = (best_val_loss - avg_val_loss) / best_val_loss
        else:
            relative_improvement = 1.0

        if relative_improvement < min_improvement:
            no_improve_count += 1
            print(f"  Improvement: {relative_improvement:.4f} < {min_improvement} "
                  f"(patience: {no_improve_count}/{patience})")
        else:
            no_improve_count = 0
            best_val_loss = avg_val_loss
            best_state_dict = {k: v.clone() for k, v in head_system.state_dict().items()}
            print(f"  New best val loss: {best_val_loss:.6f}")

        if no_improve_count >= patience:
            print(f"\n  Early stopping at epoch {epoch+1}: "
                  f"<{min_improvement:.0%} improvement for {patience} epochs")
            break

    # Restore best checkpoint
    if best_state_dict is not None:
        head_system.load_state_dict(best_state_dict)
        print(f"  Restored best checkpoint (val loss: {best_val_loss:.6f})")

    head_system.eval()
    print("Head training complete.\n")
    return head_system

In [ ]:
# ╔═══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 8 — REPLACE ENTIRELY                                              ║
# ║  HEAD QUALITY VALIDATION (spec §4.3)                                     ║
# ║                                                                          ║
# ║  Changes from v1:                                                        ║
# ║    1. Handles delta mode (different error magnitudes/distributions)       ║
# ║    2. Reports raw error magnitude per level (useful for debugging scale)  ║
# ║    3. Clearer failure mode diagnosis with recommendations                ║
# ╚═══════════════════════════════════════════════════════════════════════════╝

def validate_heads(
    base_model: nn.Module,
    tokenizer,
    head_system: PredictionHeadSystem,
    val_texts: List[str],
    device: torch.device,
    max_length: int = 2048,
) -> Dict[str, dict]:
    """
    Validate prediction head quality (spec §4.3).

    Checks:
    1. Coefficient of variation (CV) of errors — should be 0.5-3.0
    2. Correlation with CE loss — should be 0.2-0.8
    3. Cross-correlation between heads — should be < 0.6

    v2: prints actionable recommendations for each failure mode.
    """

    print("=" * 65)
    print("VALIDATING PREDICTION HEAD QUALITY")
    print(f"  Mode: {'DELTA' if head_system.use_deltas else 'FULL HIDDEN STATE'}")
    print("=" * 65)

    base_model.eval()
    head_system.eval()

    all_ce = []
    all_errors = {name: [] for name, _, _ in PredictionHeadSystem.LEVELS}

    with torch.no_grad():
        for text in tqdm(val_texts, desc="Validation inference"):
            encodings = tokenizer(
                text, return_tensors="pt", truncation=True, max_length=max_length
            ).to(device)

            if encodings.input_ids.shape[1] < 30:
                continue

            outputs = base_model(
                **encodings, output_hidden_states=True, labels=encodings.input_ids
            )

            # CE loss per token
            logits = outputs.logits[0, :-1, :].float()
            labels = encodings.input_ids[0, 1:]
            ce_per_token = F.cross_entropy(logits, labels, reduction='none')

            # Prediction errors
            errors = head_system.compute_errors(outputs.hidden_states)

            # Align (spec §5.2): slice [0, 1:]
            min_len = len(ce_per_token)
            for name in all_errors:
                err = errors[name][0, 1:].cpu().numpy()
                min_len = min(min_len, len(err))

            all_ce.append(ce_per_token[:min_len].cpu().numpy())
            for name in all_errors:
                all_errors[name].append(errors[name][0, 1:][:min_len].cpu().numpy())

    ce_flat = np.concatenate(all_ce)
    error_flat = {name: np.concatenate(arrs) for name, arrs in all_errors.items()}

    for name in error_flat:
        assert len(error_flat[name]) == len(ce_flat), \
            f"Alignment error: {name} has {len(error_flat[name])}, CE has {len(ce_flat)}"

    results = {}
    all_pass = True

    for name, _, _ in PredictionHeadSystem.LEVELS:
        err = error_flat[name]
        nz_mask = err > 0
        err_nz = err[nz_mask]
        ce_nz = ce_flat[nz_mask]

        if len(err_nz) < 100:
            print(f"\n  {name}: INSUFFICIENT DATA ({len(err_nz)} nonzero tokens)")
            results[name] = {"status": "insufficient_data"}
            continue

        cv = err_nz.std() / (err_nz.mean() + 1e-8)
        corr_with_ce, p_val = scipy_stats.pearsonr(err_nz, ce_nz)

        results[name] = {
            "mean": float(err_nz.mean()),
            "std": float(err_nz.std()),
            "cv": float(cv),
            "corr_with_ce": float(corr_with_ce),
            "corr_p_value": float(p_val),
            "n_tokens": len(err_nz),
        }

        cv_ok = 0.5 <= cv <= 3.0
        corr_ok = 0.2 <= abs(corr_with_ce) <= 0.8

        cv_status = "✓" if cv_ok else "✗"
        corr_status = "✓" if corr_ok else "✗"
        if not (cv_ok and corr_ok):
            all_pass = False

        print(f"\n  {name.upper()}:")
        print(f"    Mean error:     {err_nz.mean():.6f}")
        print(f"    Std error:      {err_nz.std():.6f}")
        print(f"    CV:             {cv:.3f}  {cv_status}  (target: 0.5-3.0)")
        print(f"    Corr with CE:   {corr_with_ce:.3f}  {corr_status}  (target: 0.2-0.8)")

        # v2: actionable diagnosis
        if cv < 0.5:
            print(f"    → OVERTRAINED: errors are too uniform. Need fewer epochs or smaller bottleneck.")
        elif cv > 3.0:
            print(f"    → UNDERTRAINED/UNSTABLE: errors are too noisy. Need more epochs or larger bottleneck.")
        if abs(corr_with_ce) < 0.2:
            print(f"    → DECORRELATED from CE: head may be predicting noise, not meaningful structure.")
            print(f"      If delta mode: this can be acceptable if CV is good (deltas measure different info than CE).")
        elif abs(corr_with_ce) > 0.8:
            print(f"    → REDUNDANT with CE: head is just relearning output prediction. Widen layer gap.")

    # Cross-correlations
    print(f"\n  CROSS-CORRELATIONS:")
    level_names = [name for name, _, _ in PredictionHeadSystem.LEVELS]
    for i in range(len(level_names)):
        for j in range(i + 1, len(level_names)):
            name_i, name_j = level_names[i], level_names[j]
            err_i = error_flat[name_i]
            err_j = error_flat[name_j]
            mask = (err_i > 0) & (err_j > 0)
            if mask.sum() < 100:
                continue
            cross_corr, _ = scipy_stats.pearsonr(err_i[mask], err_j[mask])
            status = "✓" if abs(cross_corr) < 0.6 else "✗"
            if abs(cross_corr) >= 0.6:
                all_pass = False
            print(f"    {name_i} ↔ {name_j}: {cross_corr:.3f}  {status}  (target: < 0.6)")
            if abs(cross_corr) >= 0.6:
                print(f"      → Heads are too correlated. In full mode, try delta mode (use_deltas=True).")
                print(f"        In delta mode, try widening layer gap or different bottleneck sizes.")
            results[f"cross_{name_i}_{name_j}"] = float(cross_corr)

    # v2: overall verdict
    print(f"\n  OVERALL: {'ALL CHECKS PASSED ✓' if all_pass else 'SOME CHECKS FAILED ✗'}")
    if all_pass:
        print(f"  Heads are well-calibrated. Proceed to wake phase.")
    else:
        print(f"  See recommendations above. If CV < 0.5: reduce max_epochs or bottleneck_factor.")
        print(f"  Note: in delta mode, low CE correlation is acceptable if CV is healthy,")
        print(f"  because layer deltas capture different information than the output prediction.")

    print("=" * 65)
    return results

In [ ]:
# ╔═══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 9 — REPLACE ENTIRELY                                              ║
# ║  HIERARCHICAL SURPRISE TRACKER — WAKE PHASE (spec §5, §3)               ║
# ║                                                                          ║
# ║  Changes from v2:                                                        ║
# ║    1. Computes unigram token frequencies during the wake phase            ║
# ║    2. Exposes token_frequencies dict for use in classification            ║
# ╚═══════════════════════════════════════════════════════════════════════════╝

@dataclass
class SurpriseVector:
    """4D surprise signal for a single token (spec §3.3)."""
    ce_loss: float
    surface: float
    structural: float
    semantic: float


@dataclass
class PassageSurprise:
    """All surprise data for one corpus passage."""
    passage_idx: int
    input_ids: torch.Tensor           # [T] on CPU
    vectors: List[SurpriseVector]     # Length T-1 (shifted for next-token prediction)


class HierarchicalSurpriseTracker:
    """
    Wake phase: process corpus passages and compute 4D surprise vectors.

    For each token, produces:
        s(t) = [CE_loss, surface_error_normalized, structural_error_normalized,
                semantic_error_normalized]

    v3 additions:
        - Tracks unigram token frequencies across the wake corpus.
          Used by the classifier to identify rare tokens (proper nouns,
          technical terms) that should be routed to vocabulary_gap
          regardless of hierarchical signal.
    """

    def __init__(
        self,
        base_model: nn.Module,
        tokenizer,
        head_system: PredictionHeadSystem,
        device: torch.device,
        max_length: int = 2048,
        cold_start_skip: int = 20,
    ):
        self.model = base_model
        self.tokenizer = tokenizer
        self.heads = head_system
        self.device = device
        self.max_length = max_length
        self.cold_start_skip = cold_start_skip

        # Running normalization stats — one per level (spec §3.2)
        self.stats = {
            name: RunningStats(momentum=0.99)
            for name, _, _ in PredictionHeadSystem.LEVELS
        }

        # Junk token filter (spec §5.1)
        junk_chars = "@ = [ ] < > { } | # * ~"
        self.junk_token_ids = set()
        for ch in junk_chars.split():
            self.junk_token_ids.update(tokenizer.encode(ch, add_special_tokens=False))

        # v3: Token frequency tracking
        self.token_counts: Counter = Counter()
        self.total_tokens_seen: int = 0

        # Storage
        self.passage_data: List[PassageSurprise] = []

        # Global flat storage for statistics and thresholding
        self.all_ce: List[float] = []
        self.all_errors: Dict[str, List[float]] = {
            name: [] for name, _, _ in PredictionHeadSystem.LEVELS
        }

    def get_token_rarity_percentile(self, token_id: int) -> float:
        """
        Returns approximate percentile rank of this token's frequency.
        0.0 = never seen (rarest), 1.0 = most frequent.
        """
        if self.total_tokens_seen == 0:
            return 0.5  # No data yet
        count = self.token_counts.get(token_id, 0)
        if count == 0:
            return 0.0
        # Approximate: fraction of vocabulary that is less frequent
        freq = count / self.total_tokens_seen
        return min(freq * len(self.token_counts), 1.0)

    @torch.no_grad()
    def process_passage(self, text: str, passage_idx: int) -> Optional[PassageSurprise]:
        """
        Process one passage through the full forward pass (spec §5.1).
        Returns PassageSurprise or None if passage is too short.
        """
        encodings = self.tokenizer(
            text, return_tensors="pt", truncation=True, max_length=self.max_length
        ).to(self.device)

        input_ids = encodings.input_ids[0]  # [T]
        T = len(input_ids)

        if T < self.cold_start_skip + 5:
            return None

        # v3: Update token frequency counts
        for tid in input_ids.tolist():
            self.token_counts[tid] += 1
        self.total_tokens_seen += T

        # ---- Base model forward pass ----
        outputs = self.model(
            **encodings, output_hidden_states=True, labels=encodings.input_ids
        )

        # ---- CE loss per token (spec §5.1) ----
        logits = outputs.logits[0, :-1, :].float()
        labels = input_ids[1:]
        ce_per_token = F.cross_entropy(logits, labels, reduction='none')

        # ---- Hierarchical prediction errors (spec §3.1) ----
        h = outputs.hidden_states
        errors_raw = self.heads.compute_errors(h)

        # Align prediction errors with CE loss: slice [:, 1:] (spec §5.2)
        errors_aligned = {}
        for name in errors_raw:
            err = errors_raw[name][0, 1:]
            min_len = min(len(ce_per_token), len(err))
            errors_aligned[name] = err[:min_len]
        ce_per_token = ce_per_token[:min_len]

        # ---- ALIGNMENT VERIFICATION (spec §5.2) ----
        lengths = [len(ce_per_token)] + [len(v) for v in errors_aligned.values()]
        assert len(set(lengths)) == 1, f"Alignment broken: {lengths}"
        seq_len = lengths[0]

        # ---- Build junk mask ----
        junk_mask = torch.zeros(seq_len, dtype=torch.bool, device=self.device)
        for pos in range(seq_len):
            token_id = input_ids[pos + 1].item()
            if token_id in self.junk_token_ids:
                junk_mask[pos] = True

        # ---- Apply filters: zero out cold-start and junk (spec §5.1) ----
        filter_mask = torch.zeros(seq_len, dtype=torch.bool, device=self.device)
        filter_mask[:self.cold_start_skip] = True
        filter_mask |= junk_mask

        ce_per_token[filter_mask] = 0.0
        for name in errors_aligned:
            errors_aligned[name][filter_mask] = 0.0

        # ---- Normalize errors via running stats (spec §3.2) ----
        errors_normalized = {}
        for name in errors_aligned:
            raw = errors_aligned[name]
            self.stats[name].update(raw)
            errors_normalized[name] = self.stats[name].normalize(raw)
            errors_normalized[name][filter_mask] = 0.0

        # ---- Construct surprise vectors (spec §3.3) ----
        ce_np = ce_per_token.cpu().numpy()
        err_np = {name: errors_normalized[name].cpu().numpy() for name in errors_normalized}

        vectors = []
        for t in range(seq_len):
            vectors.append(SurpriseVector(
                ce_loss=float(ce_np[t]),
                surface=float(err_np["surface"][t]),
                structural=float(err_np["structural"][t]),
                semantic=float(err_np["semantic"][t]),
            ))

        # ---- Store for global statistics ----
        for t in range(seq_len):
            if ce_np[t] > 0:
                self.all_ce.append(float(ce_np[t]))
                for name in err_np:
                    self.all_errors[name].append(float(err_np[name][t]))

        result = PassageSurprise(
            passage_idx=passage_idx,
            input_ids=input_ids.cpu(),
            vectors=vectors,
        )
        self.passage_data.append(result)
        return result

    def run_wake_phase(self, corpus: List[str], log_interval: int = 200):
        """Process entire corpus through hierarchical surprise tracking."""
        print("=" * 65)
        print("WAKE PHASE: Hierarchical Surprise Tracking")
        print(f"  Corpus: {len(corpus)} passages")
        print(f"  Cold start skip: {self.cold_start_skip} tokens")
        print(f"  Junk tokens filtered: {len(self.junk_token_ids)} token IDs")
        print("=" * 65)

        self.model.eval()
        self.heads.eval()
        start_time = time.time()

        for i, text in enumerate(tqdm(corpus, desc="Processing")):
            self.process_passage(text, i)

            if (i + 1) % log_interval == 0:
                elapsed = time.time() - start_time
                rate = (i + 1) / elapsed
                self._print_running_stats(i + 1, rate)

        elapsed = time.time() - start_time
        n_unique = len(self.token_counts)
        print(f"\nWake phase complete: {len(self.passage_data)} passages, "
              f"{len(self.all_ce)} tokens, {elapsed:.1f}s")
        print(f"  Unique tokens seen: {n_unique}, Total: {self.total_tokens_seen}")
        self._print_running_stats(len(corpus), len(corpus) / elapsed)

    def _print_running_stats(self, n_processed, rate):
        print(f"  [{n_processed}] {rate:.1f} passages/s | "
              f"CE μ={np.mean(self.all_ce[-5000:]):.3f} | "
              + " | ".join(
                  f"{name}: {str(self.stats[name])}"
                  for name, _, _ in PredictionHeadSystem.LEVELS
              ))

In [ ]:
# ╔═══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 10 — REPLACE ENTIRELY                                             ║
# ║  SURPRISE CLASSIFICATION (spec §6) — v3b                                ║
# ║                                                                          ║
# ║  Changes from v3:                                                        ║
# ║    - Eliminated "ambiguous" as a classification bucket.                   ║
# ║    - High CE + no hierarchical signal = output-only vocabulary gap.       ║
# ║      Model processes token normally at every level but can't predict      ║
# ║      the specific output. This IS vocabulary gap, not ambiguous.          ║
# ║    - Kept rarity routing as supplementary signal.                         ║
# ║    - Added vocabulary_gap sub-types for diagnostics:                      ║
# ║        surface:    high surface error only (rare surface form)            ║
# ║        output_only: no hierarchical signal (proper nouns, specific names) ║
# ║        rare_token: rarity-routed regardless of errors                     ║
# ╚═══════════════════════════════════════════════════════════════════════════╝

@dataclass
class ClassificationThresholds:
    """Thresholds for surprise classification (spec §6.1-6.2)."""
    ce: float
    surface: float
    structural: float
    semantic: float


def compute_thresholds(
    tracker: HierarchicalSurpriseTracker,
    ce_percentile: float = 85,
    surface_percentile: float = 75,
    structural_percentile: float = 85,
    semantic_percentile: float = 85,
) -> ClassificationThresholds:
    """Compute adaptive thresholds from wake phase statistics."""

    nz_ce = [v for v in tracker.all_ce if v > 0]
    nz_surf = [v for v in tracker.all_errors["surface"] if v > 0]
    nz_struct = [v for v in tracker.all_errors["structural"] if v > 0]
    nz_sem = [v for v in tracker.all_errors["semantic"] if v > 0]

    thresholds = ClassificationThresholds(
        ce=float(np.percentile(nz_ce, ce_percentile)),
        surface=float(np.percentile(nz_surf, surface_percentile)),
        structural=float(np.percentile(nz_struct, structural_percentile)),
        semantic=float(np.percentile(nz_sem, semantic_percentile)),
    )

    print(f"Classification thresholds:")
    print(f"  CE (P{ce_percentile}):           {thresholds.ce:.4f}")
    print(f"  Surface (P{surface_percentile}):       {thresholds.surface:.4f}")
    print(f"  Structural (P{structural_percentile}):    {thresholds.structural:.4f}")
    print(f"  Semantic (P{semantic_percentile}):      {thresholds.semantic:.4f}")

    return thresholds


def compute_rarity_threshold(
    tracker: HierarchicalSurpriseTracker,
    rarity_percentile: float = 10,
) -> int:
    """
    Compute the count threshold below which a token is considered 'rare.'
    """
    counts = np.array([c for c in tracker.token_counts.values() if c > 0])
    if len(counts) == 0:
        return 1
    threshold = int(np.percentile(counts, rarity_percentile))
    threshold = max(threshold, 2)

    n_rare = sum(1 for c in tracker.token_counts.values() if c <= threshold)
    n_total = len(tracker.token_counts)
    print(f"Rarity threshold: count ≤ {threshold} "
          f"({n_rare}/{n_total} unique tokens = {n_rare/max(n_total,1)*100:.1f}% of vocabulary)")

    return threshold


def classify_surprise(
    tracker: HierarchicalSurpriseTracker,
    thresholds: ClassificationThresholds,
    rarity_count_threshold: int = 2,
) -> Dict[str, List[Tuple[int, int]]]:
    """
    Classify every high-surprise token (spec §6.1).

    v3b: No more "ambiguous" bucket.

    The key insight: if CE is high but NO hierarchical threshold is crossed,
    the model's internal processing is fine — it just can't map to the
    specific output token. This is definitionally a vocabulary/output issue.
    Training strategy: standard next-token CE loss (same as surface vocab gap).

    Classification logic:
        high CE + rare token               → vocabulary_gap (rare_token)
        high CE + hi_surf only             → vocabulary_gap (surface)
        high CE + lo_surf + hi_struct/sem  → undergeneralization
        high CE + hi_surf + hi_struct/sem  → out_of_distribution
        high CE + nothing crosses          → vocabulary_gap (output_only)

    Categories (for System 2/3):
        vocabulary_gap:        Train with standard CE
        undergeneralization:   Train with weighted CE + structural emphasis
        out_of_distribution:   Train with CE + label smoothing + high EWC
    """

    categories: Dict[str, List[Tuple[int, int]]] = {
        "vocabulary_gap": [],
        "undergeneralization": [],
        "out_of_distribution": [],
    }
    n_well_understood = 0

    # Sub-type tracking for diagnostics
    subtype_counts = {
        "surface": 0,       # High surface error only
        "output_only": 0,   # No hierarchical signal (proper nouns etc)
        "rare_token": 0,    # Rarity-routed
    }

    for data_idx, passage in enumerate(tracker.passage_data):
        for t_pos, sv in enumerate(passage.vectors):
            if sv.ce_loss < thresholds.ce:
                if sv.ce_loss > 0:
                    n_well_understood += 1
                continue

            # Check token rarity first
            token_id = passage.input_ids[min(t_pos + 1, len(passage.input_ids) - 1)].item()
            token_count = tracker.token_counts.get(token_id, 0)
            is_rare = token_count <= rarity_count_threshold

            if is_rare:
                categories["vocabulary_gap"].append((data_idx, t_pos))
                subtype_counts["rare_token"] += 1
                continue

            # Hierarchical classification
            hi_surf = sv.surface > thresholds.surface
            hi_struct = sv.structural > thresholds.structural
            hi_sem = sv.semantic > thresholds.semantic

            if not hi_surf and (hi_struct or hi_sem):
                # Low surface + high deep = model recognizes surface form
                # but fails at structural/semantic level.
                # THIS IS THE HIGHEST-VALUE TRAINING TARGET.
                categories["undergeneralization"].append((data_idx, t_pos))

            elif hi_surf and (hi_struct or hi_sem):
                # Everything is failing — genuinely novel content.
                categories["out_of_distribution"].append((data_idx, t_pos))

            elif hi_surf and not hi_struct and not hi_sem:
                # Only surface processing fails — rare surface form.
                categories["vocabulary_gap"].append((data_idx, t_pos))
                subtype_counts["surface"] += 1

            else:
                # High CE, no hierarchical signal.
                # Internal processing is normal — the model "knows" what
                # kind of token should come next but can't predict the
                # specific one. Pure output vocabulary issue.
                # Examples: proper nouns, specific names, dates, numbers.
                # Training: standard CE loss.
                categories["vocabulary_gap"].append((data_idx, t_pos))
                subtype_counts["output_only"] += 1

    # ---- Report ----
    total_classified = sum(len(v) for v in categories.values())
    total_tokens = total_classified + n_well_understood

    print(f"\nSURPRISE CLASSIFICATION:")
    print(f"  Total tokens (nonzero): {total_tokens}")
    print(f"  Well understood:        {n_well_understood} "
          f"({n_well_understood/max(total_tokens,1)*100:.1f}%)")

    for cat, events in categories.items():
        pct = len(events) / max(total_tokens, 1) * 100
        print(f"  {cat:25s}: {len(events):6d} ({pct:.1f}%)")

    # Sub-type breakdown for vocabulary_gap
    print(f"\n  vocabulary_gap breakdown:")
    for subtype, count in subtype_counts.items():
        print(f"    {subtype:15s}: {count:6d}")

    # Distribution check
    pcts = {cat: len(events) / max(total_tokens, 1) * 100 for cat, events in categories.items()}
    if pcts["undergeneralization"] < 1:
        print(f"\n  ⚠ Undergeneralization low ({pcts['undergeneralization']:.1f}%). "
              f"Consider lowering structural/semantic thresholds.")
    if pcts["undergeneralization"] > 10:
        print(f"\n  ⚠ Undergeneralization high ({pcts['undergeneralization']:.1f}%). "
              f"Consider raising structural/semantic thresholds.")

    return categories

In [ ]:
# ╔═══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 11: VISUALIZATION (4-panel diagnostic plot)                        ║
# ╚═══════════════════════════════════════════════════════════════════════════╝

def plot_system1_diagnostics(
    tracker: HierarchicalSurpriseTracker,
    thresholds: ClassificationThresholds,
    passage_idx: Optional[int] = None,
    save_path: str = "system1_diagnostics.png",
):
    """
    4-panel visualization:
        Top left:     CE loss distribution with threshold
        Top right:    Layerwise normalized error distributions
        Bottom left:  Surface vs structural scatter (color = CE)
        Bottom right: Single passage with all surprise curves overlaid
    """
    fig, axes = plt.subplots(2, 2, figsize=(16, 11))

    # ---- Panel 1: CE distribution ----
    nz_ce = [v for v in tracker.all_ce if v > 0]
    axes[0, 0].hist(nz_ce, bins=150, alpha=0.7, color="steelblue", density=True)
    axes[0, 0].axvline(thresholds.ce, color="red", linestyle="--", linewidth=2,
                       label=f"P85 = {thresholds.ce:.2f}")
    axes[0, 0].set_title("CE Loss Distribution", fontsize=12)
    axes[0, 0].set_xlabel("Cross-entropy loss")
    axes[0, 0].legend()
    axes[0, 0].set_xlim(0, np.percentile(nz_ce, 99))

    # ---- Panel 2: Layerwise error distributions ----
    colors = {"surface": "#e74c3c", "structural": "#2ecc71", "semantic": "#3498db"}
    for name in ["surface", "structural", "semantic"]:
        nz = [v for v in tracker.all_errors[name] if v > 0]
        if nz:
            axes[0, 1].hist(nz, bins=150, alpha=0.35, color=colors[name],
                           label=name.capitalize(), density=True)
            thresh = getattr(thresholds, name)
            axes[0, 1].axvline(thresh, color=colors[name], linestyle="--",
                              linewidth=1.5, alpha=0.8)
    axes[0, 1].set_title("Normalized Prediction Errors by Level", fontsize=12)
    axes[0, 1].set_xlabel("Normalized error")
    axes[0, 1].legend()

    # ---- Panel 3: Surface vs Structural scatter ----
    surf = np.array(tracker.all_errors["surface"])
    struct = np.array(tracker.all_errors["structural"])
    ce = np.array(tracker.all_ce)

    # Subsample for readability
    n_plot = min(30000, len(surf))
    idx = np.random.choice(len(surf), n_plot, replace=False)
    s_sub, st_sub, ce_sub = surf[idx], struct[idx], ce[idx]

    # Filter out exact zeros
    mask = (s_sub != 0) & (st_sub != 0)
    sc = axes[1, 0].scatter(
        s_sub[mask], st_sub[mask], c=ce_sub[mask],
        cmap="hot", alpha=0.15, s=1, rasterized=True
    )
    axes[1, 0].axvline(thresholds.surface, color="#e74c3c", linestyle="--",
                       alpha=0.5, label="Surface thresh")
    axes[1, 0].axhline(thresholds.structural, color="#2ecc71", linestyle="--",
                       alpha=0.5, label="Structural thresh")
    axes[1, 0].set_xlabel("Surface Error (normalized)")
    axes[1, 0].set_ylabel("Structural Error (normalized)")
    axes[1, 0].set_title("Surface vs Structural (color = CE)", fontsize=12)
    axes[1, 0].legend(fontsize=8)
    plt.colorbar(sc, ax=axes[1, 0], label="CE loss")

    # ---- Panel 4: Single passage surprise curves ----
    # Pick a passage with high variance in structural error (most interesting)
    if passage_idx is None:
        variances = []
        for pd in tracker.passage_data:
            struct_vals = [sv.structural for sv in pd.vectors if sv.structural != 0]
            variances.append(np.var(struct_vals) if len(struct_vals) > 20 else 0)
        passage_idx = int(np.argmax(variances)) if variances else 0

    if passage_idx < len(tracker.passage_data):
        pd = tracker.passage_data[passage_idx]
        n_tok = len(pd.vectors)
        x = range(n_tok)

        axes[1, 1].plot(x, [sv.ce_loss for sv in pd.vectors],
                       alpha=0.4, color="gray", label="CE loss", linewidth=0.8)
        axes[1, 1].plot(x, [sv.surface for sv in pd.vectors],
                       alpha=0.8, color="#e74c3c", label="Surface", linewidth=1.2)
        axes[1, 1].plot(x, [sv.structural for sv in pd.vectors],
                       alpha=0.8, color="#2ecc71", label="Structural", linewidth=1.2)
        axes[1, 1].plot(x, [sv.semantic for sv in pd.vectors],
                       alpha=0.8, color="#3498db", label="Semantic", linewidth=1.2)

        axes[1, 1].axhline(thresholds.structural, color="#2ecc71", linestyle=":",
                           alpha=0.3)
        axes[1, 1].set_title(f"Passage {pd.passage_idx} — Layerwise Surprise",
                            fontsize=12)
        axes[1, 1].set_xlabel("Token position")
        axes[1, 1].set_ylabel("Normalized error")
        axes[1, 1].legend(fontsize=8)

    plt.suptitle("System 1: Predictive Coding Diagnostics", fontsize=14, y=1.01)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved to {save_path}")

In [ ]:
# ╔═══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 12: INSPECTION TOOLS                                              ║
# ╚═══════════════════════════════════════════════════════════════════════════╝

def inspect_surprise_categories(
    tracker: HierarchicalSurpriseTracker,
    categories: Dict[str, List[Tuple[int, int]]],
    corpus: List[str],
    n_per_category: int = 8,
):
    """
    Show example tokens from each surprise category with context.
    This is the primary qualitative sanity check — look at what the system
    considers undergeneralization vs vocabulary gap vs OOD and verify it
    makes intuitive sense.
    """
    for cat_name, events in categories.items():
        print(f"\n{'=' * 65}")
        print(f"  {cat_name.upper()} ({len(events)} tokens)")
        print(f"{'=' * 65}")

        if not events:
            print("  (none)")
            continue

        # Sort by CE loss descending (most surprising first)
        scored = []
        for data_idx, t_pos in events:
            pd = tracker.passage_data[data_idx]
            if t_pos < len(pd.vectors):
                sv = pd.vectors[t_pos]
                scored.append((data_idx, t_pos, sv))
        scored.sort(key=lambda x: x[2].ce_loss, reverse=True)

        for rank, (data_idx, t_pos, sv) in enumerate(scored[:n_per_category]):
            pd = tracker.passage_data[data_idx]
            ids = pd.input_ids

            # Context window around the surprise token
            ctx_start = max(0, t_pos - 12)
            ctx_end = min(len(ids), t_pos + 5)
            context_tokens = ids[ctx_start:ctx_end]
            context_text = tracker.tokenizer.decode(context_tokens)

            # The actual surprise token (+1 because CE[t] predicts token[t+1])
            surprise_token_id = ids[min(t_pos + 1, len(ids) - 1)]
            surprise_token = tracker.tokenizer.decode([surprise_token_id])

            print(f"\n  #{rank+1} | CE: {sv.ce_loss:.2f} | "
                  f"Surf: {sv.surface:.3f} | Struct: {sv.structural:.3f} | "
                  f"Sem: {sv.semantic:.3f}")
            print(f"  Token: [{surprise_token.strip()}]")
            print(f"  Context: ...{context_text.strip()}")

In [ ]:
# ╔═══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 13 — REPLACE ENTIRELY                                             ║
# ║  RUN SYSTEM 1 (v3)                                                       ║
# ║                                                                          ║
# ║  Changes from v2:                                                        ║
# ║    - Computes rarity threshold after wake phase                          ║
# ║    - Passes rarity_count_threshold to classify_surprise                  ║
# ╚═══════════════════════════════════════════════════════════════════════════╝

def run_system1():
    """
    Complete System 1 pipeline:
        1. Initialize prediction heads
        2. Train heads on frozen base model (with early stopping)
        3. Validate head quality
        4. Run wake phase (hierarchical surprise tracking)
        5. Compute classification thresholds + rarity threshold
        6. Classify all tokens
        7. Visualize diagnostics
        8. Inspect example tokens per category
    """

    print("\n" + "=" * 65)
    print("  SYSTEM 1: PREDICTIVE CODING — FULL PIPELINE (v3)")
    print("=" * 65)
    pipeline_start = time.time()

    # ---- 1. Initialize prediction heads ----
    print("\n[1/8] Initializing prediction heads...")
    head_system = PredictionHeadSystem(
        hidden_size=HIDDEN_DIM,
        bottleneck_factor=0.25,
        use_deltas=True,
    )

    # ---- 2. Train heads ----
    print("\n[2/8] Training prediction heads...")
    head_system = train_prediction_heads(
        base_model=model,
        tokenizer=tokenizer,
        head_system=head_system,
        train_texts=head_train_corpus,
        val_texts=head_val_corpus,
        device=device,
        max_epochs=3,
        lr=1e-3,
        patience=2,
        min_improvement=0.05,
    )

    # ---- 3. Validate heads ----
    print("\n[3/8] Validating head quality...")
    validation_results = validate_heads(
        base_model=model,
        tokenizer=tokenizer,
        head_system=head_system,
        val_texts=head_val_corpus,
        device=device,
    )

    # ---- 4. Wake phase ----
    print("\n[4/8] Running wake phase...")
    tracker = HierarchicalSurpriseTracker(
        base_model=model,
        tokenizer=tokenizer,
        head_system=head_system,
        device=device,
    )
    tracker.run_wake_phase(wake_corpus)

    # ---- 5. Compute thresholds ----
    print("\n[5/8] Computing classification thresholds...")
    thresholds = compute_thresholds(tracker)
    rarity_threshold = compute_rarity_threshold(tracker, rarity_percentile=10)

    # ---- 6. Classify ----
    print("\n[6/8] Classifying surprise tokens...")
    categories = classify_surprise(tracker, thresholds, rarity_count_threshold=rarity_threshold)

    # ---- 7. Visualize ----
    print("\n[7/8] Generating diagnostic plots...")
    plot_system1_diagnostics(tracker, thresholds)

    # ---- 8. Inspect ----
    print("\n[8/8] Inspecting examples per category...")
    inspect_surprise_categories(tracker, categories, wake_corpus)

    # ---- Summary ----
    elapsed = time.time() - pipeline_start
    print(f"\n{'=' * 65}")
    print(f"  SYSTEM 1 COMPLETE (v3) — {elapsed:.1f}s total")
    print(f"  Passages processed: {len(tracker.passage_data)}")
    print(f"  Total tokens tracked: {len(tracker.all_ce)}")
    print(f"  Unique tokens: {len(tracker.token_counts)}")
    print(f"  Rarity threshold: ≤{rarity_threshold} occurrences")
    print(f"  Running stats:")
    for name, _, _ in PredictionHeadSystem.LEVELS:
        print(f"    {name}: {tracker.stats[name]}")
    print(f"{'=' * 65}")

    return tracker, head_system, thresholds, categories, validation_results

In [ ]:
# ╔═══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 14: LAUNCH                                                        ║
# ║  Uncomment and run.                                                      ║
# ╚═══════════════════════════════════════════════════════════════════════════╝

tracker, head_system, thresholds, categories, val_results = run_system1()

In [ ]:
# ╔═══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 15: EPISODE DATA STRUCTURE + ADAPTIVE SURPRISE GATE               ║
# ║  (spec §2.1, §4)                                                        ║
# ║                                                                          ║
# ║  Add this cell after Cell 14 in your notebook.                           ║
# ║  Requires: Cells 1-14 (System 1) already executed.                       ║
# ╚═══════════════════════════════════════════════════════════════════════════╝

from collections import deque, defaultdict
import math

@dataclass
class Episode:
    """
    Atomic unit of episodic memory (spec §2.1).
    Represents a context window around a high-surprise event.
    All tensors stored on CPU to avoid VRAM fragmentation (spec §9.5).
    """
    # ---- Identity ----
    episode_id: int                            # Unique monotonic ID (assigned by buffer)
    created_at: int                            # Step when stored
    passage_idx: int                           # Source corpus passage

    # ---- Content ----
    input_ids: torch.Tensor                    # [T_w] token IDs, CPU
    text: str                                  # Decoded text (inspection only)

    # ---- Surprise Profile ----
    surprise_type: str                         # Primary type (for indexing/routing)
    type_distribution: Dict[str, float]        # Full distribution across types
    peak_ce: float                             # Max CE in window
    mean_ce: float                             # Mean CE in window
    peak_position: int                         # Token position of peak surprise (within window)
    surprise_vectors: List[SurpriseVector]     # Per-token 4D surprise within window

    # ---- Retrieval Key ----
    embedding_key: torch.Tensor                # [D], unit norm, CPU

    # ---- Lifecycle ----
    retrieval_count: int = 0
    last_retrieved: int = -1
    consolidation_score: float = 0.0
    is_consolidated: bool = False

    # ---- Learning Objective (PhysMem-inspired typed hypotheses) ----
    learning_objective: str = "ACQUIRE"        # ACQUIRE | RESTRUCTURE | CONTEXTUALIZE
                                                # | REINFORCE | CORRECT
    objective_confidence: float = 0.0           # 0.0-1.0, increases with successful
                                                # consolidation, decays over time
    consolidation_attempts: int = 0             # How many times Phase 1 has trained on this
    last_consolidated_cycle: int = -1           # Cycle number of last consolidation
    phase2_history: List[str] = field(default_factory=list)
                                                # List of Phase 2 outcomes across cycles
                                                # e.g. ["FRAGILE", "FRAGILE", "GENERALIZED"]

    # ---- Window bounds (for deduplication) ----
    window_start: int = 0                      # Start position in original passage
    window_end: int = 0                        # End position in original passage

def compute_learning_objective(episode: Episode) -> str:
    """
    Determine what the model should learn from this episode.

    Maps from (surprise_type × lifecycle_state) → learning_objective.

    The key insight from PhysMem: untyped replay gets 23% success vs 76%
    for typed principled abstraction. The learning objective tells the
    consolidation loop not just WHAT failed, but WHAT TO DO ABOUT IT.

    Objectives:
        ACQUIRE:        Learn this surface pattern (new vocabulary, names, terms)
                        → High LR, low rank, low EWC, standard CE loss

        RESTRUCTURE:    Modify attention patterns for this construction
                        → Low LR, high rank, high EWC, weighted CE loss

        CONTEXTUALIZE:  Integrate novel domain knowledge carefully
                        → Very low LR, medium rank, very high EWC, label-smoothed CE

        REINFORCE:      Was partially learned but fragile/unstable — needs repetition
                        → Medium LR, medium rank, medium EWC, standard CE
                        → Fewer steps than ACQUIRE (already partially there)

        CORRECT:        Previous consolidation caused interference — undo damage
                        → Very low LR, high EWC, focus on restoring old behavior
                        → Train with both the episode AND high-similarity coreset
                          passages to anchor the correction
    """

    # Priority 1: If Phase 2 found interference, objective is CORRECT
    if episode.phase2_history and episode.phase2_history[-1] == "INTERFERENCE":
        return "CORRECT"

    # Priority 2: If previously consolidated but now fragile/memorized,
    # objective is REINFORCE
    if episode.consolidation_attempts > 0:
        if episode.phase2_history:
            last_outcome = episode.phase2_history[-1]
            if last_outcome in ("FRAGILE", "MEMORIZED", "RIGID"):
                return "REINFORCE"
            if last_outcome == "GENERALIZED":
                # Already learned — no further objective needed
                # (eviction policy will handle removal)
                return "REINFORCE"  # Light touch to maintain

        # Trained before but no Phase 2 data — check consolidation score
        if episode.consolidation_score > 0.3:
            return "REINFORCE"

    # Priority 3: First-time episodes — objective from surprise type
    objective_map = {
        "vocabulary_gap":       "ACQUIRE",
        "undergeneralization":  "RESTRUCTURE",
        "out_of_distribution":  "CONTEXTUALIZE",
    }
    return objective_map.get(episode.surprise_type, "ACQUIRE")


def update_episode_objectives(buffer_episodes: Dict[int, Episode]):
    """
    Batch-update all episode learning objectives.
    Call after Phase 2 diagnosis or at the start of each cycle.
    """
    for ep in buffer_episodes.values():
        if not ep.is_consolidated:
            ep.learning_objective = compute_learning_objective(ep)

class AdaptiveSurpriseGate:
    """
    Decides which surprise events warrant episodic storage (spec §4.2).

    Uses EMA-tracked statistics with a composite criterion:
        1. CE must exceed adaptive threshold (necessary)
        2. At least one hierarchical error must exceed its threshold
           (filters output-only vocab gap noise)

    Self-regulates admission rate to [min_rate, max_rate].

    z_threshold controls selectivity:
        z=1.0 → ~16% admission, z=1.5 → ~7%, z=2.0 → ~2.5%
    """

    def __init__(
        self,
        momentum: float = 0.95,
        z_threshold: float = 1.5,
        min_admission_rate: float = 0.05,
        max_admission_rate: float = 0.25,
    ):
        self.momentum = momentum
        self.z_threshold = z_threshold
        self.min_rate = min_admission_rate
        self.max_rate = max_admission_rate

        self.ema_mean: Dict[str, Optional[float]] = {
            "ce": None, "surface": None, "structural": None, "semantic": None,
        }
        self.ema_var: Dict[str, Optional[float]] = {
            "ce": None, "surface": None, "structural": None, "semantic": None,
        }

        self.recent_admissions: deque = deque(maxlen=10000)
        self.total_evaluated: int = 0
        self.total_admitted: int = 0

    def _update_ema(self, key: str, value: float):
        """Update EMA mean and variance for one dimension (spec §4.3)."""
        m = self.momentum
        if self.ema_mean[key] is None:
            self.ema_mean[key] = value
            self.ema_var[key] = 0.0
        else:
            delta = value - self.ema_mean[key]
            self.ema_mean[key] = m * self.ema_mean[key] + (1 - m) * value
            self.ema_var[key] = m * self.ema_var[key] + (1 - m) * delta * (value - self.ema_mean[key])
            self.ema_var[key] = max(self.ema_var[key], 1e-10)

    def _get_threshold(self, key: str) -> float:
        """Threshold = mean + z * std."""
        if self.ema_mean[key] is None:
            return float('-inf')  # Uninitialized → admit everything
        return self.ema_mean[key] + self.z_threshold * math.sqrt(self.ema_var[key] + 1e-8)

    def _admission_rate(self) -> float:
        if len(self.recent_admissions) < 100:
            return 0.5
        return sum(self.recent_admissions) / len(self.recent_admissions)

    def should_store(self, sv: SurpriseVector) -> bool:
        """
        Decide whether a token warrants episodic storage (spec §4.2).

        Composite criterion:
            Gate 1: CE > adaptive CE threshold
            Gate 2: At least one of {surface, structural, semantic} > its threshold
        """
        if sv.ce_loss <= 0:
            return False

        # Update running stats for all dimensions
        self._update_ema("ce", sv.ce_loss)
        self._update_ema("surface", sv.surface)
        self._update_ema("structural", sv.structural)
        self._update_ema("semantic", sv.semantic)
        self.total_evaluated += 1

        # Gate 1: CE must exceed threshold
        if sv.ce_loss < self._get_threshold("ce"):
            self.recent_admissions.append(0)
            return False

        # Gate 2: At least one hierarchical level must be surprising
        hier_surprise = (
            sv.surface > self._get_threshold("surface") or
            sv.structural > self._get_threshold("structural") or
            sv.semantic > self._get_threshold("semantic")
        )
        if not hier_surprise:
            self.recent_admissions.append(0)
            return False

        # Self-regulation: if admitting too much, require extra surprise
        if self._admission_rate() > self.max_rate:
            extra_ce = self._get_threshold("ce") + self.z_threshold * math.sqrt(
                self.ema_var["ce"] + 1e-8
            )
            if sv.ce_loss < extra_ce:
                self.recent_admissions.append(0)
                return False

        self.recent_admissions.append(1)
        self.total_admitted += 1
        return True

    def warmup_from_tracker(self, tracker: HierarchicalSurpriseTracker, n_passages: int = 200):
        """
        Initialize gate statistics from System 1's stored surprise vectors
        WITHOUT making storage decisions (spec §9.4).
        """
        print(f"Warming up gate on {min(n_passages, len(tracker.passage_data))} passages...")
        count = 0
        for pd in tracker.passage_data[:n_passages]:
            for sv in pd.vectors:
                if sv.ce_loss <= 0:
                    continue
                self._update_ema("ce", sv.ce_loss)
                self._update_ema("surface", sv.surface)
                self._update_ema("structural", sv.structural)
                self._update_ema("semantic", sv.semantic)
                count += 1

        print(f"  Gate warmed on {count} tokens")
        for key in self.ema_mean:
            if self.ema_mean[key] is not None:
                thresh = self._get_threshold(key)
                print(f"  {key:12s}: μ={self.ema_mean[key]:.4f}, "
                      f"σ={math.sqrt(self.ema_var[key]):.4f}, "
                      f"thresh={thresh:.4f}")

    def __repr__(self):
        rate = self._admission_rate()
        return (f"AdaptiveSurpriseGate(z={self.z_threshold}, "
                f"rate={rate:.3f}, eval={self.total_evaluated}, "
                f"admitted={self.total_admitted})")

In [ ]:
# ╔═══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 16: BRUTE FORCE INDEX + EPISODIC BUFFER                           ║
# ║  (spec §8, §9.1)                                                        ║
# ╚═══════════════════════════════════════════════════════════════════════════╝

class BruteForceIndex:
    """
    Cosine similarity search over unit-normalized embeddings (spec §9.1).
    At <5k episodes, brute force is <1ms on GPU — FAISS not needed.
    """
    def __init__(self):
        self.keys: Dict[int, torch.Tensor] = {}  # episode_id → [D] CPU tensor

    def add(self, episode_id: int, key: torch.Tensor):
        self.keys[episode_id] = key.detach().cpu().clone()

    def remove(self, episode_id: int):
        self.keys.pop(episode_id, None)

    def search(self, query: torch.Tensor, k: int = 10) -> List[Tuple[int, float]]:
        """Returns list of (episode_id, cosine_similarity) sorted descending."""
        if not self.keys:
            return []
        eids = list(self.keys.keys())
        key_matrix = torch.stack([self.keys[eid] for eid in eids])  # [N, D]
        # Cosine sim = dot product for unit vectors
        sims = torch.mv(key_matrix, query.cpu())  # [N]
        topk_k = min(k, len(eids))
        topk_sims, topk_idx = sims.topk(topk_k)
        return [(eids[i], topk_sims[j].item())
                for j, i in enumerate(topk_idx.tolist())]

    def __len__(self):
        return len(self.keys)


class EpisodicBuffer:
    """
    Content-addressable episodic memory with adaptive gating,
    intelligent forgetting, and consolidation scoring (spec §8).

    This is the "hippocampus" — stores surprising episodes rapidly,
    retrieves by semantic similarity, and evicts once consolidated.
    """

    def __init__(self, max_capacity: int = 2000):
        self.episodes: Dict[int, Episode] = {}
        self.max_capacity = max_capacity

        # Indexing
        self.embedding_index = BruteForceIndex()
        self.category_index: Dict[str, set] = defaultdict(set)
        self.passage_index: Dict[int, set] = defaultdict(set)

        # Gate
        self.surprise_gate = AdaptiveSurpriseGate()

        # Counters
        self.next_id: int = 0
        self.total_stored: int = 0
        self.total_evicted: int = 0
        self.current_step: int = 0

        # Eviction log (spec §9.6)
        self.eviction_log: List[dict] = []

    # ---- Store ----

    def store(self, episode: Episode) -> bool:
        """
        Attempt to store an episode (spec §8 store()).
        Returns False if rejected (duplicate or capacity + no eviction candidate).
        """

        # Check for duplicates (spec §7.2)
        if self._is_duplicate(episode):
            return False

        # Check capacity
        if len(self.episodes) >= self.max_capacity:
            evicted = self._try_evict_for(episode)
            if not evicted:
                return False

        # Assign ID and store
        episode.episode_id = self.next_id
        episode.created_at = self.current_step
        self.next_id += 1
        self.total_stored += 1

        self.episodes[episode.episode_id] = episode
        self.embedding_index.add(episode.episode_id, episode.embedding_key)
        self.category_index[episode.surprise_type].add(episode.episode_id)
        self.passage_index[episode.passage_idx].add(episode.episode_id)

        return True

    def _is_duplicate(self, candidate: Episode, threshold: float = 0.92) -> bool:
        """Check for near-duplicates (spec §7.2)."""
        if not self.embedding_index.keys:
            return False

        neighbors = self.embedding_index.search(candidate.embedding_key, k=3)
        for eid, sim in neighbors:
            if sim > threshold:
                existing = self.episodes[eid]
                # If candidate is 10% harder, replace (upgrade)
                if candidate.peak_ce > existing.peak_ce * 1.1:
                    self._remove_episode(eid)
                    return False  # Not a dup — it's an upgrade
                return True  # Genuine duplicate
        return False

    def _try_evict_for(self, candidate: Episode) -> bool:
        """Evict the most eligible episode to make room, if worthwhile."""
        priorities = [
            (eid, self._eviction_priority(ep))
            for eid, ep in self.episodes.items()
        ]
        worst_eid, worst_priority = max(priorities, key=lambda x: x[1])
        worst_ep = self.episodes[worst_eid]

        # Only evict if: worst is consolidated, OR candidate is more surprising
        if worst_ep.consolidation_score < 0.3 and candidate.peak_ce <= worst_ep.peak_ce:
            return False  # Worst is unconsolidated AND more surprising → reject candidate

        self._remove_episode(worst_eid)
        return True

    def _remove_episode(self, eid: int):
        """Remove an episode from all indices."""
        if eid not in self.episodes:
            return
        ep = self.episodes.pop(eid)
        self.embedding_index.remove(eid)
        self.category_index[ep.surprise_type].discard(eid)
        self.passage_index[ep.passage_idx].discard(eid)
        self.total_evicted += 1
        self.eviction_log.append({
            "step": self.current_step,
            "evicted_id": eid,
            "consolidation_score": ep.consolidation_score,
            "surprise_type": ep.surprise_type,
            "peak_ce": ep.peak_ce,
        })

    def _eviction_priority(self, ep: Episode) -> float:
        """
        Higher = more likely to be evicted (spec §6.3).
        Balances consolidation score, age, redundancy, and type value.
        """
        # Factor 1: Consolidation (most important — learned episodes go first)
        f1 = ep.consolidation_score * 3.0

        # Factor 2: Age
        age = max(self.current_step - ep.created_at, 0)
        f2 = (age / 10000) * 0.5

        # Factor 3: Redundancy proxy (frequently retrieved = well-represented)
        f3 = math.log1p(ep.retrieval_count) * 1.0

        # Factor 4: Type penalty (undergeneralization = most valuable, penalize eviction)
                # Factor 4: Objective-aware eviction penalty
        # CORRECT episodes are highest priority to KEEP (they fix damage)
        # RESTRUCTURE is most valuable learning target
        # ACQUIRE is easiest to learn, most disposable
        objective_penalties = {
            "ACQUIRE":       0.0,    # Easy to learn, first to evict
            "REINFORCE":     0.5,    # Partially learned, moderate protection
            "CONTEXTUALIZE": 1.0,    # Novel domain, worth protecting
            "RESTRUCTURE":   1.5,    # Highest-value learning, strong protection
            "CORRECT":       2.0,    # Fixing damage — never evict these
        }
        f4 = -objective_penalties.get(ep.learning_objective, 0.0)

        # Deterministic tie-breaking (spec §9.6)
        return f1 + f2 + f3 + f4 + (ep.episode_id * 1e-10)

    # ---- Retrieve ----

    def retrieve_similar(
        self, query_key: torch.Tensor, k: int = 10,
        category_filter: Optional[str] = None,
    ) -> List[Tuple[Episode, float]]:
        """Find k most similar episodes (spec §7.1)."""
        if category_filter:
            candidate_eids = self.category_index.get(category_filter, set())
            if not candidate_eids:
                return []
            # Filter search to category
            results = []
            for eid in candidate_eids:
                ep = self.episodes[eid]
                sim = torch.dot(query_key.cpu(), ep.embedding_key).item()
                results.append((ep, sim))
            results.sort(key=lambda x: x[1], reverse=True)
            results = results[:k]
        else:
            neighbors = self.embedding_index.search(query_key, k=k)
            results = [(self.episodes[eid], sim) for eid, sim in neighbors
                       if eid in self.episodes]

        # Update retrieval counts
        for ep, sim in results:
            ep.retrieval_count += 1
            ep.last_retrieved = self.current_step

        return results

    # ---- Consolidation interface ----

    def retrieve_for_consolidation(
        self, n_episodes: int = 500, strategy: str = "priority",
    ) -> List[Episode]:
        """
        Sample episodes for System 3 sleep phase (spec §8 retrieve_for_consolidation).

        Strategies:
            "balanced":  Equal per type, hardest first
            "priority":  Weighted by (1 - consolidation_score)
            "curriculum": Partially consolidated first, then unconsolidated
        """
        unconsolidated = [
            ep for ep in self.episodes.values() if not ep.is_consolidated
        ]
        if not unconsolidated:
            return []

        n = min(n_episodes, len(unconsolidated))

        if strategy == "balanced":
            per_cat = max(1, n // max(len(self.category_index), 1))
            selected = []
            for cat, eids in self.category_index.items():
                cat_eps = [self.episodes[eid] for eid in eids
                           if not self.episodes[eid].is_consolidated]
                cat_eps.sort(key=lambda e: e.peak_ce, reverse=True)
                selected.extend(cat_eps[:per_cat])
            return selected[:n]

        elif strategy == "priority":
            weights = np.array([1 - ep.consolidation_score + 0.1 for ep in unconsolidated])
            probs = weights / weights.sum()
            indices = np.random.choice(len(unconsolidated), size=n, replace=False, p=probs)
            return [unconsolidated[i] for i in indices]

        elif strategy == "curriculum":
            partial = [ep for ep in unconsolidated if 0.3 <= ep.consolidation_score < 0.7]
            hard = [ep for ep in unconsolidated if ep.consolidation_score < 0.3]
            half = n // 2
            return (partial[:half] + hard[:n - half])[:n]

        else:
            raise ValueError(f"Unknown strategy: {strategy}")

    @torch.no_grad()
    def update_consolidation_scores(self, model: nn.Module, device: torch.device):
        """
        Re-score all unconsolidated episodes after a sleep phase (spec §6.1).
        Measures how much the model's surprise decreased on each episode.
        """
        print("Updating consolidation scores...")
        model.eval()
        scored = 0

        for eid, ep in self.episodes.items():
            if ep.is_consolidated:
                continue

            ids = ep.input_ids.unsqueeze(0).to(device)
            outputs = model(ids, labels=ids)
            logits = outputs.logits[0, :-1, :].float()
            labels = ep.input_ids[1:]
            new_ce = F.cross_entropy(logits, labels.to(device), reduction='none')

            # Score: weighted peak + mean reduction (spec §6.1)
            peak = ep.peak_position
            if peak < len(new_ce):
                peak_reduction = (ep.peak_ce - new_ce[peak].item()) / max(ep.peak_ce, 1e-8)
                mean_reduction = (ep.mean_ce - new_ce.mean().item()) / max(ep.mean_ce, 1e-8)
                new_score = 0.6 * max(0, min(1, peak_reduction)) + \
                            0.4 * max(0, min(1, mean_reduction))
            else:
                new_score = 0.0

            old_score = ep.consolidation_score
            ep.consolidation_score = new_score

            # State transition (spec §6.2): must be ≥0.7 for TWO consecutive cycles
            if new_score >= 0.7 and old_score >= 0.7:
                ep.is_consolidated = True

            scored += 1

        # Report
        scores = [ep.consolidation_score for ep in self.episodes.values()]
        n_uncon = sum(1 for s in scores if s < 0.3)
        n_partial = sum(1 for s in scores if 0.3 <= s < 0.7)
        n_full = sum(1 for s in scores if s >= 0.7)
        n_evictable = sum(1 for ep in self.episodes.values() if ep.is_consolidated)

        print(f"  Scored {scored} episodes")
        print(f"  Unconsolidated (<0.3):  {n_uncon}")
        print(f"  Partial (0.3-0.7):      {n_partial}")
        print(f"  Full (≥0.7):            {n_full}")
        print(f"  Ready to evict:         {n_evictable}")

    def evict_consolidated(self) -> int:
        """Remove all episodes marked is_consolidated. Returns count evicted."""
        to_evict = [eid for eid, ep in self.episodes.items() if ep.is_consolidated]
        for eid in to_evict:
            self._remove_episode(eid)
        return len(to_evict)

    # ---- Statistics ----

    def get_statistics(self) -> Dict:
        """Comprehensive buffer stats (spec §8 get_statistics)."""
        if not self.episodes:
            return {"empty": True, "current_size": 0}

        types = Counter(ep.surprise_type for ep in self.episodes.values())
        scores = [ep.consolidation_score for ep in self.episodes.values()]
        peak_ces = [ep.peak_ce for ep in self.episodes.values()]

        return {
            "total_stored": self.total_stored,
            "total_evicted": self.total_evicted,
            "current_size": len(self.episodes),
            "capacity_pct": len(self.episodes) / self.max_capacity * 100,
            "by_type": dict(types),
            "mean_consolidation_score": float(np.mean(scores)),
            "mean_peak_ce": float(np.mean(peak_ces)),
            "max_peak_ce": float(np.max(peak_ces)),
            "admission_rate": self.surprise_gate._admission_rate(),
        }

In [ ]:
# ╔═══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 17: EPISODE EXTRACTION PIPELINE                                   ║
# ║  (spec §3.2, §5, §5.2)                                                  ║
# ║                                                                          ║
# ║  Bridges token-level surprise (System 1) to episodes (System 2):         ║
# ║    admitted tokens → cluster into regions → extract windows →            ║
# ║    compute retrieval keys → create Episode objects → deduplicate          ║
# ╚═══════════════════════════════════════════════════════════════════════════╝

def build_token_type_lookup(
    categories: Dict[str, List[Tuple[int, int]]],
) -> Dict[Tuple[int, int], str]:
    """
    Build reverse lookup from System 1's classification output.
    Maps (passage_data_index, token_position) → surprise category.
    """
    lookup = {}
    for cat_name, events in categories.items():
        for (data_idx, t_pos) in events:
            lookup[(data_idx, t_pos)] = cat_name
    return lookup


@torch.no_grad()
def compute_episode_key(
    model: nn.Module,
    input_ids: torch.Tensor,
    peak_position: int,
    device: torch.device,
    key_layer: int = 11,
) -> torch.Tensor:
    """
    Extract a D-dimensional retrieval key from model hidden states (spec §3.2).

    Uses the middle layer (layer 11) at the peak surprise position.
    Returns unit-normalized vector on CPU.
    """
    ids = input_ids.unsqueeze(0).to(device) if input_ids.dim() == 1 else input_ids.to(device)
    outputs = model(ids, output_hidden_states=True)

    # hidden_states[l+1] = output of transformer layer l
    h = outputs.hidden_states[key_layer + 1]  # [1, T, D]
    peak_pos = min(peak_position, h.shape[1] - 1)
    h_peak = h[0, peak_pos, :].float()  # [D]

    key = h_peak / (h_peak.norm() + 1e-8)
    return key.detach().cpu().clone()


def adaptive_window_size(surprise_type: str, base_size: int = 512) -> int:
    """Different window sizes per type (spec §5.3)."""
    multipliers = {
        "vocabulary_gap": 0.5,        # 256
        "undergeneralization": 1.0,   # 512
        "out_of_distribution": 1.0,   # 512
    }
    return int(base_size * multipliers.get(surprise_type, 1.0))


def extract_episodes_from_passage(
    passage_data: PassageSurprise,
    data_idx: int,
    gate: AdaptiveSurpriseGate,
    type_lookup: Dict[Tuple[int, int], str],
    model: nn.Module,
    tokenizer,
    device: torch.device,
    base_window_size: int = 512,
    min_gap: int = 64,
    key_layer: int = 11,
) -> List[Episode]:
    """
    Extract Episode objects from one passage (spec §5.1).

    Flow:
        1. Feed each surprise vector through gate → admitted positions
        2. Cluster nearby admitted tokens into regions (min_gap apart)
        3. For each region: find peak, determine type, extract window,
           compute retrieval key, build Episode
    """
    vectors = passage_data.vectors
    input_ids = passage_data.input_ids  # [T] on CPU

    if len(vectors) < 30:
        return []

    # ---- Step 1: Gate ----
    admitted = []
    for t_pos, sv in enumerate(vectors):
        if gate.should_store(sv):
            admitted.append(t_pos)

    if not admitted:
        return []

    # ---- Step 2: Cluster into regions ----
    regions = []
    current_region = [admitted[0]]
    for pos in admitted[1:]:
        if pos - current_region[-1] <= min_gap:
            current_region.append(pos)
        else:
            regions.append(current_region)
            current_region = [pos]
    regions.append(current_region)

    # ---- Step 3: Create episodes ----
    episodes = []
    for region in regions:
        # Find peak CE in region
        ce_values = [vectors[t].ce_loss for t in region]
        peak_idx_in_region = int(np.argmax(ce_values))
        peak_position = region[peak_idx_in_region]

        # Determine dominant surprise type from System 1's classification
        type_counts = Counter()
        for t in region:
            cat = type_lookup.get((data_idx, t), "vocabulary_gap")
            type_counts[cat] += 1
        dominant_type = type_counts.most_common(1)[0][0]

        # Type distribution
        total_in_region = sum(type_counts.values())
        type_dist = {k: v / total_in_region for k, v in type_counts.items()}

        # Window size depends on type (spec §5.3)
        window_size = adaptive_window_size(dominant_type, base_window_size)

        # Extract window centered on peak
        T = len(input_ids)
        window_start = max(0, peak_position - window_size // 2)
        window_end = min(T, window_start + window_size)
        window_start = max(0, window_end - window_size)  # Re-adjust near boundaries

        window_ids = input_ids[window_start:window_end].detach().cpu().clone()
        peak_in_window = peak_position - window_start

        # Surprise vectors within window (aligned: vectors[t] corresponds to position t)
        sv_start = max(0, window_start)
        sv_end = min(len(vectors), window_end - 1)  # -1 for next-token shift
        window_svs = vectors[sv_start:sv_end]

        # CE statistics
        window_ces = [sv.ce_loss for sv in window_svs if sv.ce_loss > 0]
        if not window_ces:
            continue

        # Compute retrieval key (spec §3.2)
        key = compute_episode_key(model, window_ids, peak_in_window, device, key_layer)

        episode = Episode(
            episode_id=-1,  # Assigned by buffer
            created_at=-1,  # Assigned by buffer
            passage_idx=passage_data.passage_idx,
            input_ids=window_ids,
            text=tokenizer.decode(window_ids, skip_special_tokens=True)[:300],
            surprise_type=dominant_type,
            type_distribution=type_dist,
            peak_ce=float(max(window_ces)),
            mean_ce=float(np.mean(window_ces)),
            peak_position=peak_in_window,
            surprise_vectors=window_svs,
            embedding_key=key,
            window_start=window_start,
            window_end=window_end,
        )
        episodes.append(episode)

    return episodes


def deduplicate_episodes(episodes: List[Episode], max_overlap: float = 0.5) -> List[Episode]:
    """
    Remove within-passage overlapping episodes (spec §5.2).
    Keeps the most surprising when windows overlap >50%.
    """
    if len(episodes) <= 1:
        return episodes

    episodes.sort(key=lambda e: e.peak_ce, reverse=True)
    kept = []

    for candidate in episodes:
        overlaps = False
        for existing in kept:
            if candidate.passage_idx != existing.passage_idx:
                continue
            overlap_start = max(candidate.window_start, existing.window_start)
            overlap_end = min(candidate.window_end, existing.window_end)
            overlap = max(0, overlap_end - overlap_start)
            window_len = candidate.window_end - candidate.window_start
            if window_len > 0 and overlap / window_len > max_overlap:
                overlaps = True
                break
        if not overlaps:
            kept.append(candidate)

    return kept

In [ ]:
# ╔═══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 18: INGESTION PIPELINE (System 1 → System 2)                      ║
# ║                                                                          ║
# ║  Main function: takes System 1 outputs, populates the EpisodicBuffer.    ║
# ╚═══════════════════════════════════════════════════════════════════════════╝

def ingest_system1_to_buffer(
    tracker: HierarchicalSurpriseTracker,
    categories: Dict[str, List[Tuple[int, int]]],
    buffer: EpisodicBuffer,
    model: nn.Module,
    tokenizer,
    device: torch.device,
    warmup_passages: int = 200,
    key_layer: int = 11,
    log_interval: int = 500,
) -> EpisodicBuffer:
    """
    Full System 1 → System 2 ingestion pipeline.

    Steps:
        1. Warm up the adaptive gate from System 1's surprise statistics
        2. Build type lookup from System 1's classification
        3. For each passage: gate → cluster → window → key → Episode → store
        4. Report statistics
    """
    print("=" * 65)
    print("SYSTEM 2: INGESTING EPISODES FROM SYSTEM 1")
    print(f"  Passages to process: {len(tracker.passage_data)}")
    print(f"  Buffer capacity: {buffer.max_capacity}")
    print("=" * 65)

    start_time = time.time()

    # ---- 1. Warm up gate ----
    buffer.surprise_gate.warmup_from_tracker(tracker, n_passages=warmup_passages)

    # ---- 2. Type lookup ----
    type_lookup = build_token_type_lookup(categories)
    print(f"  Type lookup built: {len(type_lookup)} classified tokens")

    # ---- 3. Extract and store ----
    n_stored = 0
    n_rejected = 0
    n_episodes_extracted = 0

    for i, pd in enumerate(tqdm(tracker.passage_data, desc="Extracting episodes")):
        episodes = extract_episodes_from_passage(
            passage_data=pd,
            data_idx=i,
            gate=buffer.surprise_gate,
            type_lookup=type_lookup,
            model=model,
            tokenizer=tokenizer,
            device=device,
            key_layer=key_layer,
        )

        episodes = deduplicate_episodes(episodes)
        n_episodes_extracted += len(episodes)

        for ep in episodes:
            if buffer.store(ep):
                n_stored += 1
            else:
                n_rejected += 1

        if (i + 1) % log_interval == 0:
            print(f"  [{i+1}] Extracted: {n_episodes_extracted}, "
                  f"Stored: {n_stored}, Rejected: {n_rejected}, "
                  f"Buffer: {len(buffer.episodes)}/{buffer.max_capacity}, "
                  f"Gate rate: {buffer.surprise_gate._admission_rate():.3f}")

    elapsed = time.time() - start_time
    print(f"\nIngestion complete in {elapsed:.1f}s")
    print(f"  Episodes extracted: {n_episodes_extracted}")
    print(f"  Stored: {n_stored}, Rejected: {n_rejected}")
    print(f"  Buffer size: {len(buffer.episodes)}/{buffer.max_capacity}")
    print(f"  Gate: {buffer.surprise_gate}")

    return buffer

In [ ]:
# ╔═══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 19: CONSOLIDATION BATCH + RE-KEYING                               ║
# ║  (spec §10.2, §9.2)                                                     ║
# ║                                                                          ║
# ║  Interface between System 2 and System 3.                                ║
# ╚═══════════════════════════════════════════════════════════════════════════╝

@dataclass
class ConsolidationBatch:
    """Payload from System 2 → System 3 (spec §10.2)."""
    episodes: List[Episode]
    routing: Dict[str, List[int]]     # learning_objective → list of indices
    coreset_episodes: List[Episode]   # Consolidated episodes for stability replay


def prepare_consolidation_batch(
    buffer: EpisodicBuffer,
    n_episodes: int = 500,
    coreset_ratio: float = 0.1,
    strategy: str = "priority",
) -> ConsolidationBatch:
    """
    Select and route episodes for System 3 (spec §10.2).
    """
    episodes = buffer.retrieve_for_consolidation(n_episodes, strategy=strategy)

    # Route by learning objective (not just surprise type)
    routing = defaultdict(list)
    for i, ep in enumerate(episodes):
        ep.learning_objective = compute_learning_objective(ep)
        routing[ep.learning_objective].append(i)

    # Coreset: recently consolidated episodes for replay
    consolidated = [
        ep for ep in buffer.episodes.values()
        if ep.is_consolidated or ep.consolidation_score >= 0.7
    ]
    n_coreset = int(n_episodes * coreset_ratio)
    if consolidated and n_coreset > 0:
        coreset = random.sample(consolidated, min(n_coreset, len(consolidated)))
    else:
        coreset = []

    print(f"Consolidation batch prepared:")
    print(f"  Episodes: {len(episodes)}")
    for objective, indices in routing.items():
        print(f"    {objective}: {len(indices)}")
    print(f"  Coreset: {len(coreset)}")

    return ConsolidationBatch(
        episodes=episodes,
        routing=dict(routing),
        coreset_episodes=coreset,
    )


@torch.no_grad()
def rekey_buffer(
    buffer: EpisodicBuffer,
    model: nn.Module,
    device: torch.device,
    key_layer: int = 11,
):
    """
    Re-compute all episode keys after model update (spec §9.2, Option A).
    Cost: ~2 min for 2000 episodes on A100.
    """
    print(f"Re-keying {len(buffer.episodes)} episodes...")
    model.eval()
    start = time.time()

    for eid, ep in tqdm(buffer.episodes.items(), desc="Re-keying"):
        new_key = compute_episode_key(model, ep.input_ids, ep.peak_position, device, key_layer)
        ep.embedding_key = new_key
        buffer.embedding_index.keys[eid] = new_key

    print(f"  Re-keyed in {time.time() - start:.1f}s")

In [ ]:
# ╔═══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 20: BUFFER DIAGNOSTICS & VISUALIZATION                            ║
# ╚═══════════════════════════════════════════════════════════════════════════╝

def print_buffer_diagnostics(buffer: EpisodicBuffer, tokenizer, n_examples: int = 5):
    """Print comprehensive buffer diagnostics + example episodes per type."""

    stats = buffer.get_statistics()
    if stats.get("empty"):
        print("Buffer is empty.")
        return

    print("=" * 65)
    print("EPISODIC BUFFER DIAGNOSTICS")
    print("=" * 65)
    print(f"  Size: {stats['current_size']}/{buffer.max_capacity} "
          f"({stats['capacity_pct']:.1f}% full)")
    print(f"  Lifetime stored: {stats['total_stored']}, evicted: {stats['total_evicted']}")
    print(f"  Mean peak CE: {stats['mean_peak_ce']:.2f}, "
          f"Max: {stats['max_peak_ce']:.2f}")
    print(f"  Mean consolidation score: {stats['mean_consolidation_score']:.3f}")
    print(f"  Gate admission rate: {stats['admission_rate']:.3f}")

    print(f"\n  Type distribution:")
    for cat, count in sorted(stats["by_type"].items(), key=lambda x: -x[1]):
        pct = count / stats["current_size"] * 100
        print(f"    {cat:25s}: {count:5d} ({pct:.1f}%)")

    # Example episodes per type
    for cat in sorted(stats["by_type"].keys()):
        eids = list(buffer.category_index.get(cat, set()))
        if not eids:
            continue
        eps = [buffer.episodes[eid] for eid in eids]
        eps.sort(key=lambda e: e.peak_ce, reverse=True)

        print(f"\n  Top {cat.upper()} episodes:")
        for ep in eps[:n_examples]:
            # Decode a short context around peak
            peak = ep.peak_position
            ctx_start = max(0, peak - 10)
            ctx_end = min(len(ep.input_ids), peak + 5)
            context = tokenizer.decode(ep.input_ids[ctx_start:ctx_end])
            peak_token = tokenizer.decode(ep.input_ids[min(peak + 1, len(ep.input_ids) - 1):
                                                        min(peak + 2, len(ep.input_ids))])

            print(f"    id={ep.episode_id:4d} | CE={ep.peak_ce:.1f} | "
                  f"score={ep.consolidation_score:.2f} | "
                  f"[{peak_token.strip()}] ...{context.strip()}")

    print("=" * 65)


def plot_buffer_diagnostics(buffer: EpisodicBuffer, save_path: str = "system2_diagnostics.png"):
    """3-panel visualization of buffer state."""
    if not buffer.episodes:
        print("Buffer empty, nothing to plot.")
        return

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    eps = list(buffer.episodes.values())
    types = [ep.surprise_type for ep in eps]
    peak_ces = [ep.peak_ce for ep in eps]
    scores = [ep.consolidation_score for ep in eps]
    type_colors = {
        "vocabulary_gap": "#e74c3c",
        "undergeneralization": "#2ecc71",
        "out_of_distribution": "#3498db",
    }
    colors = [type_colors.get(t, "#999") for t in types]

    # Panel 1: Peak CE distribution by type
    for cat in type_colors:
        cat_ces = [ep.peak_ce for ep in eps if ep.surprise_type == cat]
        if cat_ces:
            axes[0].hist(cat_ces, bins=40, alpha=0.5, label=cat, color=type_colors[cat])
    axes[0].set_title("Peak CE by Type")
    axes[0].set_xlabel("Peak CE")
    axes[0].legend(fontsize=8)

    # Panel 2: Consolidation score distribution
    axes[1].hist(scores, bins=30, color="steelblue", alpha=0.7, edgecolor="white")
    axes[1].axvline(0.3, color="orange", linestyle="--", label="Partial (0.3)")
    axes[1].axvline(0.7, color="red", linestyle="--", label="Full (0.7)")
    axes[1].set_title("Consolidation Scores")
    axes[1].set_xlabel("Score")
    axes[1].legend(fontsize=8)

    # Panel 3: Type pie chart
    type_counts = Counter(types)
    labels = list(type_counts.keys())
    sizes = list(type_counts.values())
    pie_colors = [type_colors.get(l, "#999") for l in labels]
    axes[2].pie(sizes, labels=labels, colors=pie_colors, autopct='%1.0f%%', startangle=90)
    axes[2].set_title("Episode Types")

    plt.suptitle("System 2: Episodic Buffer Diagnostics", fontsize=14)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved to {save_path}")

In [ ]:
# ╔═══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 21: RUN SYSTEM 2                                                  ║
# ║                                                                          ║
# ║  Requires System 1 outputs: tracker, categories                          ║
# ║  Produces: buffer (EpisodicBuffer) ready for System 3                    ║
# ╚═══════════════════════════════════════════════════════════════════════════╝

def run_system2(
    tracker: HierarchicalSurpriseTracker,
    categories: Dict[str, List[Tuple[int, int]]],
    model: nn.Module,
    tokenizer,
    device: torch.device,
    max_capacity: int = 2000,
) -> EpisodicBuffer:
    """
    Complete System 2 pipeline:
        1. Create buffer with adaptive gate
        2. Ingest System 1's surprise data into episodes
        3. Print diagnostics
        4. Prepare a sample consolidation batch (dry run for System 3)
    """

    print("\n" + "=" * 65)
    print("  SYSTEM 2: EPISODIC BUFFER — FULL PIPELINE")
    print("=" * 65)
    pipeline_start = time.time()

    # ---- 1. Create buffer ----
    print("\n[1/4] Creating episodic buffer...")
    buffer = EpisodicBuffer(max_capacity=max_capacity)

    # ---- 2. Ingest ----
    print("\n[2/4] Ingesting episodes from System 1...")
    buffer = ingest_system1_to_buffer(
        tracker=tracker,
        categories=categories,
        buffer=buffer,
        model=model,
        tokenizer=tokenizer,
        device=device,
    )

    # ---- 3. Diagnostics ----
    print("\n[3/4] Buffer diagnostics...")
    print_buffer_diagnostics(buffer, tokenizer)
    plot_buffer_diagnostics(buffer)

    # ---- 4. Sample consolidation batch (dry run) ----
    print("\n[4/4] Preparing sample consolidation batch...")
    if len(buffer.episodes) > 0:
        batch = prepare_consolidation_batch(
            buffer,
            n_episodes=min(500, len(buffer.episodes)),
            strategy="priority",
        )
    else:
        print("  Buffer empty — no batch to prepare.")
        batch = None

    elapsed = time.time() - pipeline_start
    print(f"\n{'=' * 65}")
    print(f"  SYSTEM 2 COMPLETE — {elapsed:.1f}s")
    print(f"  Buffer: {len(buffer.episodes)} episodes")
    print(f"  Gate: {buffer.surprise_gate}")
    print(f"{'=' * 65}")

    return buffer

In [ ]:
# ╔═══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 22: LAUNCH SYSTEM 2                                               ║
# ║  Uncomment and run. Requires System 1 outputs.                           ║
# ╚═══════════════════════════════════════════════════════════════════════════╝

buffer = run_system2(tracker, categories, model, tokenizer, device, max_capacity=800)

In [ ]:
# ╔═══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 23: PEFT INSTALL + TRAINING STRATEGY CONFIG                       ║
# ║  (spec §2, §5)                                                           ║
# ║                                                                          ║
# ║  Add after Cell 22 (System 2 launch).                                    ║
# ║  Requires: pip install peft                                               ║
# ╚═══════════════════════════════════════════════════════════════════════════╝

# !pip install peft -q

from peft import LoraConfig, get_peft_model, TaskType
from torch.utils.data import Dataset, DataLoader


@dataclass
class TrainingConfig:
    """Type-specific consolidation strategy (spec §5.1)."""
    name: str
    loss_fn_name: str            # "standard_ce" | "weighted_ce" | "label_smoothed_ce"
    learning_rate: float
    epochs: int
    lora_rank: int
    ewc_lambda: float
    label_smoothing: float = 0.0
    weight_beta: float = 0.0     # For weighted CE: structural emphasis strength


def get_training_strategy(objective_or_type: str) -> TrainingConfig:
    """
    Route learning objectives to training configs.

    Accepts either a learning_objective (ACQUIRE, RESTRUCTURE, etc.)
    or a legacy surprise_type (vocabulary_gap, etc.) for backward compatibility.

    The objective-based routing produces better consolidation because it
    differentiates between first-time learning (ACQUIRE) and remediation
    (REINFORCE, CORRECT), which require fundamentally different training
    approaches even when the underlying surprise type is the same.
    """

    # Objective-based strategies (primary)
    objective_strategies = {
        "ACQUIRE": TrainingConfig(
            name="acquisition",
            loss_fn_name="standard_ce",
            learning_rate=3e-4,
            epochs=2,
            lora_rank=8,
            ewc_lambda=500,
            # Fast learning of new surface patterns.
            # Low rank is sufficient — surface patterns have low intrinsic dimension.
            # Low EWC — okay to adjust output-facing weights.
        ),
        "RESTRUCTURE": TrainingConfig(
            name="structural_consolidation",
            loss_fn_name="weighted_ce",
            learning_rate=1e-4,
            epochs=4,
            lora_rank=32,
            ewc_lambda=2000,
            weight_beta=2.0,
            # Slow, careful modification of attention patterns.
            # High rank needed — structural changes require more capacity.
            # High EWC — protect existing structural knowledge.
            # Weighted CE emphasizes structurally-surprising positions.
        ),
        "CONTEXTUALIZE": TrainingConfig(
            name="domain_expansion",
            loss_fn_name="label_smoothed_ce",
            learning_rate=5e-5,
            epochs=3,
            lora_rank=16,
            ewc_lambda=5000,
            label_smoothing=0.1,
            # Maximum caution for genuinely novel content.
            # Label smoothing prevents overfitting to exact token sequences.
            # Very high EWC — maximum protection of existing knowledge.
        ),
        "REINFORCE": TrainingConfig(
            name="reinforcement",
            loss_fn_name="standard_ce",
            learning_rate=2e-4,
            epochs=2,
            lora_rank=16,
            ewc_lambda=1000,
            # Medium everything — the pattern is partially learned.
            # Fewer epochs than initial learning (already partially there).
            # Standard CE is fine — the model roughly knows what to predict,
            # it just needs to strengthen the connection.
        ),
        "CORRECT": TrainingConfig(
            name="correction",
            loss_fn_name="standard_ce",
            learning_rate=5e-5,
            epochs=3,
            lora_rank=8,
            ewc_lambda=5000,
            # Very conservative: previous consolidation caused harm.
            # Very low LR to avoid compounding the damage.
            # Very high EWC to anchor corrections to known-good state.
            # Low rank — corrections should be minimal, targeted adjustments.
        ),
    }

    # Legacy surprise_type fallback
    type_to_objective = {
        "vocabulary_gap":       "ACQUIRE",
        "undergeneralization":  "RESTRUCTURE",
        "out_of_distribution":  "CONTEXTUALIZE",
    }

    # Try objective first, then map type to objective
    key = objective_or_type
    if key not in objective_strategies:
        key = type_to_objective.get(objective_or_type, "ACQUIRE")

    return objective_strategies.get(key, objective_strategies["ACQUIRE"])

def select_lora_targets(batch: ConsolidationBatch) -> List[str]:
    """
    Choose LoRA target modules based on batch composition (spec §2.2).

    Surface failures → early layers (0-7)
    Structural failures → mid layers (6-15)
    OOD failures → late layers + MLP (12-21)
    """
    type_counts = Counter(ep.surprise_type for ep in batch.episodes)
    total = max(len(batch.episodes), 1)
    vocab_frac = type_counts.get("vocabulary_gap", 0) / total
    under_frac = type_counts.get("undergeneralization", 0) / total
    ood_frac = type_counts.get("out_of_distribution", 0) / total

    targets = set()

    if vocab_frac > 0.2:
        for i in range(0, 8):
            targets.add(f"model.layers.{i}.self_attn.q_proj")
            targets.add(f"model.layers.{i}.self_attn.v_proj")

    if under_frac > 0.2:
        for i in range(6, 16):
            targets.add(f"model.layers.{i}.self_attn.q_proj")
            targets.add(f"model.layers.{i}.self_attn.v_proj")

    if ood_frac > 0.2:
        for i in range(12, 22):
            targets.add(f"model.layers.{i}.self_attn.q_proj")
            targets.add(f"model.layers.{i}.self_attn.v_proj")
        for i in range(14, 22):
            targets.add(f"model.layers.{i}.mlp.gate_proj")
            targets.add(f"model.layers.{i}.mlp.up_proj")

    if not targets:
        return ["q_proj", "v_proj"]  # Fallback: all layers

    return list(targets)

In [ ]:
# ╔═══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 24: LOSS FUNCTIONS                                                 ║
# ║  (spec §5.2)                                                             ║
# ╚═══════════════════════════════════════════════════════════════════════════╝

def standard_ce_loss(
    logits: torch.Tensor,
    labels: torch.Tensor,
    **kwargs,
) -> torch.Tensor:
    """Standard next-token CE loss (spec §5.2). Used for vocabulary_gap."""
    shift_logits = logits[:, :-1, :].contiguous()
    shift_labels = labels[:, 1:].contiguous()
    return F.cross_entropy(
        shift_logits.view(-1, shift_logits.size(-1)),
        shift_labels.view(-1),
        ignore_index=-100,
    )


def weighted_ce_loss(
    logits: torch.Tensor,
    labels: torch.Tensor,
    structural_errors: Optional[torch.Tensor] = None,
    beta: float = 2.0,
    **kwargs,
) -> torch.Tensor:
    """
    CE weighted by structural surprise (spec §5.2).
    Used for undergeneralization: tokens where the model failed structurally
    get upweighted so the model pays more attention to learning those positions.

    w_t = 1 + β * normalized_structural_error(t)
    """
    shift_logits = logits[:, :-1, :].contiguous()
    shift_labels = labels[:, 1:].contiguous()

    per_token_ce = F.cross_entropy(
        shift_logits.view(-1, shift_logits.size(-1)),
        shift_labels.view(-1),
        ignore_index=-100,
        reduction='none',
    ).view(shift_labels.shape)

    if structural_errors is not None and structural_errors.numel() > 0:
        # Align: structural_errors may be shorter than labels
        se = structural_errors[:, :shift_labels.size(1)]
        # Pad if needed
        if se.size(1) < shift_labels.size(1):
            pad = torch.zeros(se.size(0), shift_labels.size(1) - se.size(1), device=se.device)
            se = torch.cat([se, pad], dim=1)
        # Normalize to [0, 1] per sequence
        se_min = se.min(dim=-1, keepdim=True).values
        se_max = se.max(dim=-1, keepdim=True).values
        se_norm = (se - se_min) / (se_max - se_min + 1e-8)
        weights = 1.0 + beta * se_norm
    else:
        weights = torch.ones_like(per_token_ce)

    mask = (shift_labels != -100).float()
    weights = weights * mask
    return (per_token_ce * weights).sum() / (weights.sum() + 1e-8)


def label_smoothed_ce_loss(
    logits: torch.Tensor,
    labels: torch.Tensor,
    smoothing: float = 0.1,
    **kwargs,
) -> torch.Tensor:
    """
    CE with label smoothing (spec §5.2).
    Used for OOD: prevents overconfident learning on novel tokens.
    """
    V = logits.size(-1)
    shift_logits = logits[:, :-1, :].contiguous()
    shift_labels = labels[:, 1:].contiguous()

    log_probs = F.log_softmax(shift_logits.float(), dim=-1)

    nll_loss = F.nll_loss(
        log_probs.view(-1, V), shift_labels.view(-1),
        ignore_index=-100, reduction='mean',
    )
    smooth_loss = -log_probs.mean(dim=-1)
    mask = (shift_labels != -100).float()
    smooth_loss = (smooth_loss * mask).sum() / (mask.sum() + 1e-8)

    return (1 - smoothing) * nll_loss + smoothing * smooth_loss


# Dispatch table
LOSS_FUNCTIONS = {
    "standard_ce": standard_ce_loss,
    "weighted_ce": weighted_ce_loss,
    "label_smoothed_ce": label_smoothed_ce_loss,
}

In [ ]:
# ╔═══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 25: EWC REGULARIZER + FISHER COMPUTATION                          ║
# ║  (spec §3)                                                               ║
# ╚═══════════════════════════════════════════════════════════════════════════╝

class EWCRegularizer:
    """
    Elastic Weight Consolidation (spec §3.4-3.5).

    Penalizes changes to parameters that are important for existing knowledge:
        L_ewc = (λ/2) Σ_i F_ii · (θ_i - θ*_i)²

    Fisher diagonal F_ii: how sensitive output is to parameter θ_i
    Anchor θ*: parameter values before consolidation training began
    """

    def __init__(
        self,
        model: nn.Module,
        fisher_diagonal: Dict[str, torch.Tensor],
        lambda_ewc: float = 1000.0,
    ):
        self.lambda_ewc = lambda_ewc
        self.fisher = fisher_diagonal

        # Snapshot of anchor parameters (before training starts)
        self.anchor_params: Dict[str, torch.Tensor] = {}
        for name, param in model.named_parameters():
            if param.requires_grad:
                self.anchor_params[name] = param.data.detach().clone()

    def compute_penalty(self, model: nn.Module) -> torch.Tensor:
        """Compute EWC penalty term to add to task loss."""
        device = next(model.parameters()).device
        penalty = torch.tensor(0.0, device=device)

        for name, param in model.named_parameters():
            if name not in self.fisher or name not in self.anchor_params:
                continue
            delta = param - self.anchor_params[name].to(device)
            weighted = self.fisher[name].to(device) * delta ** 2
            penalty = penalty + weighted.sum()

        return (self.lambda_ewc / 2) * penalty


def compute_fisher_diagonal(
    model: nn.Module,
    tokenizer,
    reference_texts: List[str],
    device: torch.device,
    n_samples: int = 200,
    max_length: int = 512,
) -> Dict[str, torch.Tensor]:
    """
    Compute diagonal Fisher Information for LoRA parameters (spec §3.3).

    F_ii = E[(∂L/∂θ_i)²] — measures how important each parameter is
    for the model's current performance on reference data.
    """
    print(f"Computing Fisher diagonal over {min(n_samples, len(reference_texts))} samples...")

    model.eval()
    fisher: Dict[str, torch.Tensor] = {}

    for name, param in model.named_parameters():
        if param.requires_grad:
            fisher[name] = torch.zeros_like(param, device='cpu')

    samples = random.sample(reference_texts, min(n_samples, len(reference_texts)))
    n_valid = 0

    for text in tqdm(samples, desc="Fisher diagonal"):
        model.zero_grad()
        encodings = tokenizer(
            text, return_tensors="pt", truncation=True, max_length=max_length,
        ).to(device)

        if encodings.input_ids.shape[1] < 10:
            continue

        outputs = model(**encodings, labels=encodings.input_ids)
        outputs.loss.backward()
        n_valid += 1

        for name, param in model.named_parameters():
            if param.requires_grad and param.grad is not None:
                fisher[name] += (param.grad.data.cpu() ** 2)

    # Average
    for name in fisher:
        fisher[name] /= max(n_valid, 1)

    # Fisher clipping: cap at 5x the P99 value to prevent explosion (spec §9.2 Solution B)
    # Sample a subset for quantile computation — full cat is too large for 1.1B params
    all_vals = torch.cat([f.flatten().float() for f in fisher.values()])
    if all_vals.numel() > 1_000_000:
        indices = torch.randperm(all_vals.numel())[:1_000_000]
        p99 = float(all_vals[indices].quantile(0.99))
    else:
        p99 = float(all_vals.quantile(0.99))
    del all_vals
    for name in fisher:
        fisher[name] = torch.clamp(fisher[name], min=1e-8, max=p99 * 5)

    print(f"  Computed over {n_valid} valid samples")
    print(f"  Mean importance: {all_vals.mean():.6f}")
    print(f"  P99: {p99:.6f}, Clip at: {p99 * 5:.6f}")

    return fisher

def recompute_fisher_post_merge(
    model: nn.Module,
    tokenizer,
    reference_texts: List[str],
    device: torch.device,
    n_samples: int = 200,
    max_length: int = 512,
) -> Dict[str, torch.Tensor]:
    """
    Compute Fisher diagonal on a merged (fully frozen) model.
    Temporarily enables gradients on all parameters, computes Fisher,
    then re-freezes everything.
    """
    print("  Recomputing Fisher on merged model...")

    # Temporarily enable gradients
    originally_frozen = {}
    for name, param in model.named_parameters():
        originally_frozen[name] = param.requires_grad
        param.requires_grad = True

    # Reuse existing Fisher computation
    fisher = compute_fisher_diagonal(
        model, tokenizer, reference_texts, device,
        n_samples=n_samples, max_length=max_length,
    )

    # Re-freeze
    for name, param in model.named_parameters():
        param.requires_grad = originally_frozen[name]

    return fisher

In [ ]:
# ╔═══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 26: CORESET BUILDING                                              ║
# ║  (spec §4.2)                                                             ║
# ║                                                                          ║
# ║  Selects a small, diverse set of LOW-surprise passages the model         ║
# ║  already handles well — the "don't forget this" anchor.                  ║
# ╚═══════════════════════════════════════════════════════════════════════════╝

@torch.no_grad()
def build_coreset(
    model: nn.Module,
    tokenizer,
    corpus: List[str],
    device: torch.device,
    target_size: int = 200,
    max_length: int = 512,
    diversity_weight: float = 0.3,
    candidate_pool_frac: float = 0.30,
) -> List[dict]:
    """
    Select diverse, low-loss passages as a stability anchor (spec §4.2).

    Steps:
        1. Score all passages by model confidence (low loss = well understood)
        2. Keep bottom 30% as candidates
        3. Greedy farthest-point sampling for topic diversity
    """
    print("=" * 65)
    print("BUILDING CORESET")
    print(f"  Corpus: {len(corpus)} passages, Target: {target_size}")
    print("=" * 65)

    model.eval()

    # ---- Step 1: Score all passages ----
    passage_scores = []
    for i, text in enumerate(tqdm(corpus, desc="Scoring passages")):
        encodings = tokenizer(
            text, return_tensors="pt", truncation=True, max_length=max_length,
        ).to(device)
        if encodings.input_ids.shape[1] < 50:
            continue
        outputs = model(**encodings, labels=encodings.input_ids)
        passage_scores.append({
            "index": i,
            "text": text,
            "loss": outputs.loss.item(),
            "length": encodings.input_ids.shape[1],
        })

    # ---- Step 2: Filter — keep bottom 30% by loss ----
    passage_scores.sort(key=lambda x: x["loss"])
    cutoff = int(len(passage_scores) * candidate_pool_frac)
    candidates = passage_scores[:cutoff]
    print(f"  Candidates (bottom {candidate_pool_frac*100:.0f}%): {len(candidates)}")

    if len(candidates) <= target_size:
        print(f"  Not enough candidates, using all {len(candidates)}")
        return candidates[:target_size]

    # ---- Step 3: Diversity sampling via passage embeddings ----
    print("  Computing passage embeddings for diversity...")
    embeddings = []
    for p in tqdm(candidates, desc="Embeddings"):
        encodings = tokenizer(
            p["text"], return_tensors="pt", truncation=True, max_length=max_length,
        ).to(device)
        outputs = model(**encodings, output_hidden_states=True)
        # Mean pool at middle layer
        h_mid = outputs.hidden_states[12]  # Layer 11 output
        emb = h_mid[0].mean(dim=0).float()
        emb = emb / (emb.norm() + 1e-8)
        embeddings.append(emb.cpu())

    embedding_matrix = torch.stack(embeddings)  # [N_candidates, D]

    # Greedy farthest-point sampling (spec §4.2)
    selected_indices = [0]  # Start with lowest loss
    selected_embs = [embedding_matrix[0]]

    while len(selected_indices) < target_size:
        unselected_mask = torch.ones(len(candidates), dtype=torch.bool)
        for idx in selected_indices:
            unselected_mask[idx] = False
        unselected_idx = torch.where(unselected_mask)[0]

        if len(unselected_idx) == 0:
            break

        selected_matrix = torch.stack(selected_embs)
        unselected_matrix = embedding_matrix[unselected_idx]

        sims = torch.matmul(unselected_matrix, selected_matrix.T)
        max_sims = sims.max(dim=1).values

        losses = torch.tensor([candidates[i]["loss"] for i in unselected_idx.tolist()])
        loss_norm = (losses - losses.min()) / (losses.max() - losses.min() + 1e-8)
        sim_norm = (max_sims - max_sims.min()) / (max_sims.max() - max_sims.min() + 1e-8)

        combined = (1 - diversity_weight) * loss_norm + diversity_weight * sim_norm
        best_local = combined.argmin().item()
        best_global = unselected_idx[best_local].item()

        selected_indices.append(best_global)
        selected_embs.append(embedding_matrix[best_global])

    coreset = [candidates[i] for i in selected_indices]
    losses = [p["loss"] for p in coreset]
    print(f"\n  Coreset built: {len(coreset)} passages")
    print(f"  Loss: mean={np.mean(losses):.4f}, range=[{min(losses):.4f}, {max(losses):.4f}]")

    return coreset

def validate_coreset(
    model: nn.Module,
    tokenizer,
    coreset: List[dict],
    device: torch.device,
    staleness_threshold: float = 0.3,
) -> Tuple[bool, float]:
    """
    Check if coreset still contains nontrivial material.
    If loss variance collapses, coreset is stale (model has memorized it).
    Returns (is_stale, loss_variance).
    """
    model.eval()
    losses = []
    with torch.no_grad():
        for item in coreset[:100]:
            text = item["text"] if isinstance(item, dict) else item
            enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=512).to(device)
            if enc.input_ids.shape[1] < 10:
                continue
            out = model(**enc, labels=enc.input_ids)
            losses.append(out.loss.item())

    if not losses:
        return True, 0.0

    variance = float(np.var(losses))
    mean_loss = float(np.mean(losses))
    cv = np.std(losses) / max(mean_loss, 1e-8)

    print(f"  Coreset validation — mean_loss={mean_loss:.3f}, CV={cv:.3f}, var={variance:.4f}")

    # Stale if CV is very low (all losses similar = model handles everything equally)
    # or mean loss is very low (model has memorized the coreset)
    is_stale = cv < staleness_threshold or mean_loss < 0.5
    if is_stale:
        print(f"  ⚠ Coreset appears stale (CV={cv:.3f}, mean_loss={mean_loss:.3f})")

    return is_stale, variance

In [ ]:
# ╔═══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 27: DATASET CLASSES + COLLATION                                   ║
# ║  (spec §8)                                                               ║
# ╚═══════════════════════════════════════════════════════════════════════════╝

class EpisodeDataset(Dataset):
    """Wraps System 2 episodes for DataLoader (spec §8)."""

    def __init__(self, episodes: List[Episode], tokenizer, max_length: int = 512):
        self.examples = []

        for ep in episodes:
            ids = ep.input_ids
            if len(ids) < 32:
                continue
            if len(ids) > max_length:
                start = max(0, ep.peak_position - max_length // 2)
                end = min(len(ids), start + max_length)
                start = max(0, end - max_length)
                ids = ids[start:end]

            # Extract structural errors for weighted CE
            struct_errs = None
            if ep.surprise_vectors:
                raw_struct = [sv.structural for sv in ep.surprise_vectors]
                # Slice to match ids length (surprise_vectors are shifted)
                raw_struct = raw_struct[:len(ids) - 1]
                if raw_struct:
                    struct_errs = torch.tensor(raw_struct, dtype=torch.float32)

            self.examples.append({
                "input_ids": ids.clone(),
                "labels": ids.clone(),
                "surprise_type": ep.surprise_type,
                "structural_errors": struct_errs,
            })

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        return self.examples[idx]


class CoresetDataset(Dataset):
    """Wraps coreset passages for DataLoader (spec §8)."""

    def __init__(self, coreset_passages: List[dict], tokenizer, max_length: int = 512):
        self.examples = []
        for p in coreset_passages:
            tokens = tokenizer(
                p["text"], truncation=True, max_length=max_length, return_tensors="pt",
            )
            ids = tokens.input_ids[0]
            if len(ids) < 32:
                continue
            self.examples.append({
                "input_ids": ids,
                "labels": ids.clone(),
                "surprise_type": "coreset",
                "structural_errors": None,
            })

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        return self.examples[idx]


def collate_episodes(batch: List[dict], pad_token_id: int = 0) -> dict:
    """
    Pad variable-length episode sequences into a batch.
    Returns dict with input_ids, labels, attention_mask, structural_errors.
    """
    max_len = max(len(item["input_ids"]) for item in batch)
    input_ids = []
    labels = []
    attention_mask = []
    structural_errors = []
    has_struct = any(item["structural_errors"] is not None for item in batch)

    for item in batch:
        ids = item["input_ids"]
        pad_len = max_len - len(ids)

        input_ids.append(F.pad(ids, (0, pad_len), value=pad_token_id))
        labels.append(F.pad(item["labels"], (0, pad_len), value=-100))
        attention_mask.append(torch.cat([
            torch.ones(len(ids), dtype=torch.long),
            torch.zeros(pad_len, dtype=torch.long),
        ]))

        if has_struct:
            if item["structural_errors"] is not None:
                se = item["structural_errors"]
                # Pad structural errors to max_len - 1 (shifted)
                se_pad = max_len - 1 - len(se)
                if se_pad > 0:
                    se = F.pad(se, (0, se_pad), value=0.0)
                structural_errors.append(se[:max_len - 1])
            else:
                structural_errors.append(torch.zeros(max_len - 1))

    result = {
        "input_ids": torch.stack(input_ids),
        "labels": torch.stack(labels),
        "attention_mask": torch.stack(attention_mask),
    }
    if has_struct:
        result["structural_errors"] = torch.stack(structural_errors)

    return result


class InterleavedSampler:
    """
    Yields batches mixing surprise episodes and coreset examples (spec §4.3).

    Each batch: (batch_size - n_coreset) episodes + n_coreset coreset items.
    Coreset sampled with replacement (small pool). Shuffled within batch.
    """

    def __init__(
        self,
        episode_dataset: Dataset,
        coreset_dataset: Dataset,
        batch_size: int = 8,
        coreset_ratio: float = 0.125,
        pad_token_id: int = 0,
    ):
        self.episodes = episode_dataset
        self.coreset = coreset_dataset
        self.batch_size = batch_size
        self.n_coreset = max(1, int(batch_size * coreset_ratio))
        self.n_episode = batch_size - self.n_coreset
        self.pad_token_id = pad_token_id

    def __iter__(self):
        ep_indices = list(range(len(self.episodes)))
        random.shuffle(ep_indices)
        ptr = 0

        while ptr < len(ep_indices):
            items = []
            end = min(ptr + self.n_episode, len(ep_indices))
            for i in range(ptr, end):
                items.append(self.episodes[ep_indices[i]])
            ptr = end

            for _ in range(self.n_coreset):
                c_idx = random.randint(0, len(self.coreset) - 1)
                items.append(self.coreset[c_idx])

            random.shuffle(items)
            yield collate_episodes(items, self.pad_token_id)

    def __len__(self):
        return (len(self.episodes) + self.n_episode - 1) // self.n_episode

In [ ]:
# ╔═══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 28: EVALUATION UTILITY                                            ║
# ║                                                                          ║
# ║  Measures perplexity on held-out data for pre/post comparison.           ║
# ╚═══════════════════════════════════════════════════════════════════════════╝

@torch.no_grad()
def evaluate_perplexity(
    model: nn.Module,
    tokenizer,
    texts: List[str],
    device: torch.device,
    max_length: int = 512,
    label: str = "",
) -> dict:
    """Compute mean perplexity over a set of texts."""
    model.eval()
    total_loss = 0.0
    total_tokens = 0

    for text in texts:
        encodings = tokenizer(
            text, return_tensors="pt", truncation=True, max_length=max_length,
        ).to(device)
        if encodings.input_ids.shape[1] < 10:
            continue
        outputs = model(**encodings, labels=encodings.input_ids)
        n_tokens = encodings.input_ids.shape[1] - 1
        total_loss += outputs.loss.item() * n_tokens
        total_tokens += n_tokens

    avg_loss = total_loss / max(total_tokens, 1)
    ppl = math.exp(min(avg_loss, 100))  # Clamp to avoid overflow

    if label:
        print(f"  {label}: loss={avg_loss:.4f}, PPL={ppl:.2f} ({total_tokens} tokens)")

    return {"loss": avg_loss, "perplexity": ppl, "n_tokens": total_tokens}

In [ ]:
# ╔═══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 29: CONSOLIDATION TRAINING LOOP                                   ║
# ║  (spec §7)                                                               ║
# ║                                                                          ║
# ║  Core training with curriculum phases, type-specific loss, dynamic EWC.  ║
# ╚═══════════════════════════════════════════════════════════════════════════╝

def consolidation_training_loop(
    model: nn.Module,
    episode_dataset: EpisodeDataset,
    coreset_dataset: CoresetDataset,
    ewc: EWCRegularizer,
    config: TrainingConfig,
    device: torch.device,
    pad_token_id: int = 0,
    batch_size: int = 8,
    coreset_ratio: float = 0.125,
    log_interval: int = 10,
    dynamic_lambda: bool = True,
) -> Dict[str, list]:
    """
    Run the consolidation training loop (spec §7.1).

    Simplified curriculum for proof-of-concept:
        Phase A — Warmup (10%):  coreset only, LR ramps 0.1→1.0
        Phase B — Main (80%):    episodes + coreset interleaved, full LR
        Phase C — Cooldown (10%): coreset only, LR decays 1.0→0.1

    Returns per-step metrics dict.
    """

    loss_fn = LOSS_FUNCTIONS[config.loss_fn_name]
    model.train()

    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(trainable_params, lr=config.learning_rate, weight_decay=0.01)

    # Total steps across all epochs
    steps_per_epoch = max(len(episode_dataset) // batch_size, 1)
    total_steps = steps_per_epoch * config.epochs

    # Phase boundaries
    warmup_steps = max(int(total_steps * 0.10), 1)
    cooldown_steps = max(int(total_steps * 0.10), 1)
    main_steps = total_steps - warmup_steps - cooldown_steps

    print(f"  Training schedule: {total_steps} total steps")
    print(f"    Warmup (coreset): {warmup_steps}")
    print(f"    Main (interleaved): {main_steps}")
    print(f"    Cooldown (coreset): {cooldown_steps}")

    metrics = {"task_loss": [], "ewc_loss": [], "total_loss": [], "lr": [], "phase": []}
    global_step = 0

    # ---- Helpers ----
    def get_lr_multiplier(step: int) -> float:
        if step < warmup_steps:
            return 0.1 + 0.9 * (step / warmup_steps)
        elif step >= total_steps - cooldown_steps:
            progress = (step - (total_steps - cooldown_steps)) / cooldown_steps
            return max(0.1, 1.0 - 0.9 * progress)
        return 1.0

    def get_phase(step: int) -> str:
        if step < warmup_steps:
            return "warmup"
        elif step >= total_steps - cooldown_steps:
            return "cooldown"
        return "main"

    def make_coreset_batch():
        indices = [random.randint(0, len(coreset_dataset) - 1) for _ in range(batch_size)]
        items = [coreset_dataset[i] for i in indices]
        return collate_episodes(items, pad_token_id)

    # ---- Build interleaved loader for main phase ----
    interleaved = InterleavedSampler(
        episode_dataset, coreset_dataset,
        batch_size=batch_size, coreset_ratio=coreset_ratio,
        pad_token_id=pad_token_id,
    )

    # ---- Training ----
    running_task = 0.0
    running_ewc = 0.0
    running_total = 0.0

    for epoch in range(config.epochs):
        loader_iter = iter(interleaved)

        for _ in range(steps_per_epoch):
            if global_step >= total_steps:
                break

            phase = get_phase(global_step)
            lr_mult = get_lr_multiplier(global_step)

            for pg in optimizer.param_groups:
                pg['lr'] = config.learning_rate * lr_mult

            # ---- Get batch ----
            if phase in ("warmup", "cooldown"):
                batch = make_coreset_batch()
            else:
                try:
                    batch = next(loader_iter)
                except StopIteration:
                    loader_iter = iter(interleaved)
                    batch = next(loader_iter)

            # Move to device
            input_ids = batch["input_ids"].to(device)
            labels_tensor = batch["labels"].to(device)
            attn_mask = batch["attention_mask"].to(device)

            struct_errs = None
            if "structural_errors" in batch:
                struct_errs = batch["structural_errors"].to(device)

            # ---- Forward ----
            outputs = model(input_ids=input_ids, attention_mask=attn_mask)
            task_loss = loss_fn(
                outputs.logits, labels_tensor,
                structural_errors=struct_errs,
                beta=config.weight_beta,
                smoothing=config.label_smoothing,
            )

            # ---- EWC penalty ----
            ewc_penalty_raw = ewc.compute_penalty(model)

            # Dynamic lambda: cap so EWC never exceeds 2x task loss (spec §9.2)
            if dynamic_lambda and ewc_penalty_raw.item() > 1e-10:
                effective_lambda = min(
                    ewc.lambda_ewc,
                    task_loss.item() / ewc_penalty_raw.item() * 2.0
                )
                ewc_penalty = (effective_lambda / ewc.lambda_ewc) * ewc_penalty_raw
            else:
                ewc_penalty = ewc_penalty_raw

            total_loss = task_loss + ewc_penalty

            # ---- Backward ----
            optimizer.zero_grad()
            total_loss.backward()
            torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=1.0)
            optimizer.step()

            # ---- Log ----
            t_loss = task_loss.item()
            e_loss = ewc_penalty.item()
            tot_loss = total_loss.item()

            metrics["task_loss"].append(t_loss)
            metrics["ewc_loss"].append(e_loss)
            metrics["total_loss"].append(tot_loss)
            metrics["lr"].append(config.learning_rate * lr_mult)
            metrics["phase"].append(phase)

            running_task += t_loss
            running_ewc += e_loss
            running_total += tot_loss
            global_step += 1

            if global_step % log_interval == 0:
                n = log_interval
                print(f"  Step {global_step:4d}/{total_steps} [{phase:8s}] | "
                      f"Task: {running_task/n:.4f} | EWC: {running_ewc/n:.4f} | "
                      f"Total: {running_total/n:.4f} | "
                      f"LR: {config.learning_rate * lr_mult:.2e}")
                running_task = running_ewc = running_total = 0.0

    model.eval()
    return metrics

In [ ]:
# ╔═══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 30: CONSOLIDATION DIAGNOSTICS                                     ║
# ╚═══════════════════════════════════════════════════════════════════════════╝

def plot_consolidation_metrics(
    metrics: Dict[str, list],
    save_path: str = "system3_consolidation.png",
):
    """3-panel training diagnostics."""
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    steps = list(range(len(metrics["task_loss"])))

    # Panel 1: Loss curves
    axes[0].plot(steps, metrics["task_loss"], label="Task", alpha=0.7, linewidth=0.8)
    axes[0].plot(steps, metrics["ewc_loss"], label="EWC", alpha=0.7, linewidth=0.8)
    axes[0].plot(steps, metrics["total_loss"], label="Total", alpha=0.5, linewidth=0.8)
    axes[0].set_title("Loss Components")
    axes[0].set_xlabel("Step")
    axes[0].set_ylabel("Loss")
    axes[0].legend(fontsize=8)
    axes[0].set_yscale("log")

    # Panel 2: Learning rate schedule
    axes[1].plot(steps, metrics["lr"], color="darkorange")
    axes[1].set_title("Learning Rate")
    axes[1].set_xlabel("Step")

    # Panel 3: Phase indicators + smoothed task loss
    phase_colors = {"warmup": "#3498db", "main": "#2ecc71", "cooldown": "#e74c3c"}
    for i, (step, phase) in enumerate(zip(steps, metrics["phase"])):
        axes[2].axvspan(step, step + 1, alpha=0.1, color=phase_colors.get(phase, "#999"))

    # Smoothed task loss
    window = min(20, len(steps) // 4)
    if window > 1:
        smoothed = np.convolve(metrics["task_loss"], np.ones(window)/window, mode='valid')
        axes[2].plot(range(window - 1, len(steps)), smoothed, color="black", linewidth=1.5)
    axes[2].set_title("Task Loss (smoothed) + Phases")
    axes[2].set_xlabel("Step")

    # Phase legend
    from matplotlib.patches import Patch
    legend_elements = [Patch(facecolor=c, alpha=0.3, label=n)
                       for n, c in phase_colors.items()]
    axes[2].legend(handles=legend_elements, fontsize=8)

    plt.suptitle("System 3 Phase 1: Consolidation Metrics", fontsize=14)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved to {save_path}")

In [ ]:
# ╔═══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 31: RUN SYSTEM 3 PHASE 1                                          ║
# ║  (spec §7.1, §10)                                                        ║
# ║                                                                          ║
# ║  Requires: System 1 (model, tokenizer, device) + System 2 (buffer)       ║
# ║  Produces: consolidated model with LoRA merged                           ║
# ╚═══════════════════════════════════════════════════════════════════════════╝

def run_system3_phase1(
    model: nn.Module,
    tokenizer,
    buffer: 'EpisodicBuffer',
    eval_corpus: List[str],
    reference_corpus: List[str],
    coreset_corpus: List[str],
    device: torch.device,
    batch_strategy: str = "balanced",
    n_batch_episodes: int = 500,
    batch_size: int = 4,
    max_degradation: float = 0.05,
    resonance_ewc_multiplier: float = 1.0,
    resonance_lr_multiplier: float = 1.0,
) -> Tuple[nn.Module, dict, 'EWCRegularizer']:
    """
    Complete System 3 Phase 1: NREM consolidation.

    Steps:
        1. Prepare consolidation batch from buffer (balanced across types)
        2. Determine dominant type → select training strategy
        3. Build coreset (diverse low-loss passages)
        4. Apply LoRA adapters to model
        5. Compute Fisher diagonal for EWC
        6. Calibrate EWC lambda
        7. Run consolidation training loop
        8. Evaluate: compare pre vs post perplexity
        9. If degradation acceptable → merge LoRA, else revert
        10. Update buffer consolidation scores
    """

    print("\n" + "=" * 65)
    print("  SYSTEM 3 PHASE 1: NREM CONSOLIDATION")
    print("=" * 65)
    pipeline_start = time.time()

    # ---- 1. Prepare batch ----
    print("\n[1/10] Preparing consolidation batch...")
    batch = prepare_consolidation_batch(
        buffer, n_episodes=n_batch_episodes, strategy=batch_strategy,
    )
    if not batch.episodes:
        print("  No episodes to consolidate. Returning.")
        return model, {"skipped": True}

        # ---- 2. Training strategy ----
    print("\n[2/10] Selecting training strategy...")

    # Ensure all episodes have current learning objectives
    for ep in batch.episodes:
        ep.learning_objective = compute_learning_objective(ep)

    # Count by objective (not surprise_type)
    obj_counts = Counter(ep.learning_objective for ep in batch.episodes)
    dominant_objective = obj_counts.most_common(1)[0][0]
    config = get_training_strategy(dominant_objective)

    print(f"  Batch composition by objective:")
    for obj, count in obj_counts.most_common():
        print(f"    {obj}: {count}")
    print(f"  Dominant objective: {dominant_objective}")
    print(f"  Strategy: {config.name}")
    print(f"  LR={config.learning_rate}, rank={config.lora_rank}, "
          f"EWC λ={config.ewc_lambda}, loss={config.loss_fn_name}")

    # Also print by surprise type for comparison
    type_counts = Counter(ep.surprise_type for ep in batch.episodes)
    print(f"  (By surprise type: {dict(type_counts)})")

    print(f"  Batch composition: {dict(type_counts)}")
    print(f"  Dominant type: {dominant_type}")
    print(f"  Strategy: {config.name}")
    print(f"  LR={config.learning_rate}, rank={config.lora_rank}, "
          f"EWC λ={config.ewc_lambda}, loss={config.loss_fn_name}")
    # Apply resonance-based adjustments (PhysMem-inspired)
    if resonance_ewc_multiplier != 1.0 or resonance_lr_multiplier != 1.0:
        original_lr = config.learning_rate
        original_ewc = config.ewc_lambda
        config.learning_rate *= resonance_lr_multiplier
        config.ewc_lambda *= resonance_ewc_multiplier
        print(f"  Resonance adjustment applied:")
        print(f"    LR: {original_lr:.1e} → {config.learning_rate:.1e} "
              f"({resonance_lr_multiplier:.2f}x)")
        print(f"    EWC λ: {original_ewc:.0f} → {config.ewc_lambda:.0f} "
              f"({resonance_ewc_multiplier:.2f}x)")

    # ---- 3. Build coreset ----
    print("\n[3/10] Building coreset...")
    coreset = build_coreset(
        model, tokenizer, coreset_corpus, device,
        target_size=200, max_length=512,
    )

    # ---- 4. Pre-consolidation evaluation ----
    print("\n[4/10] Pre-consolidation evaluation...")
    pre_eval = evaluate_perplexity(model, tokenizer, eval_corpus, device, label="Pre-consolidation")

    # ---- 5. Apply LoRA ----
    print("\n[5/10] Applying LoRA adapters...")
    lora_targets = select_lora_targets(batch)
    # Check if specific layer paths or generic patterns
    if any("model.layers" in t for t in lora_targets):
        target_modules = lora_targets
    else:
        target_modules = ["q_proj", "v_proj"]

    lora_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=config.lora_rank,
        lora_alpha=config.lora_rank * 2,
        lora_dropout=0.05,
        target_modules=target_modules,
    )
    peft_model = get_peft_model(model, lora_config)
    peft_model.print_trainable_parameters()

    # ---- 6. Compute Fisher diagonal ----
    print("\n[6/10] Computing Fisher Information Matrix...")
    fisher = compute_fisher_diagonal(
        peft_model, tokenizer, reference_corpus, device, n_samples=200,
    )

    # ---- 7. Initialize EWC ----
    print("\n[7/10] Initializing EWC regularizer...")
    ewc = EWCRegularizer(peft_model, fisher, lambda_ewc=config.ewc_lambda)

    # Calibrate lambda on a sample episode
    sample_ids = batch.episodes[0].input_ids.unsqueeze(0).to(device)
    with torch.no_grad():
        sample_out = peft_model(input_ids=sample_ids, labels=sample_ids)
    task_sample = sample_out.loss.item()
    ewc_sample = ewc.compute_penalty(peft_model).item()
    if ewc_sample > 1e-10:
        calibrated = max(100, min(10000, task_sample / ewc_sample))
        ewc.lambda_ewc = calibrated
        print(f"  Calibrated EWC λ: {calibrated:.1f} "
              f"(task={task_sample:.4f}, ewc@1={ewc_sample:.6f})")
    else:
        print(f"  Using default EWC λ: {ewc.lambda_ewc}")

    # ---- 8. Build datasets ----
    print("\n[8/10] Building datasets...")
    episode_ds = EpisodeDataset(batch.episodes, tokenizer, max_length=512)
    coreset_ds = CoresetDataset(coreset, tokenizer, max_length=512)
    print(f"  Episode dataset: {len(episode_ds)} examples")
    print(f"  Coreset dataset: {len(coreset_ds)} examples")

    # ---- 9. Train ----
    print("\n[9/10] Consolidation training...")
    pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 0
    metrics = consolidation_training_loop(
        model=peft_model,
        episode_dataset=episode_ds,
        coreset_dataset=coreset_ds,
        ewc=ewc,
        config=config,
        device=device,
        pad_token_id=pad_id,
        batch_size=batch_size,
    )

    plot_consolidation_metrics(metrics)
    # Mark episodes as having been through consolidation
    current_cycle = buffer.current_step  # Or pass cycle number explicitly
    for ep in batch.episodes:
        ep.consolidation_attempts += 1
        ep.last_consolidated_cycle = current_cycle

    # ---- 10. Post-consolidation evaluation + merge/revert ----
    print("\n[10/10] Post-consolidation evaluation...")
    post_eval = evaluate_perplexity(peft_model, tokenizer, eval_corpus, device, label="Post-consolidation")

    ppl_change = (post_eval["perplexity"] - pre_eval["perplexity"]) / pre_eval["perplexity"]

    report = {
        "pre_ppl": pre_eval["perplexity"],
        "post_ppl": post_eval["perplexity"],
        "ppl_change_pct": ppl_change * 100,
        "config": config.name,
        "dominant_type": dominant_type,
        "n_episodes": len(batch.episodes),
        "n_coreset": len(coreset),
        "training_steps": len(metrics["task_loss"]),
        "final_task_loss": np.mean(metrics["task_loss"][-20:]) if metrics["task_loss"] else 0,
        "final_ewc_loss": np.mean(metrics["ewc_loss"][-20:]) if metrics["ewc_loss"] else 0,
    }

    if ppl_change < max_degradation:
        if ppl_change < 0:
            print(f"\n  ✓ PASSED: PPL improved by {-ppl_change*100:.2f}%")
        else:
            print(f"\n  ✓ PASSED: PPL increased by {ppl_change*100:.2f}% "
                  f"(within {max_degradation*100:.0f}% tolerance)")

        # Merge LoRA into base model
        print("  Merging LoRA weights into base model...")
        merged_model = peft_model.merge_and_unload()
        report["merged"] = True

        # Update buffer consolidation scores with merged model
        print("  Updating buffer consolidation scores...")
        buffer.update_consolidation_scores(merged_model, device)

        # Evict fully consolidated episodes
        n_evicted = buffer.evict_consolidated()
        print(f"  Evicted {n_evicted} consolidated episodes")
        report["n_evicted"] = n_evicted

    else:
        print(f"\n  ✗ FAILED: PPL increased by {ppl_change*100:.2f}% "
              f"(exceeds {max_degradation*100:.0f}% tolerance)")
        print("  REVERTING to pre-consolidation model.")

        # Discard LoRA, return original model
        merged_model = peft_model.get_base_model()
        report["merged"] = False

    elapsed = time.time() - pipeline_start
    report["elapsed_s"] = elapsed

    print(f"\n{'=' * 65}")
    print(f"  SYSTEM 3 PHASE 1 COMPLETE — {elapsed:.1f}s")
    print(f"  Pre PPL:  {report['pre_ppl']:.2f}")
    print(f"  Post PPL: {report['post_ppl']:.2f} ({report['ppl_change_pct']:+.2f}%)")
    print(f"  Merged: {report['merged']}")
    print(f"  Buffer: {len(buffer.episodes)} episodes remaining")
    print(f"{'=' * 65}")

    return merged_model, report, ewc

In [ ]:
# ╔═══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 32: LAUNCH SYSTEM 3 PHASE 1                                       ║
# ║                                                                          ║
# ║  Requires:                                                                ║
# ║    - model, tokenizer, device (from System 1)                            ║
# ║    - buffer (from System 2)                                               ║
# ║    - eval_corpus, wake_corpus (from Cell 4 data loading)                  ║
# ║                                                                          ║
# ║  Uncomment and run.                                                      ║
# ╚═══════════════════════════════════════════════════════════════════════════╝

consolidated_model, consolidation_report, consolidation_ewc  = run_system3_phase1(
    model=model,
    tokenizer=tokenizer,
    buffer=buffer,
    eval_corpus=eval_corpus,          # From Cell 4
    reference_corpus=head_val_corpus,  # Low-cost Fisher reference
    coreset_corpus=wake_corpus,        # Full corpus for coreset selection
    device=device,
    batch_strategy="balanced",         # Equal per type, not buffer-proportional
    n_batch_episodes=min(500, len(buffer.episodes)),
    batch_size=4,                      # Small batch — A100 has room but LoRA + EWC is memory-heavy
)

In [ ]:
# ╔═══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 33: PHASE 2 DATA STRUCTURES + CONFIG                              ║
# ║  (spec §3.1, §4.2, §5.1, §6)                                            ║
# ║                                                                          ║
# ║  Add after Cell 32 (System 3 Phase 1 launch).                            ║
# ║  Requires: Phase 1 completed (consolidated model + buffer).              ║
# ╚═══════════════════════════════════════════════════════════════════════════╝

from contextlib import nullcontext

@dataclass
class GeneralizationProbe:
    """A synthetic test passage for evaluating generalization (spec §3.1)."""
    source_episode_id: int
    source_type: str
    probe_ids: torch.Tensor          # [T] token IDs, CPU
    probe_text: str
    prefix_len: int
    method: str                       # "prefix_continuation" | "token_perturbation"
    # Filled during scoring:
    pre_consolidation_loss: float = 0.0
    post_consolidation_loss: float = 0.0
    generalization_score: float = 0.0


@dataclass
class EpisodeDiagnosis:
    """Per-episode generalization verdict (spec §4.2)."""
    episode_id: int
    outcome: str                     # GENERALIZED | FRAGILE | RIGID | MEMORIZED | INTERFERENCE
    mean_generalization: float
    prefix_generalization: float = 0.0
    perturbation_generalization: float = 0.0
    n_probes: int = 0
    surprise_type: str = ""


@dataclass
class RemediationConfig:
    """Per-outcome corrective action (spec §5.1)."""
    action: str                      # NONE | CONTRASTIVE_REPLAY | NOISE_INJECTION_REPLAY |
                                     # GENERATIVE_REHEARSAL | SELECTIVE_ROLLBACK
    n_additional_steps: int = 0
    learning_rate_multiplier: float = 1.0
    use_contrastive_loss: bool = False
    contrastive_weight: float = 0.0
    use_token_dropout: bool = False
    dropout_schedule: List[float] = field(default_factory=list)
    use_generation_objective: bool = False
    kl_diversity_weight: float = 0.0
    rollback_fraction: float = 0.0


@dataclass
class Phase2Config:
    """Top-level Phase 2 settings."""
    n_probes_per_episode: int = 3
    base_lr: float = 2e-4
    contrastive_steps: int = 20       # Per memorized episode
    noise_steps_per_level: int = 4    # Per fragile episode per noise level
    generative_steps: int = 15        # Per rigid episode
    generalization_threshold: float = 0.03
    interference_threshold: float = -0.03
    max_episodes_to_probe: int = 200  # Cap to keep compute bounded


@dataclass
class Phase2Report:
    """Summary of Phase 2 results."""
    n_episodes: int
    n_probes: int
    diagnosis_distribution: Dict[str, int]
    remediation_results: Dict[str, dict]
    mean_generalization_phase1: float
    mean_generalization_phase2: float
    phase2_delta: float
    diagnoses: Dict[int, EpisodeDiagnosis] = field(default_factory=dict)

In [ ]:
# ╔═══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 34: PROBE GENERATION                                              ║
# ║  (spec §3.1, §3.2)                                                       ║
# ║                                                                          ║
# ║  Two methods:                                                             ║
# ║    1. Prefix continuation — model generates from episode prefix           ║
# ║    2. Token perturbation — replace content words with alternatives        ║
# ╚═══════════════════════════════════════════════════════════════════════════╝

@torch.no_grad()
def generate_token_perturbation(
    model: nn.Module,
    tokenizer,
    input_ids: torch.Tensor,
    device: torch.device,
    perturbation_rate: float = 0.3,
) -> Optional[torch.Tensor]:
    """
    Replace content tokens with plausible alternatives (spec §3.2).

    Content tokens = positions with above-median prediction entropy.
    High entropy → many plausible alternatives → "content" word.
    Low entropy → few alternatives → "function" word / fixed pattern.

    Replacements sampled from the model's own top-20 predictions.
    """
    ids = input_ids.to(device)
    outputs = model(ids.unsqueeze(0))
    logits = outputs.logits[0]  # [T, V]

    probs = torch.softmax(logits.float(), dim=-1)
    entropy = -(probs * torch.log(probs + 1e-10)).sum(dim=-1)  # [T]

    # Content = above-median entropy
    median_h = entropy.median()
    content_mask = entropy > median_h

    content_positions = torch.where(content_mask)[0]
    n_content = len(content_positions)
    n_to_perturb = int(n_content * perturbation_rate)

    if n_to_perturb < 5:
        return None

    selected = content_positions[torch.randperm(n_content, device=device)[:n_to_perturb]]
    perturbed = ids.clone()

    for pos in selected:
        pos_probs = probs[pos].clone()
        pos_probs[ids[pos]] = 0  # Exclude original
        pos_probs = pos_probs / (pos_probs.sum() + 1e-8)

        topk_probs, topk_ids = pos_probs.topk(20)
        topk_probs = topk_probs / (topk_probs.sum() + 1e-8)
        sampled_idx = torch.multinomial(topk_probs, 1)
        perturbed[pos] = topk_ids[sampled_idx]

    return perturbed.cpu()


@torch.no_grad()
def generate_generalization_probes(
    model: nn.Module,
    tokenizer,
    episodes: List[Episode],
    device: torch.device,
    n_probes_per_episode: int = 3,
    max_episodes: int = 200,
) -> List[GeneralizationProbe]:
    """
    Generate probes testing whether consolidation produced genuine
    generalization vs memorization (spec §3.1).

    For each episode:
        - n_probes_per_episode prefix continuations (model generates from prefix)
        - 1 token perturbation (swap content words)
    """
    model.eval()
    probes = []

    # Cap episode count for compute budget
    probe_episodes = episodes[:max_episodes]

    pad_id = tokenizer.pad_token_id
    if pad_id is None:
        pad_id = tokenizer.eos_token_id if tokenizer.eos_token_id is not None else 0

    for episode in tqdm(probe_episodes, desc="Generating probes"):
        input_ids = episode.input_ids.to(device)
        T = len(input_ids)

        # ---- Method 1: Prefix continuation ----
        prefix_len = max(32, T // 4)
        prefix = input_ids[:prefix_len].unsqueeze(0)
        target_new_tokens = min(T - prefix_len, 256)

        if target_new_tokens < 16:
            continue

        for probe_idx in range(n_probes_per_episode):
            temp = 0.7 + 0.3 * (probe_idx / max(n_probes_per_episode - 1, 1))
            top_p = 0.85 + 0.1 * (probe_idx / max(n_probes_per_episode - 1, 1))

            generated = model.generate(
                prefix,
                max_new_tokens=target_new_tokens,
                temperature=temp,
                top_p=top_p,
                do_sample=True,
                repetition_penalty=1.2,
                no_repeat_ngram_size=4,
                pad_token_id=pad_id,
            )
            gen_ids = generated[0].cpu()

            # Coherence filter: reject degenerate outputs (spec §7.1)
            unique_ratio = len(set(gen_ids[prefix_len:].tolist())) / max(len(gen_ids) - prefix_len, 1)
            if unique_ratio < 0.3:
                continue

            probes.append(GeneralizationProbe(
                source_episode_id=episode.episode_id,
                source_type=episode.surprise_type,
                probe_ids=gen_ids,
                probe_text=tokenizer.decode(gen_ids, skip_special_tokens=True)[:300],
                prefix_len=prefix_len,
                method="prefix_continuation",
            ))

        # ---- Method 2: Token perturbation ----
        perturbed = generate_token_perturbation(model, tokenizer, input_ids, device)
        if perturbed is not None:
            probes.append(GeneralizationProbe(
                source_episode_id=episode.episode_id,
                source_type=episode.surprise_type,
                probe_ids=perturbed,
                probe_text=tokenizer.decode(perturbed, skip_special_tokens=True)[:300],
                prefix_len=0,
                method="token_perturbation",
            ))

    print(f"Generated {len(probes)} probes from {len(probe_episodes)} episodes")
    method_counts = Counter(p.method for p in probes)
    for method, count in method_counts.items():
        print(f"  {method}: {count}")

    return probes

In [ ]:
# ╔═══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 35: GENERALIZATION SCORING + DIAGNOSIS                            ║
# ║  (spec §4.1, §4.2)                                                       ║
# ╚═══════════════════════════════════════════════════════════════════════════╝

@torch.no_grad()
def score_generalization(
    pre_model: nn.Module,
    post_model: nn.Module,
    probes: List[GeneralizationProbe],
    device: torch.device,
) -> List[GeneralizationProbe]:
    """
    Compare probe performance pre vs post Phase 1 (spec §4.1).

    generalization_score > 0:  consolidation helped on novel content
    generalization_score ≈ 0:  no transfer (memorization)
    generalization_score < 0:  consolidation hurt (interference)
    """
    pre_model.eval()
    post_model.eval()

    for probe in tqdm(probes, desc="Scoring probes"):
        ids = probe.probe_ids.unsqueeze(0).to(device)
        if ids.shape[1] < 4:
            continue

        pre_out = pre_model(input_ids=ids, labels=ids)
        probe.pre_consolidation_loss = pre_out.loss.item()

        post_out = post_model(input_ids=ids, labels=ids)
        probe.post_consolidation_loss = post_out.loss.item()

        if probe.pre_consolidation_loss > 0:
            probe.generalization_score = (
                (probe.pre_consolidation_loss - probe.post_consolidation_loss) /
                probe.pre_consolidation_loss
            )
        else:
            probe.generalization_score = 0.0

    return probes


def diagnose_episodes(
    probes: List[GeneralizationProbe],
    episodes: List[Episode],
    gen_threshold: float = 0.03,
    interf_threshold: float = -0.03,
) -> Dict[int, EpisodeDiagnosis]:
    """
    Aggregate probe scores per episode → 2D diagnosis (spec §4.2).

    Uses the generation × recognition matrix:
        prefix good + perturb good  → GENERALIZED
        prefix good + perturb bad   → FRAGILE
        prefix bad  + perturb good  → RIGID
        prefix bad  + perturb bad   → MEMORIZED
        either negative             → INTERFERENCE
    """
    probes_by_ep = defaultdict(list)
    for p in probes:
        probes_by_ep[p.source_episode_id].append(p)

    diagnoses = {}

    for episode in episodes:
        eid = episode.episode_id
        ep_probes = probes_by_ep.get(eid, [])

        if not ep_probes:
            diagnoses[eid] = EpisodeDiagnosis(
                episode_id=eid, outcome="UNKNOWN", mean_generalization=0.0,
            )
            continue

        prefix_probes = [p for p in ep_probes if p.method == "prefix_continuation"]
        perturb_probes = [p for p in ep_probes if p.method == "token_perturbation"]

        prefix_score = float(np.mean([p.generalization_score for p in prefix_probes])) \
            if prefix_probes else 0.0
        perturb_score = float(np.mean([p.generalization_score for p in perturb_probes])) \
            if perturb_probes else 0.0
        mean_score = float(np.mean([p.generalization_score for p in ep_probes]))

        gen_prefix = prefix_score > gen_threshold
        gen_perturb = perturb_score > gen_threshold
        interf_prefix = prefix_score < interf_threshold
        interf_perturb = perturb_score < interf_threshold

        if interf_prefix or interf_perturb:
            outcome = "INTERFERENCE"
        elif gen_prefix and gen_perturb:
            outcome = "GENERALIZED"
        elif gen_prefix and not gen_perturb:
            outcome = "FRAGILE"
        elif not gen_prefix and gen_perturb:
            outcome = "RIGID"
        else:
            outcome = "MEMORIZED"

        diagnoses[eid] = EpisodeDiagnosis(
            episode_id=eid,
            outcome=outcome,
            mean_generalization=mean_score,
            prefix_generalization=prefix_score,
            perturbation_generalization=perturb_score,
            n_probes=len(ep_probes),
            surprise_type=episode.surprise_type,
        )

    outcome_counts = Counter(d.outcome for d in diagnoses.values())
    print(f"\nEPISODE DIAGNOSIS SUMMARY ({len(diagnoses)} episodes):")
    for outcome, count in outcome_counts.most_common():
        pct = count / max(len(diagnoses), 1) * 100
        print(f"  {outcome:20s}: {count:4d} ({pct:.1f}%)")

    return diagnoses

def apply_diagnoses_to_buffer(
    diagnoses: Dict[int, EpisodeDiagnosis],
    buffer: EpisodicBuffer,
    current_cycle: int,
):
    """
    Write Phase 2 diagnosis outcomes back into episode metadata.
    This updates learning_objective for the next consolidation cycle.

    Called after diagnose_episodes() in the Phase 2 pipeline.
    """
    updated = 0
    for eid, diag in diagnoses.items():
        if eid not in buffer.episodes:
            continue
        ep = buffer.episodes[eid]

        # Record outcome in history
        ep.phase2_history.append(diag.outcome)

        # Update confidence based on outcome
        if diag.outcome == "GENERALIZED":
            ep.objective_confidence = min(1.0, ep.objective_confidence + 0.3)
        elif diag.outcome == "FRAGILE":
            ep.objective_confidence = max(0.0, ep.objective_confidence - 0.1)
        elif diag.outcome == "MEMORIZED":
            ep.objective_confidence = max(0.0, ep.objective_confidence - 0.2)
        elif diag.outcome == "INTERFERENCE":
            ep.objective_confidence = 0.0  # Reset — previous learning was harmful

        # Recompute learning objective with new history
        ep.learning_objective = compute_learning_objective(ep)
        updated += 1

    # Summary
    obj_counts = Counter(
        buffer.episodes[eid].learning_objective
        for eid in diagnoses if eid in buffer.episodes
    )
    print(f"\n  Updated {updated} episode objectives after Phase 2:")
    for obj, count in obj_counts.most_common():
        print(f"    {obj}: {count}")

In [ ]:
# ╔═══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 36: REMEDIATION STRATEGIES                                        ║
# ║  (spec §5.2-5.5)                                                         ║
# ║                                                                          ║
# ║  Four corrective actions:                                                 ║
# ║    MEMORIZED     → contrastive replay (push orig ≈ perturbed repr)        ║
# ║    FRAGILE       → noise injection (progressive token dropout)            ║
# ║    RIGID         → generative rehearsal (entropy-regularized CE)          ║
# ║    INTERFERENCE  → selective rollback (revert high-damage params)         ║
# ╚═══════════════════════════════════════════════════════════════════════════╝

def get_remediation_strategy(diagnosis: EpisodeDiagnosis) -> RemediationConfig:
    """Route diagnosis to corrective action (spec §5.1)."""
    if diagnosis.outcome == "GENERALIZED":
        return RemediationConfig(action="NONE")
    elif diagnosis.outcome == "MEMORIZED":
        return RemediationConfig(
            action="CONTRASTIVE_REPLAY",
            n_additional_steps=20,
            learning_rate_multiplier=0.5,
            use_contrastive_loss=True,
            contrastive_weight=0.3,
        )
    elif diagnosis.outcome == "FRAGILE":
        return RemediationConfig(
            action="NOISE_INJECTION_REPLAY",
            n_additional_steps=16,
            learning_rate_multiplier=0.3,
            use_token_dropout=True,
            dropout_schedule=[0.05, 0.10, 0.15, 0.20],
        )
    elif diagnosis.outcome == "RIGID":
        return RemediationConfig(
            action="GENERATIVE_REHEARSAL",
            n_additional_steps=15,
            learning_rate_multiplier=0.5,
            use_generation_objective=True,
            kl_diversity_weight=0.1,
        )
    elif diagnosis.outcome == "INTERFERENCE":
        return RemediationConfig(
            action="SELECTIVE_ROLLBACK",
            rollback_fraction=0.5,
        )
    else:
        return RemediationConfig(action="NONE")


# ---- Contrastive replay (MEMORIZED) ----

def contrastive_replay_step(
    model: nn.Module,
    original_ids: torch.Tensor,
    perturbed_ids: torch.Tensor,
    device: torch.device,
    contrastive_weight: float = 0.3,
    repr_layer: int = 11,
) -> torch.Tensor:
    """
    Combined CE + contrastive loss (spec §5.2).

    CE on both original and perturbed forces correct predictions.
    Contrastive term pushes mid-layer representations together,
    forcing the model to extract structure rather than memorize tokens.

    L = (CE_orig + CE_pert)/2 + α * (1 - cos(repr_orig, repr_pert))
    """
    orig = original_ids.to(device)
    pert = perturbed_ids.to(device)

    if orig.dim() == 1:
        orig = orig.unsqueeze(0)
    if pert.dim() == 1:
        pert = pert.unsqueeze(0)

    out_orig = model(input_ids=orig, labels=orig, output_hidden_states=True)
    out_pert = model(input_ids=pert, labels=pert, output_hidden_states=True)

    ce_loss = (out_orig.loss + out_pert.loss) / 2

    # Mean-pooled representations at middle layer
    h_orig = out_orig.hidden_states[repr_layer + 1][0].mean(dim=0)
    h_pert = out_pert.hidden_states[repr_layer + 1][0].mean(dim=0)
    h_orig = h_orig / (h_orig.norm() + 1e-8)
    h_pert = h_pert / (h_pert.norm() + 1e-8)

    contrastive_loss = 1.0 - torch.dot(h_orig, h_pert)

    return ce_loss + contrastive_weight * contrastive_loss


# ---- Noise injection (FRAGILE) ----

def noise_injection_step(
    model: nn.Module,
    input_ids: torch.Tensor,
    device: torch.device,
    dropout_rate: float = 0.1,
) -> torch.Tensor:
    """
    Denoising autoregressive step (spec §5.3).

    Corrupt input with random token replacement, but keep original labels.
    Forces the model to rely on structural context, not local n-grams.
    """
    ids = input_ids.to(device)
    if ids.dim() == 1:
        ids = ids.unsqueeze(0)

    vocab_size = model.config.vocab_size if hasattr(model, 'config') else model.base_model.config.vocab_size

    noisy = ids.clone()
    noise_mask = torch.rand(ids.shape, device=device) < dropout_rate
    noise_mask[:, :10] = False  # Preserve first 10 tokens as clean context

    n_corrupted = noise_mask.sum().item()
    if n_corrupted > 0:
        random_tokens = torch.randint(0, vocab_size, (int(n_corrupted),), device=device)
        noisy[noise_mask] = random_tokens

    # Labels = ORIGINAL clean tokens (denoising objective)
    outputs = model(input_ids=noisy, labels=ids)
    return outputs.loss


# ---- Generative rehearsal (RIGID) ----

def generative_rehearsal_step(
    model: nn.Module,
    input_ids: torch.Tensor,
    device: torch.device,
    prefix_fraction: float = 0.25,
    kl_diversity_weight: float = 0.1,
    target_entropy: float = 3.0,
) -> torch.Tensor:
    """
    CE + entropy regularization (spec §5.4).

    Standard CE maintains recognition. Entropy penalty pushes the model
    toward moderate prediction entropy in the continuation region —
    confident enough to be correct, diverse enough to show genuine
    structural understanding rather than single-sequence memorization.

    Target entropy ≈ log(20) ≈ 3.0 (roughly 20 plausible tokens per position).
    """
    ids = input_ids.to(device)
    if ids.dim() == 1:
        ids = ids.unsqueeze(0)

    prefix_len = max(32, int(ids.shape[1] * prefix_fraction))

    outputs = model(input_ids=ids, labels=ids)
    ce_loss = outputs.loss

    # Entropy of continuation region
    logits = outputs.logits[0, prefix_len - 1:-1, :]
    probs = torch.softmax(logits.float(), dim=-1)
    log_probs = torch.log(probs + 1e-10)
    entropy = -(probs * log_probs).sum(dim=-1)

    # Squared penalty around target entropy
    entropy_penalty = ((entropy - target_entropy) ** 2).mean()

    return ce_loss + kl_diversity_weight * entropy_penalty


# ---- Selective rollback (INTERFERENCE) ----

def selective_rollback(
    model: nn.Module,
    ewc: EWCRegularizer,
    rollback_fraction: float = 0.5,
) -> int:
    """
    Partially revert high-damage LoRA parameters toward pre-Phase-1 anchor (spec §5.5).

    damage_i = F_ii * |θ_i - θ*_i|
    Roll back the top 20% most damaging parameter elements by moving
    them rollback_fraction of the way back to anchor.
    """
    n_rolled = 0
    n_total = 0

    with torch.no_grad():
        for name, param in model.named_parameters():
            if not param.requires_grad:
                continue
            if name not in ewc.anchor_params or name not in ewc.fisher:
                continue

            anchor = ewc.anchor_params[name].to(param.device)
            fisher = ewc.fisher[name].to(param.device)

            delta_mag = (param.data - anchor).abs()
            damage = fisher * delta_mag

            if damage.numel() == 0:
                continue

            threshold = torch.quantile(damage.flatten().float(), 0.8)
            rollback_mask = damage > threshold

            n_to_roll = rollback_mask.sum().item()
            n_total += param.numel()

            if n_to_roll > 0:
                param.data[rollback_mask] = (
                    (1 - rollback_fraction) * param.data[rollback_mask] +
                    rollback_fraction * anchor[rollback_mask]
                )
                n_rolled += n_to_roll

    pct = n_rolled / max(n_total, 1) * 100
    print(f"  Rolled back {n_rolled} parameter elements ({pct:.1f}%)")
    return n_rolled

In [ ]:
# ╔═══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 37: INTERLEAVED REMEDIATION LOOP                                  ║
# ║  (spec §7.3)                                                             ║
# ║                                                                          ║
# ║  Instead of sequential per-type remediation, interleave all types        ║
# ║  in a single training loop. Each step randomly samples one episode       ║
# ║  and applies its type-specific loss.                                     ║
# ╚═══════════════════════════════════════════════════════════════════════════╝

def run_interleaved_remediation(
    model: nn.Module,
    diagnoses: Dict[int, EpisodeDiagnosis],
    episodes: List[Episode],
    probes: List[GeneralizationProbe],
    config: Phase2Config,
    device: torch.device,
    tokenizer=None,
    eval_corpus: List[str] = None,
    log_interval: int = 20,
) -> Dict[str, dict]:
    """
    Run all non-rollback remediations in a single interleaved loop (spec §7.3).

    Rollback is applied BEFORE this function (no gradients needed).
    This function handles: CONTRASTIVE (memorized), NOISE (fragile), GENERATIVE (rigid).
    """

    # Build remediation pool: list of (strategy_name, episode, perturbation_probes)
    pool = []
    probes_by_ep = defaultdict(list)
    for p in probes:
        if p.method == "token_perturbation":
            probes_by_ep[p.source_episode_id].append(p)

    ep_lookup = {ep.episode_id: ep for ep in episodes}

    memorized_count = 0
    fragile_count = 0
    rigid_count = 0

    for eid, diag in diagnoses.items():
        if eid not in ep_lookup:
            continue
        ep = ep_lookup[eid]

        if diag.outcome == "MEMORIZED":
            ep_perturbs = probes_by_ep.get(eid, [])
            if ep_perturbs:
                pool.append(("contrastive", ep, ep_perturbs))
                memorized_count += 1
        elif diag.outcome == "FRAGILE":
            pool.append(("noise", ep, None))
            fragile_count += 1
        elif diag.outcome == "RIGID":
            pool.append(("generative", ep, None))
            rigid_count += 1

    if not pool:
        print("  No episodes need remediation.")
        return {}

    # Compute total steps: proportional to pool size, capped
    total_steps = min(
        memorized_count * config.contrastive_steps +
        fragile_count * config.noise_steps_per_level * 4 +
        rigid_count * config.generative_steps,
        2000,  # Hard cap
    )

    print(f"  Remediation pool: {len(pool)} episodes "
          f"({memorized_count} contrastive, {fragile_count} noise, {rigid_count} generative)")
    print(f"  Total steps: {total_steps}")

    # After LoRA merge_and_unload(), all params have requires_grad=False.
    # Re-enable gradients on the layers we want to fine-tune during remediation.
    # Target only the last few transformer layers to minimize forgetting.
    n_layers = len(model.model.layers)
    remediation_layers = list(model.model.layers[n_layers - 4:])  # last 4 layers
    for layer in remediation_layers:
        for p in layer.parameters():
            p.requires_grad = True

    trainable = [p for p in model.parameters() if p.requires_grad]
    assert len(trainable) > 0, "No trainable params — check layer access path"
    optimizer = torch.optim.AdamW(trainable, lr=config.base_lr * 0.03, weight_decay=0.01)

    pre_ppl = evaluate_perplexity(model, tokenizer, eval_corpus[:200], device, label="remediation_baseline")["perplexity"]
    model.train()
    running_loss = 0.0
    strategy_losses = defaultdict(list)

    total_steps = min(len(pool) * 4, total_steps)

    for step in range(total_steps):
        strategy, episode, ep_probes = random.choice(pool)

        if strategy == "contrastive" and ep_probes:
            probe = random.choice(ep_probes)
            loss = contrastive_replay_step(
                model, episode.input_ids, probe.probe_ids, device,
            )
        elif strategy == "noise":
            dropout_rate = random.uniform(0.05, 0.20)
            loss = noise_injection_step(model, episode.input_ids, device, dropout_rate)
        elif strategy == "generative":
            loss = generative_rehearsal_step(model, episode.input_ids, device)
        else:
            continue

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(trainable, max_norm=1.0)
        optimizer.step()

        loss_val = loss.item()
        running_loss += loss_val
        strategy_losses[strategy].append(loss_val)

        if (step + 1) % log_interval == 0:
            avg = running_loss / log_interval
            print(f"  Remediation step {step+1}/{total_steps} | Loss: {avg:.4f}")
            running_loss = 0.0
        if (step + 1) % 50 == 0:
            model.eval()
            with torch.no_grad():
                check_eval = evaluate_perplexity(model, tokenizer, eval_corpus[:200], device, label="remediation_check")
            if check_eval["perplexity"] > pre_ppl * 1.03:
                print(f"  Early stopping at step {step+1}: PPL {check_eval['perplexity']:.2f} exceeds pre-remediation {pre_ppl:.2f}")
                break
            model.train()

    model.eval()

    results = {}
    for strat, losses in strategy_losses.items():
        results[strat] = {
            "n_steps": len(losses),
            "mean_loss": float(np.mean(losses)),
            "final_loss": float(np.mean(losses[-10:])) if len(losses) >= 10 else float(np.mean(losses)),
        }
        print(f"  {strat}: {len(losses)} steps, mean_loss={np.mean(losses):.4f}")

    for layer in remediation_layers:
        for p in layer.parameters():
            p.requires_grad = False

    return results

In [ ]:
# ╔═══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 38: PHASE 2 DIAGNOSTICS                                           ║
# ╚═══════════════════════════════════════════════════════════════════════════╝

def plot_phase2_diagnostics(
    diagnoses: Dict[int, EpisodeDiagnosis],
    probes: List[GeneralizationProbe],
    save_path: str = "system3_phase2_diagnostics.png",
):
    """3-panel Phase 2 visualization."""
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Panel 1: Diagnosis distribution pie chart
    outcome_counts = Counter(d.outcome for d in diagnoses.values())
    colors = {
        "GENERALIZED": "#2ecc71",
        "MEMORIZED": "#e74c3c",
        "FRAGILE": "#f39c12",
        "RIGID": "#9b59b6",
        "INTERFERENCE": "#e74c3c",
        "UNKNOWN": "#95a5a6",
    }
    labels = list(outcome_counts.keys())
    sizes = list(outcome_counts.values())
    pie_colors = [colors.get(l, "#999") for l in labels]
    axes[0].pie(sizes, labels=labels, colors=pie_colors, autopct='%1.0f%%', startangle=90)
    axes[0].set_title("Diagnosis Distribution")

    # Panel 2: Generalization scores by probe method
    prefix_scores = [p.generalization_score for p in probes if p.method == "prefix_continuation"]
    perturb_scores = [p.generalization_score for p in probes if p.method == "token_perturbation"]

    if prefix_scores:
        axes[1].hist(prefix_scores, bins=40, alpha=0.6, label="Prefix", color="#3498db")
    if perturb_scores:
        axes[1].hist(perturb_scores, bins=40, alpha=0.6, label="Perturbation", color="#e67e22")
    axes[1].axvline(0, color="black", linestyle="--", alpha=0.5)
    axes[1].axvline(0.03, color="green", linestyle="--", alpha=0.5, label="Gen threshold")
    axes[1].axvline(-0.03, color="red", linestyle="--", alpha=0.5, label="Interf threshold")
    axes[1].set_title("Generalization Score Distribution")
    axes[1].set_xlabel("Score")
    axes[1].legend(fontsize=8)

    # Panel 3: Diagnosis by surprise type
    type_outcome = defaultdict(lambda: Counter())
    for d in diagnoses.values():
        if d.surprise_type:
            type_outcome[d.surprise_type][d.outcome] += 1

    if type_outcome:
        types = sorted(type_outcome.keys())
        outcomes = sorted(set(o for c in type_outcome.values() for o in c.keys()))
        x = np.arange(len(types))
        width = 0.8 / max(len(outcomes), 1)

        for i, outcome in enumerate(outcomes):
            vals = [type_outcome[t].get(outcome, 0) for t in types]
            axes[2].bar(x + i * width, vals, width, label=outcome,
                       color=colors.get(outcome, "#999"), alpha=0.8)

        axes[2].set_xticks(x + width * len(outcomes) / 2)
        axes[2].set_xticklabels(types, rotation=15, ha='right', fontsize=8)
        axes[2].set_title("Diagnosis by Surprise Type")
        axes[2].legend(fontsize=7)

    plt.suptitle("System 3 Phase 2: REM Abstraction Diagnostics", fontsize=14)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved to {save_path}")

In [ ]:
# ╔═══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 39: RUN SYSTEM 3 PHASE 2                                          ║
# ║  (spec §6, §7.7)                                                         ║
# ║                                                                          ║
# ║  Requires:                                                                ║
# ║    - pre_model:  model BEFORE Phase 1 (base model or prev cycle end)     ║
# ║    - post_model: model AFTER Phase 1 (merged LoRA)                       ║
# ║    - buffer, ewc from Phase 1                                            ║
# ║  Produces:                                                                ║
# ║    - remediated model (or reverted to post_phase1 if Phase 2 hurt)       ║
# ╚═══════════════════════════════════════════════════════════════════════════╝

def run_system3_phase2(
    pre_model: nn.Module,
    post_model: nn.Module,
    batch_episodes: List[Episode],
    ewc: EWCRegularizer,
    tokenizer,
    eval_corpus: List[str],
    device: torch.device,
    config: Phase2Config = Phase2Config(),
    max_degradation: float = 0.03,
) -> Tuple[nn.Module, Phase2Report]:
    """
    Complete Phase 2: REM abstraction (spec §6).

    Steps:
        1. Generate generalization probes from post-Phase-1 model
        2. Score probes against pre vs post Phase 1
        3. Diagnose each episode (generalized / memorized / fragile / rigid / interference)
        4. Apply selective rollback for INTERFERENCE episodes
        5. Run interleaved remediation for MEMORIZED / FRAGILE / RIGID
        6. Verify Phase 2 didn't make things worse (safety gate, spec §7.7)
    """

    print("\n" + "=" * 65)
    print("  SYSTEM 3 PHASE 2: REM ABSTRACTION")
    print("=" * 65)
    phase2_start = time.time()

    # ---- 1. Generate probes ----
    print("\n[1/6] Generating generalization probes...")
    probes = generate_generalization_probes(
        post_model, tokenizer, batch_episodes, device,
        n_probes_per_episode=config.n_probes_per_episode,
        max_episodes=config.max_episodes_to_probe,
    )

    if not probes:
        print("  No probes generated. Skipping Phase 2.")
        return post_model, Phase2Report(
            n_episodes=len(batch_episodes), n_probes=0,
            diagnosis_distribution={}, remediation_results={},
            mean_generalization_phase1=0, mean_generalization_phase2=0,
            phase2_delta=0,
        )

    # ---- 2. Score probes ----
    print("\n[2/6] Scoring generalization (pre vs post Phase 1)...")
    probes = score_generalization(pre_model, post_model, probes, device)

    # ---- 3. Diagnose ----
    print("\n[3/6] Diagnosing episodes...")
    diagnoses = diagnose_episodes(
        probes, batch_episodes,
        gen_threshold=config.generalization_threshold,
        interf_threshold=config.interference_threshold,
    )

    plot_phase2_diagnostics(diagnoses, probes)

    # ---- 4. Selective rollback for INTERFERENCE ----
    interference_eids = [eid for eid, d in diagnoses.items() if d.outcome == "INTERFERENCE"]
    if interference_eids:
        print(f"\n[4/6] Selective rollback for {len(interference_eids)} INTERFERENCE episodes...")
        selective_rollback(post_model, ewc, rollback_fraction=0.5)
    else:
        print("\n[4/6] No interference detected. Skipping rollback.")

    # ---- 5. Interleaved remediation ----
    print("\n[5/6] Running interleaved remediation...")
    phase1_ppl_eval = evaluate_perplexity(
        post_model, tokenizer, eval_corpus, device, label="Pre-Phase2",
    )

    remediation_results = run_interleaved_remediation(
        model=post_model,
        diagnoses=diagnoses,
        episodes=batch_episodes,
        probes=probes,
        config=config,
        device=device,
        tokenizer=tokenizer,
        eval_corpus=eval_corpus,
    )

    # ---- 6. Safety gate (spec §7.7) ----
    print(f"\n[6/6] Post-Phase-2 verification...")
    phase2_ppl_eval = evaluate_perplexity(
        post_model, tokenizer, eval_corpus, device, label="Post-Phase2",
    )

    ppl_change = (phase2_ppl_eval["perplexity"] - phase1_ppl_eval["perplexity"]) / \
                 max(phase1_ppl_eval["perplexity"], 1e-8)

    # Re-score probes to measure Phase 2 improvement
    phase1_gen_scores = [p.generalization_score for p in probes if p.generalization_score != 0]
    mean_gen_phase1 = float(np.mean(phase1_gen_scores)) if phase1_gen_scores else 0.0

    # Quick re-score on a sample of probes
    post_model.eval()
    phase2_improvements = []
    sample_probes = probes[:min(200, len(probes))]
    with torch.no_grad():
        for p in sample_probes:
            ids = p.probe_ids.unsqueeze(0).to(device)
            if ids.shape[1] < 4:
                continue
            out = post_model(input_ids=ids, labels=ids)
            new_loss = out.loss.item()
            if p.pre_consolidation_loss > 0:
                gen_score = (p.pre_consolidation_loss - new_loss) / p.pre_consolidation_loss
                phase2_improvements.append(gen_score)

    mean_gen_phase2 = float(np.mean(phase2_improvements)) if phase2_improvements else 0.0
    phase2_delta = mean_gen_phase2 - mean_gen_phase1

    # Build report
    diag_dist = dict(Counter(d.outcome for d in diagnoses.values()))
    report = Phase2Report(
        n_episodes=len(batch_episodes),
        n_probes=len(probes),
        diagnosis_distribution=diag_dist,
        remediation_results=remediation_results,
        mean_generalization_phase1=mean_gen_phase1,
        mean_generalization_phase2=mean_gen_phase2,
        phase2_delta=phase2_delta,
        diagnoses=diagnoses,
    )

    # Safety gate decision
    if ppl_change > max_degradation:
        print(f"\n  ✗ Phase 2 degraded PPL by {ppl_change*100:.2f}% "
              f"(exceeds {max_degradation*100:.0f}% tolerance)")
        print(f"  NOTE: Remediation weights are already in post_model.")
        print(f"  Consider reverting to Phase 1 checkpoint externally.")
        report.remediation_results["SAFETY_GATE"] = {"passed": False, "ppl_change": ppl_change}
    else:
        if ppl_change < -0.01:
            print(f"\n  ✓ Phase 2 improved PPL by {-ppl_change*100:.2f}%")
        else:
            print(f"\n  ✓ Phase 2 effect on PPL: {ppl_change*100:+.2f}% (within tolerance)")
        report.remediation_results["SAFETY_GATE"] = {"passed": True, "ppl_change": ppl_change}

    elapsed = time.time() - phase2_start

    print(f"\n{'=' * 65}")
    print(f"  SYSTEM 3 PHASE 2 COMPLETE — {elapsed:.1f}s")
    print(f"  Generalization (Phase 1): {mean_gen_phase1*100:+.2f}%")
    print(f"  Generalization (Phase 2): {mean_gen_phase2*100:+.2f}%")
    print(f"  Phase 2 added: {phase2_delta*100:+.2f}% generalization")
    print(f"  Diagnosis: {diag_dist}")
    print(f"{'=' * 65}")

    return post_model, report

In [ ]:
# ╔═══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 40: LAUNCH SYSTEM 3 PHASE 2                                       ║
# ║                                                                          ║
# ║  Requires:                                                                ║
# ║    - consolidated_model from Phase 1 (Cell 32)                            ║
# ║    - model (original base model, or reload it as pre_model)               ║
# ║    - buffer, eval_corpus                                                  ║
# ║    - ewc from Phase 1 (if you want rollback capability)                   ║
# ║                                                                          ║
# ║  Note: For cycle 1, pre_model = original base model before any LoRA.     ║
# ║  For cycle 2+, pre_model = end-of-previous-cycle model.                  ║
# ║                                                                          ║
# ║  Uncomment and run.                                                      ║
# ╚═══════════════════════════════════════════════════════════════════════════╝

# Reload base model as the pre-Phase-1 reference
from transformers import AutoModelForCausalLM
pre_model = AutoModelForCausalLM.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0").to(device)
pre_model.eval()
#
# Get the batch episodes that Phase 1 trained on
phase1_batch = prepare_consolidation_batch(
    buffer, n_episodes=min(500, len(buffer.episodes)), strategy="balanced",
)
#
# Run Phase 2
final_model, phase2_report = run_system3_phase2(
    pre_model=pre_model,
    post_model=consolidated_model,
    batch_episodes=phase1_batch.episodes,
    ewc=consolidation_ewc,                          # From Phase 1 Cell 31
    tokenizer=tokenizer,
    eval_corpus=eval_corpus,
    device=device,
    config=Phase2Config(
        n_probes_per_episode=3,
        max_episodes_to_probe=75,     # Keep compute bounded
    ),
)

In [ ]:
# ╔═══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 41: STATE REPRESENTATION + CONFIG                                  ║
# ║  (spec §3, §7)                                                            ║
# ║                                                                          ║
# ║  Add after Cell 40 (System 3 Phase 2 launch).                            ║
# ║  System 4 is a rule-based policy with adaptive parameters,               ║
# ║  NOT a neural network.                                                    ║
# ╚═══════════════════════════════════════════════════════════════════════════╝

import json
from collections import deque

@dataclass
class EvalSnapshot:
    """One evaluation measurement."""
    cycle: int
    perplexity: float
    loss: float
    timestamp: float
    compute_cost: float         # GPU-hours for this cycle


@dataclass
class SystemState:
    """
    Everything the controller knows about the learning system (spec §3.1).
    Updated after each cycle. All lists are indexed by cycle number.
    """
    # Cycle tracking
    current_cycle: int = 0
    total_compute_used: float = 0.0
    compute_budget: float = float('inf')
    buffer_saturated: bool = False
    force_retrain_heads: bool = False
    coreset_stale: bool = False

    # Performance trajectory
    eval_history: List[EvalSnapshot] = field(default_factory=list)
    ppl_deltas: List[float] = field(default_factory=list)

    # Surprise landscape
    mean_surprise_per_cycle: List[float] = field(default_factory=list)

    # Buffer dynamics
    buffer_size_per_cycle: List[int] = field(default_factory=list)
    buffer_turnover_per_cycle: List[float] = field(default_factory=list)
    consolidation_rate_per_cycle: List[float] = field(default_factory=list)

    # Phase 2 metrics (sparse: only cycles where Phase 2 ran)
    generalization_rate_per_cycle: List[float] = field(default_factory=list)
    memorization_rate_per_cycle: List[float] = field(default_factory=list)
    interference_rate_per_cycle: List[float] = field(default_factory=list)

    # Surprise type distribution
    type_distribution_per_cycle: List[Dict[str, float]] = field(default_factory=list)

    # EWC balance
    ewc_penalty_ratio_per_cycle: List[float] = field(default_factory=list)
    # Resonance tracking (PhysMem-inspired)
    resonance_per_cycle: List[float] = field(default_factory=list)
    novel_territory_cycles: List[int] = field(default_factory=list)

@dataclass
class CyclePlan:
    """Blueprint for one wake-sleep cycle, produced by the planner."""
    cycle_number: int = 0
    should_execute: bool = True
    stop_reason: str = ""

    # Wake
    wake_corpus_size: int = 3000
    # Sleep Phase 1
    consolidation_epochs: int = 3
    lora_rank: int = 16
    consolidation_batch_size: int = 500
    # Sleep Phase 2
    run_phase2: bool = False
    # Maintenance
    refresh_coreset: bool = False
    retrain_heads: bool = True
    gate_z_threshold: float = 1.5


@dataclass
class StopDecision:
    should_stop: bool
    reason: str


@dataclass
class ConsolidationAssessment:
    recommendation: str       # CONTINUE | REDUCE | INCREASE | PAUSE
    reason: str
    suggested_epoch_multiplier: float = 1.0


@dataclass
class ConvergenceDiagnosis:
    bottleneck: str
    description: str
    recommendation: str = ""

@dataclass
class ResonanceAssessment:
    """
    Measures how much the current episode batch relates to existing
    consolidated knowledge. Low resonance = novel territory.

    Inspired by PhysMem's two-stage retrieval: when resonance is low,
    the system drops back to cautious fresh learning instead of applying
    potentially misleading prior knowledge.
    """
    resonance_score: float          # 0.0 = completely novel, 1.0 = highly familiar
    n_consolidated_neighbors: int   # How many consolidated episodes were found nearby
    mean_neighbor_similarity: float # Average cosine sim to nearest consolidated episodes
    is_novel_territory: bool        # True if resonance < threshold

    # Recommended adjustments
    ewc_multiplier: float = 1.0     # >1 = increase protection
    lr_multiplier: float = 1.0      # <1 = learn more cautiously
    skip_phase2: bool = False       # True if no prior learning to verify against

    # Diagnostics
    batch_centroid_norm: float = 0.0
    nearest_consolidated_type: str = ""

@dataclass
class ControllerConfig:
    """Top-level configuration for the metacognitive controller."""
    compute_budget: float = 10.0        # GPU-hours
    base_wake_size: int = 3000
    min_wake_passages: int = 200
    max_wake_passages: int = 5000
    buffer_capacity: int = 1600          # Per our optimization recommendation
    coreset_size: int = 200
    default_lora_rank: int = 16
    default_consolidation_epochs: int = 3
    default_gate_z: float = 1.5


@dataclass
class TrainingReport:
    """Final summary after all cycles complete."""
    total_cycles: int
    total_compute_hours: float
    initial_perplexity: float
    final_perplexity: float
    total_improvement_pct: float
    per_cycle_improvements: List[float]
    convergence_diagnosis: ConvergenceDiagnosis
    cycle_reports: list


@dataclass
class CycleReport:
    """Summary of one complete wake-sleep cycle."""
    cycle: int
    buffer_size: int
    new_episodes: int
    n_passages_processed: int
    phase1_ppl: float
    final_ppl: float
    final_loss: float
    mean_surprise: float
    type_distribution: Dict[str, float]
    consolidation_rate: float
    phase2_report: Optional[Phase2Report] = None

In [ ]:
# ╔═══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 42: DECISION LOG                                                   ║
# ║  (spec §9.4)                                                              ║
# ║                                                                          ║
# ║  Records every controller decision with inputs and rationale for         ║
# ║  post-hoc debugging.                                                     ║
# ╚═══════════════════════════════════════════════════════════════════════════╝

class DecisionLog:
    """Transparent decision recording (spec §9.4)."""

    def __init__(self):
        self.entries: List[dict] = []

    def log(
        self,
        cycle: int,
        decision_type: str,
        decision_value,
        inputs: dict,
        rationale: str,
    ):
        self.entries.append({
            "cycle": cycle,
            "type": decision_type,
            "value": str(decision_value),
            "inputs": {k: str(v) for k, v in inputs.items()},
            "rationale": rationale,
            "timestamp": time.time(),
        })

    def explain_cycle(self, cycle: int) -> str:
        cycle_entries = [e for e in self.entries if e["cycle"] == cycle]
        lines = [f"Cycle {cycle} Decisions:"]
        for e in cycle_entries:
            lines.append(f"  {e['type']}: {e['value']}")
            lines.append(f"    Rationale: {e['rationale']}")
        return "\n".join(lines)

    def dump(self, filepath: str):
        with open(filepath, 'w') as f:
            json.dump(self.entries, f, indent=2, default=str)
        print(f"Decision log saved to {filepath}")

In [ ]:
# ╔═══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 43: WITHIN-CYCLE MONITORS                                         ║
# ║  (spec §4.1, §4.3)                                                       ║
# ╚═══════════════════════════════════════════════════════════════════════════╝

class WakePhaseMonitor:
    """
    Real-time wake phase monitor (spec §4.1).
    Terminates early when surprise saturates or episode creation dries up.
    """

    def __init__(
        self,
        min_passages: int = 200,
        max_passages: int = 5000,
        saturation_window: int = 100,
        saturation_threshold: float = 0.02,
    ):
        self.min_passages = min_passages
        self.max_passages = max_passages
        self.saturation_window = saturation_window
        self.saturation_threshold = saturation_threshold

        self.passage_count = 0
        self.recent_surprises: deque = deque(maxlen=saturation_window * 2)
        self.episode_creation_rate: deque = deque(maxlen=saturation_window)

    def on_passage_processed(self, mean_surprise: float, n_episodes_created: int):
        self.passage_count += 1
        self.recent_surprises.append(mean_surprise)
        self.episode_creation_rate.append(n_episodes_created)

    def should_stop_wake(self) -> Tuple[bool, str]:
        if self.passage_count < self.min_passages:
            return False, ""
        if self.passage_count >= self.max_passages:
            return True, "reached maximum passages"

        # Surprise saturation: rolling mean stopped changing
        if len(self.recent_surprises) >= self.saturation_window * 2:
            first_half = list(self.recent_surprises)[:self.saturation_window]
            second_half = list(self.recent_surprises)[self.saturation_window:]
            mean_first = np.mean(first_half)
            mean_second = np.mean(second_half)
            rel_change = abs(mean_second - mean_first) / (mean_first + 1e-8)
            if rel_change < self.saturation_threshold:
                return True, f"surprise saturated (Δ={rel_change:.4f})"

        # Episode creation drought
        if len(self.episode_creation_rate) >= self.saturation_window:
            rate = sum(self.episode_creation_rate) / len(self.episode_creation_rate)
            if rate < 0.01:
                return True, f"episode creation drought (rate={rate:.4f})"

        return False, ""


class DynamicEWCCalibrator:
    """
    Monitors task/EWC loss ratio during Phase 1 and adjusts lambda (spec §4.3).
    Keeps the ratio in [0.2, 5.0] so neither term dominates.
    """

    def __init__(
        self,
        target_ratio: float = 1.0,
        adjustment_rate: float = 0.1,
        ratio_bounds: Tuple[float, float] = (0.2, 5.0),
        check_interval: int = 20,
    ):
        self.target_ratio = target_ratio
        self.adjustment_rate = adjustment_rate
        self.ratio_bounds = ratio_bounds
        self.check_interval = check_interval
        self.history: List[dict] = []

    def on_training_step(
        self,
        step: int,
        task_loss: float,
        ewc_penalty: float,
        ewc_regularizer: EWCRegularizer,
    ):
        self.history.append({"task": task_loss, "ewc": ewc_penalty})

        if step % self.check_interval != 0 or step == 0:
            return

        recent = self.history[-self.check_interval:]
        mean_task = np.mean([h["task"] for h in recent])
        mean_ewc = np.mean([h["ewc"] for h in recent])

        if mean_ewc < 1e-10:
            return

        ratio = mean_task / mean_ewc

        if ratio < self.ratio_bounds[0]:
            ewc_regularizer.lambda_ewc *= (1.0 - self.adjustment_rate)
        elif ratio > self.ratio_bounds[1]:
            ewc_regularizer.lambda_ewc *= (1.0 + self.adjustment_rate)

        ewc_regularizer.lambda_ewc = max(50, min(20000, ewc_regularizer.lambda_ewc))

    def get_mean_ratio(self) -> float:
        if not self.history:
            return 1.0
        recent = self.history[-min(50, len(self.history)):]
        mean_task = np.mean([h["task"] for h in recent])
        mean_ewc = np.mean([h["ewc"] for h in recent])
        return mean_task / (mean_ewc + 1e-10)

In [ ]:
# ╔═══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 44: BETWEEN-CYCLE DECISION FUNCTIONS                              ║
# ║  (spec §5)                                                                ║
# ╚═══════════════════════════════════════════════════════════════════════════╝

def should_stop_training(state: SystemState) -> StopDecision:
    """Global stopping criteria (spec §5.2)."""

    # Budget exhausted
    if state.total_compute_used >= state.compute_budget:
        return StopDecision(True,
            f"Compute budget exhausted ({state.total_compute_used:.2f}/{state.compute_budget:.2f} hrs)")

    # Convergence: <0.2% change for 5 consecutive cycles
    if len(state.ppl_deltas) >= 5:
        recent = state.ppl_deltas[-5:]
        if all(abs(d) < 0.002 for d in recent):
            return StopDecision(True, "Converged: <0.2% change for 5 cycles")

    # Divergence: degradation for 3 consecutive cycles
    if len(state.ppl_deltas) >= 3:
        recent = state.ppl_deltas[-3:]
        if all(d < -0.01 for d in recent):
            return StopDecision(True,
                f"Diverging: degradation for 3 cycles ({[f'{d:.4f}' for d in recent]})")

    # Oscillation: 4+ sign changes in 6 cycles
    if len(state.ppl_deltas) >= 6:
        recent = state.ppl_deltas[-6:]
        signs = [1 if d > 0.005 else (-1 if d < -0.005 else 0) for d in recent]
        changes = sum(1 for i in range(1, len(signs))
                      if signs[i] != signs[i-1] and signs[i] != 0)
        if changes >= 4:
            return StopDecision(True, f"Oscillating: {changes} sign changes in 6 cycles")

    # Buffer exhaustion
    if len(state.buffer_size_per_cycle) >= 2:
        if all(s < 10 for s in state.buffer_size_per_cycle[-2:]):
            return StopDecision(True, "Buffer exhausted (<10 episodes for 2 cycles)")

    return StopDecision(False, "")


def assess_consolidation_value(state: SystemState) -> ConsolidationAssessment:
    """Should we adjust consolidation effort? (spec §4.2)"""
    if len(state.ppl_deltas) < 2:
        return ConsolidationAssessment("CONTINUE", "Not enough history")

    recent = state.ppl_deltas[-3:] if len(state.ppl_deltas) >= 3 else state.ppl_deltas
    mean_recent = np.mean(recent)

    # Compute deceleration if we have enough history
    deceleration = 0.0
    if len(state.ppl_deltas) >= 5:
        older = state.ppl_deltas[-6:-3]
        mean_older = np.mean(older)
        deceleration = (mean_older - mean_recent) / (abs(mean_older) + 1e-8)

    if mean_recent < -0.01:
        return ConsolidationAssessment("PAUSE", f"Degrading ({mean_recent:.4f})", 0.0)
    if mean_recent < 0.001:
        return ConsolidationAssessment("REDUCE", f"Negligible ({mean_recent:.5f})", 0.5)
    if deceleration > 0.5:
        return ConsolidationAssessment("REDUCE", f"Decelerating ({deceleration:.2f})", 0.7)
    if deceleration < -0.3 and mean_recent > 0.01:
        return ConsolidationAssessment("INCREASE", f"Accelerating ({deceleration:.2f})", 1.3)

    return ConsolidationAssessment("CONTINUE", "Steady", 1.0)


def compute_wake_size(state: SystemState, base: int = 3000) -> int:
    """Adapt wake phase size (spec §5.3)."""
    if state.current_cycle == 0:
        return base

    if state.buffer_turnover_per_cycle:
        turnover = state.buffer_turnover_per_cycle[-1]
        if turnover < 0.05:
            return max(1000, int(base * 0.5))
        if turnover > 0.3:
            return min(5000, int(base * 1.5))

    if len(state.mean_surprise_per_cycle) >= 2:
        trend = (state.mean_surprise_per_cycle[-1] - state.mean_surprise_per_cycle[-2]) / \
                (state.mean_surprise_per_cycle[-2] + 1e-8)
        if trend > 0.05:
            return min(5000, int(base * 1.3))

    return base


def should_run_phase2(state: SystemState) -> bool:
    """Phase 2 trigger logic (spec §5.4)."""
    if state.current_cycle < 2:
        return False
    if state.current_cycle % 3 == 2:
        return True
    if state.memorization_rate_per_cycle and state.memorization_rate_per_cycle[-1] > 0.4:
        return True
    if state.interference_rate_per_cycle and state.interference_rate_per_cycle[-1] > 0.1:
        return True
    if len(state.ppl_deltas) >= 2:
        recent = np.mean(state.ppl_deltas[-2:])
        if 0.0 < recent < 0.005:
            return True
    return False


def compute_lora_rank(state: SystemState, base: int = 16) -> int:
    """Adapt LoRA rank (spec §5.5)."""
    if state.consolidation_rate_per_cycle:
        rate = state.consolidation_rate_per_cycle[-1]
        if rate < 0.2:
            return min(64, base * 2)
        if len(state.ppl_deltas) >= 2 and np.mean(state.ppl_deltas[-2:]) > 0.02 and rate > 0.5:
            return max(8, base // 2)

    if state.type_distribution_per_cycle:
        under_frac = state.type_distribution_per_cycle[-1].get("undergeneralization", 0)
        if under_frac > 0.4:
            return min(48, int(base * 1.5))

    return base


def compute_gate_threshold(state: SystemState, base: float = 1.5) -> float:
    """Adapt surprise gate z-threshold (spec §5.6)."""
    if len(state.buffer_size_per_cycle) >= 2:
        recent = state.buffer_size_per_cycle[-2:]
        if all(s > 700 for s in recent):    # Near 800 capacity
            return min(2.5, base + 0.3)
        if all(s < 100 for s in recent):
            return max(0.8, base - 0.3)

    if state.consolidation_rate_per_cycle and state.consolidation_rate_per_cycle[-1] > 0.6:
        return min(2.0, base + 0.2)

    return base


def compute_batch_size(state: SystemState, base: int = 500) -> int:
    """Adapt consolidation batch size (spec §5.8)."""
    if state.buffer_size_per_cycle and state.consolidation_rate_per_cycle:
        buf = state.buffer_size_per_cycle[-1]
        rate = state.consolidation_rate_per_cycle[-1]
        if buf > 600 and rate < 0.3:
            return min(1000, int(base * 1.5))

    if state.compute_budget < float('inf') and state.current_cycle > 0:
        remaining = state.compute_budget - state.total_compute_used
        avg_per_cycle = state.total_compute_used / state.current_cycle
        if remaining < avg_per_cycle * 2:
            return max(200, int(base * 0.6))

    return base


def should_refresh_coreset(state: SystemState) -> bool:
    """Coreset refresh trigger (spec §5.7)."""
    if state.current_cycle > 0 and state.current_cycle % 3 == 0:
        return True
    if state.ppl_deltas and state.ppl_deltas[-1] < -0.03:
        return True
    if getattr(state, 'coreset_stale', False):
        return True
    return False


def should_retrain_heads(state: SystemState) -> bool:
    """Prediction head retraining trigger (spec §5.7)."""
    if state.current_cycle == 0:
        return True
    if getattr(state, 'force_retrain_heads', False):
        return True
    if state.ppl_deltas and abs(state.ppl_deltas[-1]) > 0.05:
        return True
    return False

def compute_resonance(
    buffer: 'EpisodicBuffer',
    new_episode_keys: List[torch.Tensor],
    consolidated_threshold: float = 0.3,
    resonance_threshold: float = 0.3,
    k_neighbors: int = 10,
) -> ResonanceAssessment:
    """
    Measure how similar new episodes are to existing consolidated knowledge.

    Process:
        1. Compute centroid of new episode embedding keys
        2. Find k nearest episodes in buffer that are at least partially
           consolidated (score > consolidated_threshold)
        3. Compute mean cosine similarity = resonance score
        4. If resonance < threshold, flag as novel territory and recommend
           protective adjustments

    PhysMem analog: Their two-stage retrieval first does symbolic filtering
    (action type + object properties) then semantic ranking. When resonance
    is low, the system "drops back to fresh learning." We implement the same
    principle using embedding similarity as the resonance measure.

    Args:
        buffer: The episodic buffer containing existing episodes
        new_episode_keys: Embedding keys from newly extracted episodes
        consolidated_threshold: Minimum consolidation_score to count as
                                "existing knowledge"
        resonance_threshold: Below this, territory is considered novel
        k_neighbors: How many consolidated neighbors to check

    Returns:
        ResonanceAssessment with score, flags, and recommended multipliers
    """
    if not new_episode_keys or len(buffer.episodes) == 0:
        return ResonanceAssessment(
            resonance_score=0.5,
            n_consolidated_neighbors=0,
            mean_neighbor_similarity=0.0,
            is_novel_territory=False,
        )

    # Step 1: Compute centroid of new episodes
    # Stack all keys and mean-pool, then L2-normalize
    stacked = torch.stack(new_episode_keys)  # [N, D]
    centroid = stacked.mean(dim=0)           # [D]
    centroid = centroid / (centroid.norm() + 1e-8)

    # Step 2: Find consolidated episodes in buffer
    consolidated_episodes = [
        ep for ep in buffer.episodes.values()
        if ep.consolidation_score >= consolidated_threshold
    ]

    if not consolidated_episodes:
        # No consolidated knowledge exists — everything is novel by definition
        # But this is expected on early cycles, not alarming
        return ResonanceAssessment(
            resonance_score=0.0,
            n_consolidated_neighbors=0,
            mean_neighbor_similarity=0.0,
            is_novel_territory=False,  # Don't flag on early cycles
            ewc_multiplier=1.0,
            lr_multiplier=1.0,
            skip_phase2=False,
            batch_centroid_norm=float(centroid.norm()),
        )

    # Step 3: Compute cosine similarity between centroid and each consolidated episode
    similarities = []
    for ep in consolidated_episodes:
        sim = torch.dot(centroid.cpu(), ep.embedding_key.cpu()).item()
        similarities.append((ep, sim))

    # Sort by similarity (highest first) and take top k
    similarities.sort(key=lambda x: x[1], reverse=True)
    top_k = similarities[:k_neighbors]

    mean_sim = float(np.mean([sim for _, sim in top_k]))
    max_sim = top_k[0][1] if top_k else 0.0
    nearest_type = top_k[0][0].surprise_type if top_k else ""

    # Step 4: Determine resonance and compute adjustments
    #
    # Resonance score: weighted combination of mean and max similarity
    # Max matters because even one highly similar consolidated episode means
    # this territory isn't entirely novel
    resonance = 0.6 * mean_sim + 0.4 * max_sim

    is_novel = resonance < resonance_threshold and len(consolidated_episodes) >= 5
    # Require at least 5 consolidated episodes before declaring novelty —
    # on early cycles, low similarity is just lack of data, not novel domain

    # Compute adjustment multipliers
    if is_novel:
        # Novel territory: increase EWC protection, decrease learning rate
        # Scale adjustments by how novel (lower resonance = more extreme)
        novelty_degree = (resonance_threshold - resonance) / resonance_threshold
        novelty_degree = max(0.0, min(1.0, novelty_degree))

        ewc_mult = 1.0 + 2.0 * novelty_degree      # 1.0x to 3.0x
        lr_mult = 1.0 - 0.6 * novelty_degree        # 1.0x to 0.4x
        skip_p2 = novelty_degree > 0.5              # Skip Phase 2 if very novel
    else:
        ewc_mult = 1.0
        lr_mult = 1.0
        skip_p2 = False

    return ResonanceAssessment(
        resonance_score=resonance,
        n_consolidated_neighbors=len(consolidated_episodes),
        mean_neighbor_similarity=mean_sim,
        is_novel_territory=is_novel,
        ewc_multiplier=ewc_mult,
        lr_multiplier=lr_mult,
        skip_phase2=skip_p2,
        batch_centroid_norm=float(centroid.norm()),
        nearest_consolidated_type=nearest_type,
    )


def print_resonance_assessment(assessment: ResonanceAssessment):
    """Pretty-print resonance results."""
    status = "⚠ NOVEL TERRITORY" if assessment.is_novel_territory else "✓ familiar"
    print(f"\n  Resonance check: {status}")
    print(f"    Score: {assessment.resonance_score:.3f} "
          f"(threshold: 0.3)")
    print(f"    Consolidated neighbors: {assessment.n_consolidated_neighbors}")
    print(f"    Mean similarity: {assessment.mean_neighbor_similarity:.3f}")
    if assessment.is_novel_territory:
        print(f"    → EWC multiplier: {assessment.ewc_multiplier:.2f}x")
        print(f"    → LR multiplier:  {assessment.lr_multiplier:.2f}x")
        if assessment.skip_phase2:
            print(f"    → Skipping Phase 2 (no prior knowledge to verify against)")

In [ ]:
# ╔═══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 45: CORPUS MANAGER                                                ║
# ║  (spec §6.1)                                                              ║
# ║                                                                          ║
# ║  Tracks which passages have been seen, balances exploration              ║
# ║  (new passages), exploitation (high-surprise passages),                   ║
# ║  and revisitation (old passages that may now be surprising).              ║
# ╚═══════════════════════════════════════════════════════════════════════════╝

class CorpusManager:
    """
    Exploration/exploitation manager for corpus passage selection (spec §6.1).

    Each cycle selects passages from three pools:
        30% exploration — never/rarely processed (discover new surprises)
        49% exploitation — high surprise last time (consolidate)
        21% revisitation — oldest-processed (may be surprising after model changes)
    """

    def __init__(self, corpus: List[str]):
        self.corpus = corpus
        self.n_passages = len(corpus)
        self.process_count = np.zeros(self.n_passages, dtype=int)
        self.last_processed_cycle = np.full(self.n_passages, -1, dtype=int)
        self.last_surprise = np.zeros(self.n_passages, dtype=float)

    def select_passages(
        self,
        current_cycle: int,
        n_passages: int,
        explore_fraction: float = 0.3,
    ) -> List[int]:
        n_passages = min(n_passages, self.n_passages)
        n_explore = int(n_passages * explore_fraction)
        n_exploit = int(n_passages * (1 - explore_fraction) * 0.7)
        n_revisit = n_passages - n_explore - n_exploit

        selected = set()

        # Exploration: least-processed first
        explore_priority = -self.process_count + np.random.randn(self.n_passages) * 0.01
        for idx in np.argsort(explore_priority):
            if len(selected) >= n_explore:
                break
            selected.add(int(idx))

        # Exploitation: highest surprise from last processing
        exploit_priority = self.last_surprise.copy()
        exploit_priority[list(selected)] = -np.inf
        unprocessed = self.process_count == 0
        exploit_priority[unprocessed] = -np.inf
        for idx in np.argsort(-exploit_priority):
            if len(selected) >= n_explore + n_exploit:
                break
            if int(idx) not in selected:
                selected.add(int(idx))

        # Revisitation: oldest-processed
        revisit_priority = -self.last_processed_cycle.astype(float)
        revisit_priority[list(selected)] = -np.inf
        revisit_priority[unprocessed] = -np.inf
        for idx in np.argsort(-revisit_priority):
            if len(selected) >= n_passages:
                break
            if int(idx) not in selected:
                selected.add(int(idx))

        # Fill remainder if needed
        if len(selected) < n_passages:
            remaining = set(range(self.n_passages)) - selected
            extra = random.sample(list(remaining), min(n_passages - len(selected), len(remaining)))
            selected.update(extra)

        result = list(selected)
        random.shuffle(result)
        return result[:n_passages]

    def update_after_wake(
        self,
        processed_indices: List[int],
        surprises: List[float],
        current_cycle: int,
    ):
        for idx, surprise in zip(processed_indices, surprises):
            if 0 <= idx < self.n_passages:
                self.process_count[idx] += 1
                self.last_processed_cycle[idx] = current_cycle
                self.last_surprise[idx] = surprise

In [ ]:
# ╔═══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 46: CONVERGENCE DIAGNOSIS                                         ║
# ║  (spec §6.2)                                                              ║
# ╚═══════════════════════════════════════════════════════════════════════════╝

def diagnose_convergence(state: SystemState, buffer_capacity: int = 800) -> ConvergenceDiagnosis:
    """
    When improvement stalls, diagnose the bottleneck (spec §6.2).

    Checks in order: buffer capacity, interference, LoRA capacity,
    corpus exhaustion, memorization dominance.
    """
    if len(state.ppl_deltas) < 5:
        return ConvergenceDiagnosis("INSUFFICIENT_DATA", "Not enough cycles to diagnose.")

    recent = state.ppl_deltas[-5:]
    is_stalled = all(abs(d) < 0.003 for d in recent)

    if not is_stalled:
        return ConvergenceDiagnosis("NOT_CONVERGED", "Model is still improving.")

    # Buffer at capacity
    if state.buffer_size_per_cycle:
        avg_buf = np.mean(state.buffer_size_per_cycle[-3:])
        if avg_buf > buffer_capacity * 0.9:
            return ConvergenceDiagnosis(
                "BUFFER_CAPACITY",
                f"Buffer near capacity ({avg_buf:.0f}/{buffer_capacity}).",
                "Increase buffer max_capacity or raise gate z_threshold.",
            )

    # High interference
    if state.interference_rate_per_cycle and len(state.interference_rate_per_cycle) >= 3:
        avg_interf = np.mean(state.interference_rate_per_cycle[-3:])
        if avg_interf > 0.15:
            return ConvergenceDiagnosis(
                "INTERFERENCE",
                f"High interference ({avg_interf:.1%}).",
                "Increase EWC lambda or reduce learning rate.",
            )

    # Low consolidation rate
    if state.consolidation_rate_per_cycle and len(state.consolidation_rate_per_cycle) >= 3:
        avg_consol = np.mean(state.consolidation_rate_per_cycle[-3:])
        if avg_consol < 0.15:
            return ConvergenceDiagnosis(
                "LORA_CAPACITY",
                f"Low consolidation rate ({avg_consol:.1%}).",
                "Increase LoRA rank or add target modules.",
            )

    # Corpus exhaustion
    if state.mean_surprise_per_cycle and len(state.mean_surprise_per_cycle) >= 5:
        early = np.mean(state.mean_surprise_per_cycle[:3])
        recent_surp = np.mean(state.mean_surprise_per_cycle[-3:])
        reduction = (early - recent_surp) / (early + 1e-8)
        if reduction > 0.3:
            return ConvergenceDiagnosis(
                "CORPUS_EXHAUSTION",
                f"Surprise dropped {reduction:.0%} since start.",
                "Switch to a new/larger corpus or declare victory.",
            )

    # Memorization dominance
    if state.memorization_rate_per_cycle and len(state.memorization_rate_per_cycle) >= 3:
        avg_mem = np.mean(state.memorization_rate_per_cycle[-3:])
        if avg_mem > 0.5:
            return ConvergenceDiagnosis(
                "MEMORIZATION",
                f"High memorization ({avg_mem:.1%}).",
                "Run Phase 2 more frequently or increase contrastive weight.",
            )

    return ConvergenceDiagnosis(
        "UNKNOWN",
        "Stalled but no clear bottleneck.",
        "Try increasing corpus diversity or model capacity.",
    )

In [ ]:
# ╔═══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 47: CYCLE PLANNER + CHANGE BUDGET                                 ║
# ║  (spec §5.1, §9.1)                                                       ║
# ╚═══════════════════════════════════════════════════════════════════════════╝

def plan_next_cycle(
    state: SystemState,
    config: ControllerConfig,
    decision_log: DecisionLog,
    previous_plan: Optional[CyclePlan] = None,
) -> CyclePlan:
    """
    Examine system state and produce a plan for the next cycle (spec §5.1).
    Enforces a change budget of max 2 significant changes per cycle (spec §9.1).
    """
    plan = CyclePlan()
    plan.cycle_number = state.current_cycle + 1

    # Decision 1: Stop?
    stop = should_stop_training(state)
    if stop.should_stop:
        plan.should_execute = False
        plan.stop_reason = stop.reason
        decision_log.log(plan.cycle_number, "stop", True, {}, stop.reason)
        return plan
    plan.should_execute = True

    # Decision 2: Wake size
    plan.wake_corpus_size = compute_wake_size(state, config.base_wake_size)
    decision_log.log(plan.cycle_number, "wake_size", plan.wake_corpus_size,
        {"turnover": state.buffer_turnover_per_cycle[-1] if state.buffer_turnover_per_cycle else "N/A"},
        f"Base={config.base_wake_size}, adapted by turnover/surprise trend")

    # Decision 3: Phase 2?
    plan.run_phase2 = should_run_phase2(state)
    decision_log.log(plan.cycle_number, "phase2", plan.run_phase2,
        {"cycle": state.current_cycle,
         "mem_rate": state.memorization_rate_per_cycle[-1] if state.memorization_rate_per_cycle else "N/A"},
        "Phase 2 trigger check")

    # Decision 4: Consolidation epochs
    assessment = assess_consolidation_value(state)
    plan.consolidation_epochs = max(1, min(6,
        int(config.default_consolidation_epochs * assessment.suggested_epoch_multiplier)))
    decision_log.log(plan.cycle_number, "epochs", plan.consolidation_epochs,
        {"assessment": assessment.recommendation}, assessment.reason)

    # Decision 5: LoRA rank
    plan.lora_rank = compute_lora_rank(state, config.default_lora_rank)

    # Decision 6: Coreset refresh
    plan.refresh_coreset = should_refresh_coreset(state)

    # Decision 7: Head retraining
    plan.retrain_heads = should_retrain_heads(state)

    # Decision 8: Gate threshold
    plan.gate_z_threshold = compute_gate_threshold(state, config.default_gate_z)

    # Tighten gate if buffer was saturated last cycle
    if state.buffer_saturated:
        plan.gate_z_threshold = min(plan.gate_z_threshold + 0.25, 3.0)
        decision_log.log(plan.cycle_number, "gate_tighten", plan.gate_z_threshold,
                         {"reason": "buffer_saturated"})

    # Decision 9: Batch size
    plan.consolidation_batch_size = compute_batch_size(state)

    # ---- Enforce change budget (spec §9.1): max 2 significant changes ----
    if previous_plan is not None:
        plan = _enforce_change_budget(plan, previous_plan, max_changes=2)

    # Print plan
    print(f"\n  CYCLE {plan.cycle_number} PLAN:")
    print(f"    Wake: {plan.wake_corpus_size} passages | "
          f"Phase2: {plan.run_phase2} | Epochs: {plan.consolidation_epochs}")
    print(f"    LoRA rank: {plan.lora_rank} | Gate z: {plan.gate_z_threshold:.2f} | "
          f"Batch: {plan.consolidation_batch_size}")
    print(f"    Coreset refresh: {plan.refresh_coreset} | Retrain heads: {plan.retrain_heads}")

    return plan


def _enforce_change_budget(
    plan: CyclePlan,
    prev: CyclePlan,
    max_changes: int = 2,
) -> CyclePlan:
    """Limit how many parameters change significantly per cycle (spec §9.1)."""
    changes = []
    if abs(plan.lora_rank - prev.lora_rank) > 4:
        changes.append("lora_rank")
    if abs(plan.consolidation_epochs - prev.consolidation_epochs) > 1:
        changes.append("consolidation_epochs")
    if abs(plan.gate_z_threshold - prev.gate_z_threshold) > 0.3:
        changes.append("gate_z_threshold")
    if abs(plan.wake_corpus_size - prev.wake_corpus_size) > 2000:
        changes.append("wake_corpus_size")
    if plan.run_phase2 != prev.run_phase2:
        changes.append("run_phase2")

    if len(changes) > max_changes:
        priority = ["run_phase2", "lora_rank", "consolidation_epochs",
                     "gate_z_threshold", "wake_corpus_size"]
        kept = [c for c in priority if c in changes][:max_changes]
        for c in changes:
            if c not in kept:
                setattr(plan, c, getattr(prev, c))
        print(f"    Change budget: kept {kept}, reverted {[c for c in changes if c not in kept]}")

    return plan

In [ ]:
# ╔═══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 48: TRAINING DASHBOARD                                            ║
# ║  (spec §8)                                                                ║
# ╚═══════════════════════════════════════════════════════════════════════════╝

def plot_training_dashboard(
    state: SystemState,
    save_path: str = "training_dashboard.png",
):
    """6-panel comprehensive dashboard (spec §8.1)."""
    fig, axes = plt.subplots(2, 3, figsize=(20, 10))

    cycles = list(range(len(state.eval_history)))
    ppls = [s.perplexity for s in state.eval_history]

    # Panel 1: Perplexity
    axes[0, 0].plot(cycles, ppls, 'b-o', linewidth=2, markersize=4)
    if len(ppls) > 1:
        axes[0, 0].axhline(ppls[0], color='gray', linestyle='--', alpha=0.5, label='Initial')
    axes[0, 0].set_title("Perplexity")
    axes[0, 0].set_xlabel("Cycle")
    axes[0, 0].legend(fontsize=8)

    # Panel 2: Per-cycle improvement
    if state.ppl_deltas:
        colors = ['#2ecc71' if d > 0 else '#e74c3c' for d in state.ppl_deltas]
        axes[0, 1].bar(range(1, len(state.ppl_deltas) + 1), state.ppl_deltas, color=colors, alpha=0.7)
        axes[0, 1].axhline(0, color='black', linewidth=0.5)
    axes[0, 1].set_title("Per-Cycle Improvement")
    axes[0, 1].set_xlabel("Cycle")
    axes[0, 1].set_ylabel("Δ PPL (+ = better)")

    # Panel 3: Buffer dynamics
    if state.buffer_size_per_cycle:
        axes[0, 2].plot(range(1, len(state.buffer_size_per_cycle) + 1),
                        state.buffer_size_per_cycle, 'purple', linewidth=2, marker='s', markersize=3)
    axes[0, 2].set_title("Buffer Size")
    axes[0, 2].set_xlabel("Cycle")

    # Panel 4: Mean surprise
    if state.mean_surprise_per_cycle:
        axes[1, 0].plot(range(1, len(state.mean_surprise_per_cycle) + 1),
                        state.mean_surprise_per_cycle, 'orange', linewidth=2, marker='d', markersize=3)
    axes[1, 0].set_title("Mean Surprise")
    axes[1, 0].set_xlabel("Cycle")

    # Panel 5: Learning quality rates
    c_start = 1
    if state.consolidation_rate_per_cycle:
        x = range(c_start, c_start + len(state.consolidation_rate_per_cycle))
        axes[1, 1].plot(x, state.consolidation_rate_per_cycle, 'g-o', label='Consolidation', markersize=3)
    if state.generalization_rate_per_cycle:
        x = range(c_start, c_start + len(state.generalization_rate_per_cycle))
        axes[1, 1].plot(x, state.generalization_rate_per_cycle, 'b-^', label='Generalization', markersize=3)
    if state.memorization_rate_per_cycle:
        x = range(c_start, c_start + len(state.memorization_rate_per_cycle))
        axes[1, 1].plot(x, state.memorization_rate_per_cycle, 'r--v', label='Memorization', markersize=3)
    axes[1, 1].set_title("Learning Quality")
    axes[1, 1].set_xlabel("Cycle")
    axes[1, 1].legend(fontsize=8)

    # Panel 6: Surprise type evolution (stacked area)
    if state.type_distribution_per_cycle:
        types = ["vocabulary_gap", "undergeneralization", "out_of_distribution"]
        type_data = {t: [] for t in types}
        for dist in state.type_distribution_per_cycle:
            for t in types:
                type_data[t].append(dist.get(t, 0))
        x = range(1, len(state.type_distribution_per_cycle) + 1)
        axes[1, 2].stackplot(x, *[type_data[t] for t in types],
                             labels=types, colors=['#e74c3c', '#2ecc71', '#3498db'], alpha=0.7)
        axes[1, 2].set_title("Surprise Type Distribution")
        axes[1, 2].set_xlabel("Cycle")
        axes[1, 2].legend(loc='upper right', fontsize=7)
    else:
        axes[1, 2].text(0.5, 0.5, "No type data yet", ha='center', va='center')

    current_ppl = ppls[-1] if ppls else 0
    plt.suptitle(
        f"Stepwise Learning Dashboard — Cycle {state.current_cycle} | "
        f"PPL: {current_ppl:.2f} | Compute: {state.total_compute_used:.2f} hrs",
        fontsize=14,
    )
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved to {save_path}")

In [ ]:
# ╔═══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 49: METACOGNITIVE CONTROLLER (ORCHESTRATOR)                        ║
# ║  (spec §7, §10)                                                           ║
# ║                                                                          ║
# ║  Top-level class that runs the full wake-sleep loop.                     ║
# ║  Coordinates Systems 1-3, makes adaptive decisions, handles              ║
# ║  inter-system state.                                                     ║
# ╚═══════════════════════════════════════════════════════════════════════════╝

class MetacognitiveController:
    """
    Top-level orchestrator (spec §7).
    Manages the wake-sleep loop, makes adaptive decisions per cycle,
    and coordinates all four systems.
    """

    def __init__(
        self,
        model: nn.Module,
        tokenizer,
        corpus: List[str],
        eval_texts: List[str],
        device: torch.device,
        config: ControllerConfig,
    ):
        self.model = model
        self.tokenizer = tokenizer
        self.eval_texts = eval_texts
        self.device = device
        self.config = config

        # System components (initialized lazily or on first cycle)
        self.buffer = EpisodicBuffer(max_capacity=config.buffer_capacity)
        self.coreset: Optional[List[dict]] = None
        self.corpus_manager = CorpusManager(corpus)
        self.ewc: Optional[EWCRegularizer] = None  # Created during Phase 1
        self.head_system: Optional[PredictionHeadSystem] = None


        # State + logging
        self.state = SystemState(compute_budget=config.compute_budget)
        self.decision_log = DecisionLog()
        self.cycle_reports: List[CycleReport] = []
        self.plans: List[CyclePlan] = []
        self._last_phase1_report = None

    def run(self, max_cycles: int = 20) -> TrainingReport:
        """
        Main entry point (spec §10). Runs wake-sleep cycles until
        convergence, budget exhaustion, or max_cycles.
        """
        print("=" * 70)
        print("  STEPWISE LEARNING — METACOGNITIVE CONTROLLER")
        print("=" * 70)
        print(f"  Corpus: {self.corpus_manager.n_passages} passages")
        print(f"  Eval: {len(self.eval_texts)} passages")
        print(f"  Budget: {self.config.compute_budget} GPU-hrs | Max cycles: {max_cycles}")
        print(f"  Buffer capacity: {self.config.buffer_capacity}")
        print("=" * 70)

        # Initial evaluation
        initial_eval = evaluate_perplexity(
            self.model, self.tokenizer, self.eval_texts, self.device, label="Initial",
        )
        self.state.eval_history.append(EvalSnapshot(
            cycle=0, perplexity=initial_eval["perplexity"],
            loss=initial_eval["loss"], timestamp=time.time(), compute_cost=0,
        ))

        # Build initial coreset
        print("\nBuilding initial coreset...")
        self.coreset = build_coreset(
            self.model, self.tokenizer,
            self.corpus_manager.corpus, self.device,
            target_size=self.config.coreset_size,
        )

        # ---- Cycle loop ----
        previous_plan = None

        for cycle in range(max_cycles):
            cycle_start = time.time()

            plan = plan_next_cycle(self.state, self.config, self.decision_log, previous_plan)
            self.plans.append(plan)

            if not plan.should_execute:
                print(f"\n{'=' * 70}")
                print(f"  STOPPING: {plan.stop_reason}")
                print(f"{'=' * 70}")
                break

            # Execute
            report = self._execute_cycle(plan)
            self.cycle_reports.append(report)

            # Update state
            cycle_hours = (time.time() - cycle_start) / 3600
            self._update_state(report, cycle_hours)

            # Print summary
            self._print_summary(plan.cycle_number, cycle_hours)

            # Dashboard every 3 cycles
            if plan.cycle_number % 3 == 0 or plan.cycle_number == 1:
                plot_training_dashboard(self.state)

            previous_plan = plan

        # Final report
        return self._final_report()

    # ---- Private methods ----

    def _execute_cycle(self, plan: CyclePlan) -> CycleReport:
        """Execute one complete wake-sleep cycle per the plan."""

        print(f"\n{'=' * 70}")
        print(f"  CYCLE {plan.cycle_number}")
        print(f"{'=' * 70}")

        # ========== WAKE PHASE ==========
        print(f"\n--- WAKE PHASE ({plan.wake_corpus_size} passages) ---")

        # Select passages
        passage_indices = self.corpus_manager.select_passages(
            plan.cycle_number, plan.wake_corpus_size,
        )
        cycle_corpus = [self.corpus_manager.corpus[i] for i in passage_indices]

        # Update learning objectives for all buffer episodes
        update_episode_objectives(self.buffer.episodes)

        # Configure gate
        self.buffer.surprise_gate.z_threshold = plan.gate_z_threshold

        # Run System 1 → System 2 pipeline
        # (Reusing run_system2 or equivalent; here we call the existing functions)
        # Initialize or retrain prediction heads
        if self.head_system is None or plan.retrain_heads:
            hidden_size = self.model.config.hidden_size
            self.head_system = PredictionHeadSystem(hidden_size).to(self.device)
            # NOTE: heads need training data — on first cycle, use cycle_corpus
            # as a quick bootstrap. For subsequent cycles, plan.retrain_heads
            # controls whether to re-train from the updated model.
            train_prediction_heads(
              base_model=self.model,
              tokenizer=self.tokenizer,
              head_system=self.head_system,
              train_texts=cycle_corpus[:400],
              val_texts=cycle_corpus[400:500],
              device=self.device,
          )


        # Construct tracker with correct 4-arg signature
        tracker = HierarchicalSurpriseTracker(
            base_model=self.model,
            tokenizer=self.tokenizer,
            head_system=self.head_system,
            device=self.device,
        )

        # run_wake_phase returns None; stores data internally
        tracker.run_wake_phase(cycle_corpus)

        # Classify surprise tokens (same pattern as Cell 14 / run_system1)
        thresholds = compute_thresholds(tracker)
        rarity_threshold = compute_rarity_threshold(tracker, rarity_percentile=10)
        categories = classify_surprise(tracker, thresholds, rarity_count_threshold=rarity_threshold)

        # Compute mean surprise from tracker internals
        mean_surprise = float(np.mean(tracker.all_ce)) if tracker.all_ce else 0.0

        # Ingest into buffer — correct arg order: (tracker, categories, buffer, ...)
        n_before = len(self.buffer.episodes)
        ingest_system1_to_buffer(
            tracker, categories, self.buffer,
            self.model, self.tokenizer, self.device,
        )
        n_new = len(self.buffer.episodes) - n_before

        # If buffer hit capacity and rejected >50% of extracted episodes,
        # tighten the gate for next cycle to reduce wasted extraction work.
        rejection_rate = 1 - (n_new / max(n_new + 2027, 1))  # TODO: get actual reject count from ingest
        if len(self.buffer.episodes) >= self.config.buffer_capacity * 0.95:
            # Signal planner to raise z_threshold next cycle
            self.state.buffer_saturated = True
        else:
            self.state.buffer_saturated = False

        # Update corpus manager
                # Extract per-passage mean surprise from tracker.passage_data
        passage_surprises = [
            np.mean([sv.ce_loss for sv in pd.vectors]) if pd.vectors else 0.0
            for pd in tracker.passage_data
        ]
        self.corpus_manager.update_after_wake(
            passage_indices[:len(passage_surprises)],
            passage_surprises[:len(passage_indices)],
            plan.cycle_number,
        )

        # Type distribution
        type_counts = Counter(ep.surprise_type for ep in self.buffer.episodes.values())
        total_ep = max(sum(type_counts.values()), 1)
        type_dist = {t: c / total_ep for t, c in type_counts.items()}

        print(f"  Processed: {len(cycle_corpus)} passages | "
              f"New episodes: {n_new} | Buffer: {len(self.buffer.episodes)}")
        # ========== RESONANCE CHECK (PhysMem-inspired) ==========
        # Measure how similar new episodes are to existing consolidated knowledge.
        # If novel territory: increase EWC, decrease LR, possibly skip Phase 2.
        new_episode_keys = [
            ep.embedding_key for ep in self.buffer.episodes.values()
            if ep.created_at == self.buffer.current_step  # Episodes from this cycle
        ]
        # Fallback: if created_at isn't being updated properly, use the most
        # recent n_new episodes by ID
        if len(new_episode_keys) < n_new:
            sorted_eps = sorted(self.buffer.episodes.values(),
                                key=lambda e: e.episode_id, reverse=True)
            new_episode_keys = [ep.embedding_key for ep in sorted_eps[:n_new]]

        resonance = compute_resonance(
            buffer=self.buffer,
            new_episode_keys=new_episode_keys,
        )
        print_resonance_assessment(resonance)

        # Track resonance over time
        self.state.resonance_per_cycle.append(resonance.resonance_score)
        if resonance.is_novel_territory:
            self.state.novel_territory_cycles.append(plan.cycle_number)

        # Override Phase 2 if resonance says skip
        if resonance.skip_phase2 and plan.run_phase2:
            print(f"  Resonance override: skipping Phase 2 this cycle")
            plan.run_phase2 = False

        # ========== SLEEP PHASE 1: NREM ==========
        print(f"\n--- SLEEP PHASE 1: NREM CONSOLIDATION ---")

        # Snapshot model state before Phase 1 for Phase 2 comparison
        if plan.run_phase2:
            pre_phase1_state = {k: v.clone() for k, v in self.model.state_dict().items()}
        else:
            pre_phase1_state = None

        consolidated_model, phase1_report, phase1_ewc = run_system3_phase1(
            model=self.model,
            tokenizer=self.tokenizer,
            buffer=self.buffer,
            eval_corpus=self.eval_texts,
            reference_corpus=[self.corpus_manager.corpus[i]
                              for i in random.sample(range(self.corpus_manager.n_passages),
                                                     min(500, self.corpus_manager.n_passages))],
            coreset_corpus=self.corpus_manager.corpus,
            device=self.device,
            batch_strategy="balanced",
            n_batch_episodes=min(plan.consolidation_batch_size, len(self.buffer.episodes)),
            batch_size=4,
            resonance_ewc_multiplier=resonance.ewc_multiplier,
            resonance_lr_multiplier=resonance.lr_multiplier,
        )

        # Track EWC for potential Phase 2 rollback
        # (ewc is created inside run_system3_phase1; we'd need to extract it
        #  or recreate it. For simplicity, store the pre-Phase1 model state.)

        phase1_ppl = phase1_report.get("post_ppl", 0) if isinstance(phase1_report, dict) else 0

        # Consolidation rate
        consol_count = sum(1 for ep in self.buffer.episodes.values()
                          if ep.consolidation_score > 0.5)
        consol_rate = consol_count / max(len(self.buffer.episodes), 1)

        # Update model reference
        if isinstance(phase1_report, dict) and phase1_report.get("merged", False):
            self.model = consolidated_model
        # Re-compute episode embedding keys after model update (spec §9.2)
            rekey_buffer(self.buffer, self.model, self.device)
            self.ewc = phase1_ewc

            # Refresh Fisher for the merged model (prevents stale EWC next cycle)
            reference_sample = [self.corpus_manager.corpus[i]
                                for i in random.sample(
                                    range(self.corpus_manager.n_passages),
                                    min(200, self.corpus_manager.n_passages))]
            fresh_fisher = recompute_fisher_post_merge(
                self.model, self.tokenizer, reference_sample, self.device,
            )
            # Build new EWC anchored at current (merged) parameters
            # Temporarily enable grads so EWCRegularizer can snapshot anchors
            for p in self.model.parameters():
                p.requires_grad = True
            self.ewc = EWCRegularizer(self.model, fresh_fisher, lambda_ewc=self.ewc.lambda_ewc)
            for p in self.model.parameters():
                p.requires_grad = False
            # Validate heads still produce discriminative errors post-merge
            if self.head_system is not None:
                head_cv = validate_heads_post_merge(
                    self.model, self.head_system, self.tokenizer,
                    self.eval_texts, self.device,
                )
                if head_cv < 0.3:
                    print(f"  ⚠ Head CV={head_cv:.3f} < 0.3 — forcing retrain next cycle")
                    self.state.force_retrain_heads = True
                else:
                    self.state.force_retrain_heads = False
        self._last_phase1_report = phase1_report

        # ========== SLEEP PHASE 2: REM (conditional) ==========
        phase2_report = None
        if plan.run_phase2 and len(self.buffer.episodes) > 20:
            print(f"\n--- SLEEP PHASE 2: REM ABSTRACTION ---")
            # For Phase 2 we need the pre-Phase1 model as reference.
            # On first cycle, that's the original base model.
            # For simplicity in proof-of-concept, load a fresh base.
            try:
                if pre_phase1_state is not None:
                    # Load from snapshot — faster than HuggingFace, and actually correct
                    # (compares single-cycle delta, not cumulative drift from base)
                    from transformers import AutoModelForCausalLM as AMCLM
                    pre_model = AMCLM.from_pretrained(
                        "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
                    )
                    pre_model.load_state_dict(pre_phase1_state)
                    pre_model = pre_model.to(self.device)
                    pre_model.eval()
                else:
                    # Fallback: load fresh base
                    from transformers import AutoModelForCausalLM as AMCLM
                    pre_model = AMCLM.from_pretrained(
                        "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
                    ).to(self.device)
                    pre_model.eval()

                batch = prepare_consolidation_batch(
                    self.buffer,
                    n_episodes=min(plan.consolidation_batch_size, len(self.buffer.episodes)),
                    strategy="balanced",
                )

                # Use real EWC from Phase 1
                phase2_ewc = phase1_ewc

                self.model, phase2_report = run_system3_phase2(
                    pre_model=pre_model,
                    post_model=self.model,
                    batch_episodes=batch.episodes,
                    ewc=phase2_ewc,
                    tokenizer=self.tokenizer,
                    eval_corpus=self.eval_texts,
                    device=self.device,
                    config=Phase2Config(max_episodes_to_probe=100),
                )

                del pre_model
                torch.cuda.empty_cache()
                del pre_phase1_state

                # Write Phase 2 diagnoses back to episodes
                if phase2_report is not None and hasattr(phase2_report, 'diagnoses'):
                    apply_diagnoses_to_buffer(
                        phase2_report.diagnoses, self.buffer, plan.cycle_number,
                    )

            except Exception as e:
                print(f"  Phase 2 failed: {e}")
                phase2_report = None

        # ========== POST-SLEEP MAINTENANCE ==========
        # Check coreset staleness
        if self.coreset and not plan.refresh_coreset:
            is_stale, _ = validate_coreset(
                self.model, self.tokenizer, self.coreset, self.device,
            )
            if is_stale:
                print("  Coreset stale — forcing refresh")
                plan.refresh_coreset = True

        if plan.refresh_coreset:
            print("\n  Refreshing coreset...")
            self.coreset = build_coreset(
                self.model, self.tokenizer,
                self.corpus_manager.corpus, self.device,
                target_size=self.config.coreset_size,
            )

        # Final eval
        final_eval = evaluate_perplexity(
            self.model, self.tokenizer, self.eval_texts, self.device,
            label=f"Cycle {plan.cycle_number} Final",
        )

        return CycleReport(
            cycle=plan.cycle_number,
            buffer_size=len(self.buffer.episodes),
            new_episodes=n_new,
            n_passages_processed=len(cycle_corpus),
            phase1_ppl=phase1_ppl,
            final_ppl=final_eval["perplexity"],
            final_loss=final_eval["loss"],
            mean_surprise=mean_surprise,
            type_distribution=type_dist,
            consolidation_rate=consol_rate,
            phase2_report=phase2_report,
        )

    def _update_state(self, report: CycleReport, cycle_hours: float):
        """Update system state from cycle report (spec §3.2)."""
        self.state.current_cycle += 1
        self.state.total_compute_used += cycle_hours

        self.state.eval_history.append(EvalSnapshot(
            cycle=self.state.current_cycle,
            perplexity=report.final_ppl,
            loss=report.final_loss,
            timestamp=time.time(),
            compute_cost=cycle_hours,
        ))

        if len(self.state.eval_history) >= 2:
            prev_ppl = self.state.eval_history[-2].perplexity
            curr_ppl = self.state.eval_history[-1].perplexity
            delta = (prev_ppl - curr_ppl) / max(prev_ppl, 1e-8)
            self.state.ppl_deltas.append(delta)

        self.state.mean_surprise_per_cycle.append(report.mean_surprise)
        self.state.buffer_size_per_cycle.append(report.buffer_size)
        self.state.consolidation_rate_per_cycle.append(report.consolidation_rate)
        self.state.type_distribution_per_cycle.append(report.type_distribution)

        # Buffer turnover
        if len(self.state.buffer_size_per_cycle) >= 2:
            prev_buf = self.state.buffer_size_per_cycle[-2]
            turnover = abs(report.buffer_size - prev_buf + report.new_episodes) / max(prev_buf, 1)
            self.state.buffer_turnover_per_cycle.append(turnover)

        # Phase 2 metrics
        if report.phase2_report is not None:
            diag = report.phase2_report.diagnosis_distribution
            total = max(sum(diag.values()), 1)
            self.state.generalization_rate_per_cycle.append(diag.get("GENERALIZED", 0) / total)
            self.state.memorization_rate_per_cycle.append(diag.get("MEMORIZED", 0) / total)
            self.state.interference_rate_per_cycle.append(diag.get("INTERFERENCE", 0) / total)

        # Surprise entropy (spec §3.1)
        if report.type_distribution:
            probs = list(report.type_distribution.values())
            entropy = -sum(p * math.log(p + 1e-10) for p in probs if p > 0)
            self.state.surprise_entropy_per_cycle.append(entropy)

        # EWC penalty ratio (spec §3.1)
        if isinstance(self._last_phase1_report, dict):
            phase1_report = self._last_phase1_report
            task_l = phase1_report.get("final_task_loss", 0)
            ewc_l = phase1_report.get("final_ewc_loss", 0)
            ratio = ewc_l / max(task_l, 1e-8) if task_l > 0 else 0.0
            self.state.ewc_penalty_ratio_per_cycle.append(ratio)

    def _print_summary(self, cycle: int, hours: float):
        ppl = self.state.eval_history[-1].perplexity
        delta = self.state.ppl_deltas[-1] if self.state.ppl_deltas else 0
        buf = self.state.buffer_size_per_cycle[-1] if self.state.buffer_size_per_cycle else 0

        print(f"\n  ┌─ Cycle {cycle} Summary ──────────────────────────────────┐")
        print(f"  │ Perplexity:    {ppl:.2f} ({delta:+.2%} vs previous)         │")
        print(f"  │ Buffer:        {buf} episodes                              │")
        print(f"  │ Compute:       {hours:.2f} hrs "
              f"({self.state.total_compute_used:.2f} total)          │")
        if self.state.consolidation_rate_per_cycle:
            cr = self.state.consolidation_rate_per_cycle[-1]
            print(f"  │ Consolidation: {cr:.1%}                                    │")
        print(f"  └──────────────────────────────────────────────────────────┘")

    def _final_report(self) -> TrainingReport:
        initial_ppl = self.state.eval_history[0].perplexity
        final_ppl = self.state.eval_history[-1].perplexity
        improvement = (initial_ppl - final_ppl) / initial_ppl * 100

        diagnosis = diagnose_convergence(self.state, self.config.buffer_capacity)

        report = TrainingReport(
            total_cycles=self.state.current_cycle,
            total_compute_hours=self.state.total_compute_used,
            initial_perplexity=initial_ppl,
            final_perplexity=final_ppl,
            total_improvement_pct=improvement,
            per_cycle_improvements=self.state.ppl_deltas,
            convergence_diagnosis=diagnosis,
            cycle_reports=self.cycle_reports,
        )

        # Final dashboard
        plot_training_dashboard(self.state)

        # Save decision log
        self.decision_log.dump("decision_log.json")

        print(f"\n{'=' * 70}")
        print(f"  FINAL REPORT")
        print(f"{'=' * 70}")
        print(f"  Cycles:          {report.total_cycles}")
        print(f"  Compute:         {report.total_compute_hours:.2f} GPU-hrs")
        print(f"  Initial PPL:     {report.initial_perplexity:.2f}")
        print(f"  Final PPL:       {report.final_perplexity:.2f}")
        print(f"  Improvement:     {report.total_improvement_pct:.2f}%")
        print(f"  Convergence:     {report.convergence_diagnosis.bottleneck}")
        if report.convergence_diagnosis.recommendation:
            print(f"  Recommendation:  {report.convergence_diagnosis.recommendation}")
        print(f"{'=' * 70}")

        return report

In [ ]:
# ╔═══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 50: LAUNCH — FULL STEPWISE LEARNING                               ║
# ║                                                                          ║
# ║  One function call to run the entire system.                              ║
# ║                                                                          ║
# ║  Requires: model, tokenizer, device (from Cell 1-3),                     ║
# ║            wake_corpus, eval_corpus (from Cell 4).                        ║
# ║                                                                          ║
# ║  NOTE: The _execute_cycle method references HierarchicalSurpriseTracker,  ║
# ║  ingest_system1_to_buffer, run_system3_phase1, run_system3_phase2, etc.  ║
# ║  These are all defined in Cells 1-40.                                    ║
# ║                                                                          ║
# ║  You will likely need to adapt the wake phase section of _execute_cycle   ║
# ║  to match your exact System 1 API (tracker.process_corpus, etc).         ║
# ║  The implementation below uses placeholder method names that you should   ║
# ║  wire up to your actual System 1 functions.                              ║
# ║                                                                          ║
# ║  Uncomment and run.                                                      ║
# ╚═══════════════════════════════════════════════════════════════════════════╝

controller = MetacognitiveController(
    model=model,
    tokenizer=tokenizer,
    corpus=wake_corpus,
    eval_texts=eval_corpus,
    device=device,
    config=ControllerConfig(
        compute_budget=5.0,         # 5 GPU-hours
        base_wake_size=3000,
        buffer_capacity=1600,        # Force competition
        coreset_size=200,
    ),
)

report = controller.run(max_cycles=15)

In [ ]:
# ╔═══════════════════════════════════════════════════════════════════════════╗
# ║  BASELINE: Loss-Ranked LoRA Fine-Tuning                                 ║
# ║                                                                          ║
# ║  Null hypothesis comparison for the full stepwise system.                ║
# ║  Same corpus, same compute budget, same LoRA config — but instead of     ║
# ║  hierarchical surprise + sleep consolidation, just sort passages by      ║
# ║  model loss and fine-tune on the hardest ones.                           ║
# ╚═══════════════════════════════════════════════════════════════════════════╝

import torch
import numpy as np
import time
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm import tqdm


def run_baseline_comparison(
    corpus: List[str],
    eval_texts: List[str],
    device: torch.device,
    n_train_passages: int = 800,       # Match buffer capacity
    lora_rank: int = 16,               # Match stepwise system
    n_epochs: int = 3,                 # Match consolidation epochs
    lr: float = 2e-4,                  # Match Phase 1 LR
    max_length: int = 512,
    batch_size: int = 4,
) -> dict:
    """
    Baseline: rank passages by loss, LoRA fine-tune on top-K hardest.

    Returns dict with pre/post perplexity and training stats for
    direct comparison against stepwise system results.
    """

    print("=" * 65)
    print("  BASELINE: Loss-Ranked LoRA Fine-Tuning")
    print("=" * 65)

    # Load fresh model
    model = AutoModelForCausalLM.from_pretrained(
        "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
    ).to(device)
    tokenizer = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # Pre-training eval
    pre_eval = evaluate_perplexity(model, tokenizer, eval_texts, device, label="Baseline Pre")

    # ---- Step 1: Score all passages by loss ----
    print(f"\n[1/4] Scoring {len(corpus)} passages by loss...")
    model.eval()
    passage_losses = []
    with torch.no_grad():
        for text in tqdm(corpus, desc="Scoring"):
            enc = tokenizer(text, return_tensors="pt", truncation=True,
                            max_length=max_length).to(device)
            if enc.input_ids.shape[1] < 10:
                passage_losses.append(0.0)
                continue
            out = model(**enc, labels=enc.input_ids)
            passage_losses.append(out.loss.item())

    # ---- Step 2: Select top-K hardest passages ----
    ranked_indices = np.argsort(passage_losses)[::-1]
    train_indices = ranked_indices[:n_train_passages]
    train_texts = [corpus[i] for i in train_indices]

    mean_train_loss = np.mean([passage_losses[i] for i in train_indices])
    print(f"  Selected {n_train_passages} passages, mean loss={mean_train_loss:.3f}")

    # ---- Step 3: LoRA fine-tune ----
    print(f"\n[2/4] Attaching LoRA (rank={lora_rank})...")
    lora_config = LoraConfig(
        r=lora_rank,
        lora_alpha=lora_rank * 2,
        target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
        lora_dropout=0.05,
        task_type="CAUSAL_LM",
    )
    peft_model = get_peft_model(model, lora_config)
    trainable = sum(p.numel() for p in peft_model.parameters() if p.requires_grad)
    print(f"  Trainable params: {trainable / 1e6:.2f}M")

    optimizer = torch.optim.AdamW(
        [p for p in peft_model.parameters() if p.requires_grad],
        lr=lr, weight_decay=0.01,
    )

    print(f"\n[3/4] Training ({n_epochs} epochs, {len(train_texts)} passages)...")
    peft_model.train()
    train_start = time.time()
    all_losses = []

    for epoch in range(n_epochs):
        epoch_losses = []
        random.shuffle(train_texts)

        for i, text in enumerate(train_texts):
            enc = tokenizer(text, return_tensors="pt", truncation=True,
                            max_length=max_length).to(device)
            if enc.input_ids.shape[1] < 10:
                continue

            out = peft_model(**enc, labels=enc.input_ids)
            loss = out.loss

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(
                [p for p in peft_model.parameters() if p.requires_grad], 1.0)
            optimizer.step()

            epoch_losses.append(loss.item())

            if (i + 1) % 200 == 0:
                print(f"  Epoch {epoch+1} [{i+1}/{len(train_texts)}] "
                      f"loss={np.mean(epoch_losses[-50:]):.4f}")

        all_losses.extend(epoch_losses)
        print(f"  Epoch {epoch+1} complete — mean loss={np.mean(epoch_losses):.4f}")

    train_time = time.time() - train_start

    # ---- Step 4: Merge and evaluate ----
    print(f"\n[4/4] Merging LoRA and evaluating...")
    merged = peft_model.merge_and_unload()
    post_eval = evaluate_perplexity(merged, tokenizer, eval_texts, device, label="Baseline Post")

    ppl_change = (post_eval["perplexity"] - pre_eval["perplexity"]) / pre_eval["perplexity"]

    print(f"\n{'=' * 65}")
    print(f"  BASELINE COMPLETE — {train_time:.1f}s")
    print(f"  Pre PPL:  {pre_eval['perplexity']:.2f}")
    print(f"  Post PPL: {post_eval['perplexity']:.2f} ({ppl_change*100:+.2f}%)")
    print(f"  Training steps: {len(all_losses)}")
    print(f"{'=' * 65}")

    return {
        "pre_ppl": pre_eval["perplexity"],
        "post_ppl": post_eval["perplexity"],
        "ppl_change_pct": ppl_change * 100,
        "n_passages": n_train_passages,
        "n_epochs": n_epochs,
        "total_steps": len(all_losses),
        "train_time_s": train_time,
        "final_loss": np.mean(all_losses[-100:]) if all_losses else 0,
    }


# ---- Run it ----
baseline_results = run_baseline_comparison(
    corpus=wake_corpus,      # Same corpus as stepwise system
    eval_texts=eval_corpus,  # Same eval set
    device=device,
    n_train_passages=800,    # Match buffer capacity
    lora_rank=16,            # Match stepwise LoRA rank
    n_epochs=3,              # Match consolidation epochs
)